# Clean SD3.5 CityPersons Pedestrian-Insertion Img2Img Augmentation Pipeline

Refactored notebook: module files are written first, then imported with autoreload, then called.

## 1. Install Dependencies

In [ ]:
!pip install -q "diffusers>=0.30.0,<1.0.0" "transformers>=4.40.0" "accelerate>=0.30.0" sentencepiece protobuf safetensors ultralytics opencv-python


## 2. Write Modules

In [ ]:
%%writefile /kaggle/working/sd35_config.py
"""SD3.5 CityPersons augmentation: configuration."""

from pathlib import Path

# User-facing settings: change these between runs.
RUN_PRESET = "batch"  # smoke | quality | batch

USER_CONFIG = {
    "DATASET_ROOT_CANDIDATES": [
        Path('/kaggle/input/datasets/muttahirulislam/citypersons-dataset-with-bg-image/yolo_dir/yolo_dir'),
        Path('/kaggle/input/citypersons-dataset-with-bg-image/yolo_dir/yolo_dir'),
        Path('/kaggle/input/citypersons-dataset-with-bg-image'),
        Path('/kaggle/input/datasets/samyamine23/cityperson'),
        Path('/kaggle/input/cityperson'),
        Path('/kaggle/input/citypersons'),
        Path('/kaggle/input/city-persons'),
        Path('/kaggle/input/city-persons-2-0'),
        Path('/kaggle/input/city-persons-20'),
        Path('/kaggle/input/dataset'),
        Path('/kaggle/working/Dataset'),
    ],
    "MODEL_BACKEND": "sd35",
    "SD35_MODEL_ID": "stabilityai/stable-diffusion-3.5-medium",
    "OUTPUT_DIR": Path("/kaggle/working/sd35_citypersons_scale_corrected"),
    "RESOLUTION": 512,
    "MAX_TRAIN_IMAGES": 500,
    "USE_T5": False,
    "TRAIN_DEVICE": "cuda:0",
    "USE_ALL_GPUS_FOR_AUGMENTATION": True,
    "AUGMENTATION_DEVICES": None,
    "USE_MODEL_CPU_OFFLOAD": True,
    "AUGMENTATION_VARIANTS": [
        'add_single_pedestrian',
        'add_two_pedestrians',
        'add_small_group',
        'add_occluded_pedestrian',
        'add_distant_pedestrian',
        'add_near_pedestrian',
    ],
    "SEED": 42,
}

PARAMETER_OVERRIDES = {
    # Keep this short. Put manual one-off changes here, e.g.:
    # "CONTEXT_GENERATION_RETRIES": 4,
    # "MAX_PERSON_PERSON_OVERLAP_RATIO": 0.18,
}

AUTOTUNE_SETTINGS = {
    "enabled": True,
    "min_samples": 30,
    "target_accept_rate": 0.55,
    "target_quality_score": 0.72,
    "aggressiveness": 0.60,
    "max_adjustment_ratio": 1.35,
    "max_retry_budget": 5,
    "save_snapshot": True,
    "snapshot_dir": "/kaggle/working/autotune_snapshots",
    "print_report": True,
}

RUN_PRESETS = {
    "smoke": {
        "AUGMENTATIONS_PER_BUCKET": 5,
        "TARGET_SPLITS": ["train"],
        "SAVE_PATCH_DEBUG": True,
        "PATCH_DEBUG_MAX_ITEMS": 10,
        "CONTEXT_GENERATION_RETRIES": 1,
    },
    "quality": {
        "AUGMENTATIONS_PER_BUCKET": 25,
        "TARGET_SPLITS": ["train", "val"],
        "SAVE_PATCH_DEBUG": True,
        "PATCH_DEBUG_MAX_ITEMS": 24,
        "CONTEXT_GENERATION_RETRIES": 3,
    },
    "batch": {
        "AUGMENTATIONS_PER_BUCKET": 200,
        "TARGET_SPLITS": ["train", "val"],
        "SAVE_PATCH_DEBUG": True,
        "PATCH_DEBUG_MAX_ITEMS": 24,
        "CONTEXT_GENERATION_RETRIES": 3,
    },
}

# Variant profile: weights and per-variant SD3.5 generation settings.
VARIANT_PROFILE = {'add_single_pedestrian': {'weight': 0.24, 'strength': 0.72, 'guidance': 6.8, 'steps': 36},
 'add_two_pedestrians': {'weight': 0.18, 'strength': 0.74, 'guidance': 6.9, 'steps': 36},
 'add_small_group': {'weight': 0.16, 'strength': 0.76, 'guidance': 7.0, 'steps': 38},
 'add_occluded_pedestrian': {'weight': 0.18, 'strength': 0.74, 'guidance': 6.8, 'steps': 36},
 'add_distant_pedestrian': {'weight': 0.14, 'strength': 0.68, 'guidance': 6.6, 'steps': 34},
 'add_near_pedestrian': {'weight': 0.1, 'strength': 0.76, 'guidance': 7.3, 'steps': 38}}
AUGMENTATION_VARIANT_WEIGHTS = {name: cfg["weight"] for name, cfg in VARIANT_PROFILE.items()}
VARIANT_STRENGTHS = {name: cfg["strength"] for name, cfg in VARIANT_PROFILE.items()}
VARIANT_GUIDANCE_SCALES = {name: cfg["guidance"] for name, cfg in VARIANT_PROFILE.items()}
VARIANT_NUM_INFERENCE_STEPS = {name: cfg["steps"] for name, cfg in VARIANT_PROFILE.items()}

# Domain configuration groups. Keep behavior-preserving threshold changes here.

DATASET_CONFIG = {'SCENE_BUCKETS': ['urban_pedestrian_scene'],
 'IMAGE_SUBDIR': 'images',
 'CAPTION_CSV': None,
 'MATCH_LABEL_JSON': False,
 'PATCH_PERSON_CLASS_IDS': {0},
 'PATCH_VEHICLE_CLASS_IDS': {2, 5, 7}}

PLACEMENT_CONFIG = {'BACKGROUND_PRESERVATION_MODE': 'context_person_composite',
 'PATCH_CONTEXT_RATIO': 0.18,
 'PATCH_MIN_SIZE': 128,
 'PATCH_FEATHER_RADIUS': 3,
 'PATCH_MAX_PLACEMENT_TRIES': 180,
 'PATCH_ROAD_Y_RANGE': (0.66, 0.92),
 'INSERTION_EDGE_MARGIN': 16,
 'INSERTION_OVERLAP_PENALTY': 3.0,
 'ALLOW_PERSON_VEHICLE_OVERLAP': True,
 'ALLOW_PERSON_PERSON_OVERLAP': True,
 'MAX_PERSON_PERSON_OVERLAP_RATIO': 0.08,
 'OCCLUDED_PERSON_MAX_HEIGHT_RATIO': 0.68,
 'OCCLUDED_PERSON_MAX_FOOT_Y_DELTA': 0,
 'PERSON_OVERLAP_FRONT_LAYER_BONUS': 0.0,
 'MAX_VEHICLE_OVERLAP_RATIO': 0.35,
 'VEHICLE_OVERLAP_FRONT_LAYER_BONUS': 0.18,
 'INSERTION_CENTER_BIAS': 0.0,
 'PLACEMENT_SLOT_BIAS': 1.15,
 'PLACEMENT_SLOT_XS': (0.2, 0.36, 0.52, 0.68, 0.82),
 'PLACEMENT_SLOT_YS': (0.68, 0.74, 0.8, 0.86, 0.91),
 'PLACEMENT_SLOT_JITTER': 0.1,
 'SMART_PLACEMENT_VERSION': 'v2',
 'USE_SEMANTIC_PLACEMENT': True,
 'SEMANTIC_SEGMENTATION_MODEL_ID': 'nvidia/segformer-b0-finetuned-cityscapes-1024-1024',
 'VALID_PLACEMENT_LABELS': {'terrain', 'sidewalk', 'road'},
 'AVOID_PLACEMENT_LABELS': {'bicycle',
                            'building',
                            'bus',
                            'car',
                            'fence',
                            'motorcycle',
                            'person',
                            'pole',
                            'rider',
                            'sky',
                            'traffic light',
                            'traffic sign',
                            'train',
                            'truck',
                            'wall'},
 'MIN_FOOT_SUPPORT': 0.35,
 'MAX_FOOT_AVOID_SUPPORT': 0.08,
 'MIN_BODY_VALID_SUPPORT': 0.08,
 'MAX_BODY_AVOID_SUPPORT': 0.22,
 'REQUIRE_SEMANTIC_PLACEMENT': False,
 'MIN_ACCEPTED_PLACEMENT_SCORE': -1000.0,
 'SEMANTIC_FOOT_WEIGHT': 5.0,
 'SEMANTIC_AVOID_PENALTY': 7.0}

SCALE_CONFIG = {'PERSPECTIVE_SCALE_NEAR': 1.14,
 'PERSPECTIVE_SCALE_FAR': 0.50,
 'USE_REFERENCE_PERSON_SCALE': True,
 'REFERENCE_SCALE_MIN_PERSON_HEIGHT': 24,
 'REFERENCE_SCALE_MAX_PERSON_HEIGHT': 180,
 'REFERENCE_SCALE_BLEND': 0.56,
 'REFERENCE_SCALE_MAX_Y_DISTANCE': 70,
 'CAR_HEIGHT_TO_PERSON_HEIGHT_RATIO': 1.05,
 'CAR_HEIGHT_TO_PERSON_HEIGHT_MIN_RATIO': 0.9,
 'CAR_HEIGHT_TO_PERSON_HEIGHT_MAX_RATIO': 1.1,
 'NEAR_PERSON_SCALE_MULTIPLIER': 1.6,
 'PERSON_TARGET_SCALE_MULTIPLIER': 1.6,
 'CAR_REFERENCE_SCALE_BLEND': 0.7,
 'CAR_REFERENCE_MAX_Y_DISTANCE': 120,
 'CAR_REFERENCE_MIN_HEIGHT': 18,
 'CAR_REFERENCE_MAX_HEIGHT': 130,
 'REFERENCE_SCALE_MIN_FACTOR': 0.7,
 'REFERENCE_SCALE_MAX_FACTOR': 1.75,
 'REFERENCE_SCALE_MIN_SAMPLES': 2,
 'REFERENCE_SCALE_MIN_Y_GAP': 24,
 'REFERENCE_SCALE_MIN_SLOPE': 0.06,
 'REFERENCE_SCALE_MAX_SLOPE': 1.1,
 'PERSON_ASPECT_RATIO': 0.36,
 'USE_PERSPECTIVE_SCALE_CORRECTION': True,
 'SCALE_CORRECTION_SOFT_MIN': 0.62,
 'SCALE_CORRECTION_SOFT_MAX': 2.2,
 'SCALE_CORRECTION_HARD_MIN': 0.35,
 'SCALE_CORRECTION_HARD_MAX': 3.2,
 'SCALE_CORRECTION_BORDERLINE_RETRY': True,
 'SCALE_CORRECTION_BORDERLINE_MIN_CONF': 0.48,
 'SCALE_CORRECTION_BORDERLINE_MIN_MASK_AREA_RATIO': 0.0015,
 'MIN_BORDER_MARGIN_RATIO': 0.018,
 'ALLOW_PARTIAL_SMALL_GROUP': True,
 'FLEXIBLE_SCALE_INPAINT': True,
 'SCALE_ENVELOPE_HEIGHT_MULT': 1.16,
 'SCALE_ENVELOPE_WIDTH_MULT': 1.3,
 'GROUP_PERSON_GAP_RATIO': 0.1,
 'GROUP_PERSON_HEIGHT_JITTER': 0.1,
 'GROUP_SCALE_JITTER_ENABLED': False,
 'PERSPECTIVE_MONOTONIC_SCALE_ENABLED': True,
 'PERSPECTIVE_MONOTONIC_MIN_RATIO': 1.04}

YOLO_EVAL_CONFIG = {'CONTEXT_PERSON_SEGMENTATION_MODEL': 'yolov8m-seg.pt',
 'CONTEXT_PERSON_MIN_CONFIDENCE': 0.12,
 'MIN_RETRY_PERSON_CONFIDENCE': 0.35,
 'MIN_PERSON_CONF_BY_VARIANT': {'add_single_pedestrian': 0.24,
                                'add_two_pedestrians': 0.20,
                                'add_small_group': 0.18,
                                'add_occluded_pedestrian': 0.16,
                                'add_distant_pedestrian': 0.14,
                                'add_near_pedestrian': 0.24},
 'MIN_GHOST_PERSON_MASK_AREA_RATIO': 0.002,
 'MIN_GHOST_PERSON_CONTRAST_255': 12.0,
 'CONTEXT_PERSON_MASK_THRESHOLD': 0.40,
 'MIN_PERSON_DET_TARGET_HEIGHT_RATIO': 0.62,
 'MAX_PERSON_TOP_OFFSET_RATIO': 0.42,
 'MAX_PERSON_BOTTOM_GAP_RATIO': 0.3,
 'MIN_PERSON_MASK_DET_HEIGHT_RATIO': 0.62,
 'MIN_PERSON_MASK_TARGET_HEIGHT_RATIO': 0.48,
 'MIN_PERSON_MASK_VERTICAL_BAND_COVERAGE': 0.18,
 'MIN_PERSON_MASK_ASPECT_RATIO': 1.35,
 'MAX_PERSON_MASK_ASPECT_RATIO': 6.0,
 'PERSON_MASK_DILATE_FOR_ACCESSORIES': 2,
 'PERSON_MASK_ERODE_PIXELS': 0,
 'PERSON_MASK_TRIM_FRINGE_PIXELS': 1,
 'PERSON_PASTE_HARD_THRESHOLD': 88,
 'PERSON_PASTE_FEATHER_RADIUS': 0.18,
 'ACCESSORY_KEEP_COMPONENTS': 12,
 'KEEP_EXTRA_GENERATED_PEOPLE': True,
 'MAX_EXTRA_GENERATED_PEOPLE': 2,
 'EXTRA_PERSON_MIN_SCORE_DELTA': 1.35,
 'PERSON_CROP_MASK_PADDING': 4,
 'MAX_GENERATED_PERSON_SHARPNESS_STD': 9.0,
 'ACCESSORY_MIN_COMPONENT_AREA_RATIO': 0.002,
 'FILL_PERSON_MASK_HOLES': True}

MASK_CONFIG = {'HUMAN_MASK_PADDING': 10,
 'HUMAN_MASK_BLUR_RADIUS': 2,
 'BBOX_MASK_PADDING': 8,
 'BBOX_MASK_BLUR_RADIUS': 3,
 'BBOX_MASK_RADIUS': 8}

COMPOSITING_CONFIG = {'USE_SEAMLESS_CLONE': False,
 'SEAMLESS_CLONE_MODE': 'mixed',
 'SEAMLESS_CLONE_MIN_MASK_AREA_RATIO': 0.0006,
 'SEAMLESS_FOREGROUND_PRESERVE_STRENGTH': 0.92,
 'SEAMLESS_FOREGROUND_CORE_ERODE': 0,
 'SEAMLESS_EDGE_ONLY_BLEND': True,
 'HARMONIZATION_COLOR_STRENGTH': 0.32,
 'HARMONIZATION_BRIGHTNESS_STRENGTH': 0.28,
 'HARMONIZATION_CONTRAST_STRENGTH': 0.22,
 'HARMONIZATION_SATURATION_STRENGTH': 0.18,
 'HARMONIZATION_NOISE_STRENGTH': 0.28,
 'HARMONIZATION_BLUR_SIGMA': 0.55,
 'HARMONIZATION_BLUR_STRENGTH': 0.24,
 'PERSON_TONE_FILTER_ENABLED': True,
 'PERSON_TONE_FILTER_STRENGTH': 0.22,
 'PERSON_TONE_FILTER_CORE_STRENGTH': 0.16,
 'PERSON_TONE_FILTER_MAX_COLOR_SHIFT': 10.0,
 'PERSON_TONE_FILTER_MAX_BRIGHTNESS_SHIFT': 6.0,
 'CONTACT_SHADOW_ENABLED': True,
 'CONTACT_SHADOW_OPACITY_NEAR': 46,
 'CONTACT_SHADOW_OPACITY_FAR': 16,
 'DRAW_INSERTION_GUIDE': True,
 'PERSON_GENERATION_ONLY_MODE': True,
 'CONTEXT_PERSON_GENERATION_PIPELINE': 'img2img',
 'PERSON_GENERATION_NEUTRAL_STRENGTH': 0.55,
 'PERSON_GENERATION_CONTEXT_DARKEN': 0.03,
 'INSERTION_GUIDE_ALPHA': 0.44,
 'INSERTION_GUIDE_BLUR': 0.25,
 'CONTEXT_CROP_EXPAND': 3.4,
 'CONTEXT_CROP_MIN_SIZE': 192,
 'CONTEXT_INPAINT_MASK_PADDING': 46,
 'COLOR_MATCH_PERSON_TO_SCENE': True,
 'COLOR_MATCH_STRENGTH': 0.72,
 'COLOR_MATCH_CONTEXT_PAD': 18,
 'FOREGROUND_HARMONIZATION_CORE_ALPHA': 0.08,
 'FOREGROUND_HARMONIZATION_EDGE_ERODE': 2,
 'TEXTURE_MATCH_PERSON_TO_SCENE': True,
 'TEXTURE_MATCH_STRENGTH': 0.28,
 'TEXTURE_MATCH_MIN_BLUR': 0.45,
 'TEXTURE_MATCH_MAX_BLUR': 1.15,
 'TEXTURE_MATCH_CONTEXT_PAD': 24,
 'EDGE_HALO_NEUTRALIZE': True,
 'EDGE_HALO_COLOR_MATCH_STRENGTH': 0.38,
 'EDGE_HALO_WIDTH': 1,
 'EDGE_HALO_MIN_ALPHA': 0.02,
 'EDGE_HALO_MAX_ALPHA': 0.72,
 'EDGE_LOCAL_BG_RADIUS': 7.0,
 'EDGE_HORIZON_BG_BLEND': 0.70,
 'EDGE_HORIZON_BAND_RATIO': 0.10,
 'EDGE_BG_CONTEXT_PAD': 18,
 'CONTEXT_TARGET_BBOX_EXPAND_FOR_DETECTION': 2.2,
 'OCCLUSION_AWARE_COMPOSITE': True,
 'OCCLUSION_MASK_BBOX_PADDING': 3,
 'OCCLUSION_MASK_BLUR_RADIUS': 1.2,
 'MIN_OCCLUDER_OVERLAP_RATIO': 0.03,
 'VEHICLE_OCCLUDER_MAX_FOOT_Y_DELTA': 18,
 'OCCLUDER_MIN_HEIGHT_RATIO': 0.35,
 'MAX_OCCLUSION_REMOVED_MASK_RATIO': 0.55}

VALIDATION_CONFIG = {'MIN_CORRECTED_PERSON_AREA_RATIO': 0.00045,
 'MIN_CORRECTED_ASPECT_RATIO': 1.35,
 'MAX_CORRECTED_ASPECT_RATIO': 3.6,
 'HEAD_MARGIN_RATIO': 0.18,
 'FOOT_MARGIN_RATIO': 0.12,
 'SIDE_MARGIN_RATIO': 0.2,
 'MIN_PERSON_CONF': 0.35,
 'MIN_PERSON_HEIGHT_RATIO': 0.06,
 'MAX_PERSON_HEIGHT_RATIO': 0.62,
 'MIN_PERSON_ASPECT_RATIO': 1.6,
 'MAX_PERSON_ASPECT_RATIO': 4.5,
 'MAX_BOTTOM_Y_RATIO': 0.92,
 'REJECT_IF_MASK_TOUCHES_BORDER': True,
 'MIN_GENERATED_HEIGHT_RATIO': 0.75,
 'MAX_GENERATED_HEIGHT_RATIO': 2.05,
 'PERSON_BORDER_REJECT_PIXELS': 3,
 'MIN_MASK_BBOX_HEIGHT_RATIO': 0.50,
 'STRICT_EARLY_PERSON_SCALE_FILTER': True,
 'FINAL_SCALE_VALIDATION_ENABLED': True,
 'MIN_ACCEPTED_SINGLE_HEIGHT_RATIO': 0.09,
 'MIN_ACCEPTED_NEAR_HEIGHT_RATIO': 0.16,
 'MIN_ACCEPTED_DISTANT_HEIGHT_RATIO': 0.08,
 'MIN_ACCEPTED_FOREGROUND_HEIGHT_RATIO': 0.12,
 'MIN_ACCEPTED_MASK_OPAQUE_RATIO': 0.24,
 'FINAL_MIN_SCALE_RATIO': 0.76,
 'FINAL_MAX_SCALE_RATIO': 1.85,
 'MIN_ACCEPTED_PERSPECTIVE_HEIGHT_RATIO_FAR': 0.08,
 'MIN_ACCEPTED_PERSPECTIVE_HEIGHT_RATIO_NEAR': 0.17,
 'CONTEXT_MIN_GENERATED_MASK_DIFF': 0.003,
 'CONTEXT_MIN_PERSON_TARGET_OVERLAP': 0.12,
 'CONTEXT_MIN_PERSON_MASK_AREA_RATIO': 0.00045,
 'MAX_MASK_OUTSIDE_INSERTION_RATIO': 0.36,
 'CONTEXT_MIN_FINAL_PERSON_DIFF': 0.01,
 'FINAL_COMPOSITE_MIN_MAE_255': 1.0,
 'FINAL_COMPOSITE_MIN_MAE_255_SEAMLESS': 0.75,
 'POST_PASTE_RETRY_MIN_MASK_AREA_RATIO': 0.0015}

RETRY_CONFIG = {'CONTEXT_PERSON_FALLBACK_TO_BBOX_INPAINT': False,
 'CONTEXT_GENERATION_RETRIES': 4,
 'EARLY_STOP_SCALE_UNRECOVERABLE_STREAK': 2,
 'DISTANT_EXTRA_RETRIES': 1,
 'SMALL_GROUP_EXTRA_RETRIES': 1,
 'NEAR_MAX_RETRIES': 2}

GENERATION_CONFIG = {'AUGMENTATION_STRENGTH': 0.72, 'GUIDANCE_SCALE': 7.2, 'NUM_INFERENCE_STEPS': 36}

DEBUG_CONFIG = {'DEBUG_SCALE_VISUALIZATION': False}

ADDIT_CONFIG = {
 'ADDIT_CONCEPT_ENABLED': True,
 'ADDIT_WEIGHTED_EXTENDED_ATTENTION': False,
 'ADDIT_STRUCTURE_TRANSFER': False,
 'ADDIT_SUBJECT_GUIDED_BLEND_PROXY': True,
 'ADDIT_BLEND_CONTEXT_DILATE': 1,
 'ADDIT_BLEND_EDGE_RADIUS': 1.25,
 'ADDIT_BLEND_SHADOW_EXTENSION': 0.22,
 'ADDIT_BLEND_SHADOW_BLUR': 5.0,
 'ADDIT_BLEND_SHADOW_ALPHA': 0.24,
}

CONFIG_GROUPS = [
    DATASET_CONFIG,
    PLACEMENT_CONFIG,
    SCALE_CONFIG,
    YOLO_EVAL_CONFIG,
    MASK_CONFIG,
    COMPOSITING_CONFIG,
    VALIDATION_CONFIG,
    RETRY_CONFIG,
    GENERATION_CONFIG,
    DEBUG_CONFIG,
    ADDIT_CONFIG,
]

def flatten_config(*groups):
    config = {}
    for group in groups:
        config.update(group)
    return config

def cfg(name, default=None):
    return EFFECTIVE_CONFIG.get(name, default)

if RUN_PRESET not in RUN_PRESETS:
    raise ValueError(f"Unknown RUN_PRESET={RUN_PRESET!r}. Choose one of {sorted(RUN_PRESETS)}.")

EFFECTIVE_CONFIG = flatten_config(USER_CONFIG, *CONFIG_GROUPS, RUN_PRESETS[RUN_PRESET], PARAMETER_OVERRIDES)
BASE_EFFECTIVE_CONFIG = dict(EFFECTIVE_CONFIG)
BASE_VARIANT_PROFILE = {name: dict(profile) for name, profile in VARIANT_PROFILE.items()}
globals().update(EFFECTIVE_CONFIG)

def looks_like_roboflow_citypersons_root(path):
    path = Path(path)
    return (
        (path / "train" / "images").exists()
        and ((path / "valid" / "images").exists() or (path / "val" / "images").exists())
    )


def resolve_dataset_root(candidates):
    for path in candidates:
        if looks_like_roboflow_citypersons_root(path):
            return Path(path)
    kaggle_input = Path("/kaggle/input")
    if kaggle_input.exists():
        for path in sorted(kaggle_input.rglob("data.yaml")):
            root = path.parent
            if looks_like_roboflow_citypersons_root(root):
                return root
    return next((Path(path) for path in candidates if Path(path).exists()), Path(candidates[0]))


# Derived paths.
DATASET_ROOT = resolve_dataset_root(DATASET_ROOT_CANDIDATES)
VALID_SPLIT_NAME = "valid" if (DATASET_ROOT / "valid" / "images").exists() else "val"
IMAGE_ROOT = DATASET_ROOT
LABEL_ROOT = DATASET_ROOT
DATASET_SPLIT_DIRS = {
    "train": DATASET_ROOT / "train" / "images",
    "val": DATASET_ROOT / VALID_SPLIT_NAME / "images",
}
LABEL_SPLIT_DIRS = {
    "train": DATASET_ROOT / "train" / "labels",
    "val": DATASET_ROOT / VALID_SPLIT_NAME / "labels",
}
METRICS_DIR = Path("/kaggle/working/metrics")
METRICS_CSV_PATH = METRICS_DIR / "augmentation_metrics.csv"
METRICS_SUMMARY_PATH = METRICS_DIR / "augmentation_metrics_summary.csv"
METRICS_PLOT_PATH = METRICS_DIR / "augmentation_metrics_by_variant.png"
PATCH_DEBUG_DIR = OUTPUT_DIR / "patch_debug"

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
BASE_CAPTION = "urban street photo"
PRESERVATION_PROMPT = "full body visible, grounded feet, natural scale, matching light"

SCENE_PROMPTS = {
    "urban_pedestrian_scene": "urban street photo",
}

VARIANT_PROMPTS = {
    "add_single_pedestrian": "one pedestrian",
    "add_two_pedestrians": "two pedestrians",
    "add_small_group": "three pedestrians",
    "add_occluded_pedestrian": "partly occluded pedestrian behind foreground object",
    "add_distant_pedestrian": "distant visible pedestrian",
    "add_near_pedestrian": "near larger pedestrian",
}

NEGATIVE_PROMPT = "cropped, missing head, missing legs, thin body, giant, closeup, floating, ghost, bad perspective, hard seam"


def ensure_output_dirs():
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    METRICS_DIR.mkdir(parents=True, exist_ok=True)
    PATCH_DEBUG_DIR.mkdir(parents=True, exist_ok=True)
    return {
        "output_dir": OUTPUT_DIR,
        "metrics_dir": METRICS_DIR,
        "patch_debug_dir": PATCH_DEBUG_DIR,
    }


In [ ]:
%%writefile /kaggle/working/sd35_data.py
"""SD3.5 CityPersons augmentation: data scanning and previews."""

import csv
import gc
import json
from datetime import datetime
import math
import numpy as np
import os
import random
import re
import statistics
import time
import warnings
from concurrent.futures import ThreadPoolExecutor, as_completed
from dataclasses import dataclass
from pathlib import Path
from threading import Lock
from typing import Optional

import matplotlib.pyplot as plt
import torch
from PIL import Image, ImageOps, ImageDraw, ImageFilter, ImageChops

try:
    import cv2
except ImportError:
    cv2 = None

from sd35_config import *

@dataclass
class ImageRecord:
    path: Path
    split: str
    bucket: str
    caption: str
    label_path: Optional[Path] = None
    weather: Optional[str] = None
    timeofday: Optional[str] = None
    scene: Optional[str] = None


def load_caption_map(csv_path):
    if not csv_path:
        return {}
    csv_path = Path(csv_path)
    if not csv_path.exists():
        print(f"Caption CSV not found: {csv_path}. Using default captions.")
        return {}
    caption_map = {}
    with csv_path.open("r", encoding="utf-8-sig", newline="") as handle:
        reader = csv.DictReader(handle)
        for row in reader:
            file_name = row.get("file_name") or row.get("filename") or row.get("image")
            caption = row.get("caption") or row.get("prompt")
            if file_name and caption:
                caption_map[Path(file_name).name] = caption
    return caption_map


def infer_split(path):
    parts = [part.lower() for part in Path(path).parts]
    if "train" in parts:
        return "train"
    if "valid" in parts or "val" in parts:
        return "val"
    if "test" in parts:
        return "test"
    return "train"


def find_label_path(image_path, label_dir=None):
    image_path = Path(image_path)
    split = infer_split(image_path)
    split_label_dir = LABEL_SPLIT_DIRS.get(split)
    sibling_label_dir = image_path.parent.parent / "labels" if image_path.parent.name == "images" else None
    candidates = [
        image_path.with_suffix(".json"),
        image_path.parent / f"{image_path.stem}.json",
        image_path.parent / f"{image_path.stem}.txt",
        sibling_label_dir / f"{image_path.stem}.txt" if sibling_label_dir else None,
        sibling_label_dir / f"{image_path.stem}.json" if sibling_label_dir else None,
        split_label_dir / f"{image_path.stem}.txt" if split_label_dir else None,
        split_label_dir / f"{image_path.stem}.json" if split_label_dir else None,
        image_path.parent / f"{image_path.stem}_gtBboxCityPersons.json",
        image_path.parent / f"{image_path.stem}_gtFine_polygons.json",
    ]
    for candidate in candidates:
        if candidate and candidate.exists():
            return candidate
    return None


def build_caption(path, bucket, caption_map, metadata=None, include_weather=True):
    if Path(path).name in caption_map:
        return caption_map[Path(path).name]
    return BASE_CAPTION


def build_generation_prompt(record, variant):
    variant_prompt = VARIANT_PROMPTS[variant]
    placement_clause = "in an empty road or sidewalk area"
    if variant == "add_occluded_pedestrian":
        placement_clause = "with realistic occlusion"
    return (
        f"{record.caption}. Add {variant_prompt} {placement_clause}. "
        f"Keep scene unchanged. {PRESERVATION_PROMPT}"
    )


def build_variant_negative_prompt(variant):
    return (
        NEGATIVE_PROMPT
        + ", missing feet, amputated legs, tiny ghost, sticker outline"
    )


def is_source_image(path):
    path = Path(path)
    if path.suffix.lower() not in IMAGE_EXTS:
        return False
    name = path.name.lower()
    if any(token in name for token in ["mask", "label", "gtfine", "gtbbox", "instance", "polygon", "color"]):
        return False
    return True


def list_image_paths_fast(split_dir, max_images=None):
    split_dir = Path(split_dir)
    if not split_dir.exists():
        return []
    paths = []
    for path in sorted(split_dir.rglob("*")):
        if path.is_file() and is_source_image(path):
            paths.append(path)
            if max_images and len(paths) >= max_images:
                break
    return paths


def scan_dataset(split_dirs=DATASET_SPLIT_DIRS, caption_csv=CAPTION_CSV, max_images=MAX_TRAIN_IMAGES):
    caption_map = load_caption_map(caption_csv)
    records = []
    found_split_images = False
    for split, split_dir in split_dirs.items():
        image_paths = list_image_paths_fast(split_dir, max_images=None)
        if image_paths:
            found_split_images = True
        for image_path in image_paths:
            records.append(ImageRecord(
                path=image_path,
                split=split,
                bucket="urban_pedestrian_scene",
                caption=build_caption(image_path, "urban_pedestrian_scene", caption_map),
                label_path=find_label_path(image_path),
                weather=None,
                timeofday=None,
                scene="urban",
            ))
            if max_images and len(records) >= max_images:
                return records

    if not found_split_images:
        for image_path in sorted(DATASET_ROOT.rglob("*")):
            if not image_path.is_file() or not is_source_image(image_path):
                continue
            records.append(ImageRecord(
                path=image_path,
                split=infer_split(image_path),
                bucket="urban_pedestrian_scene",
                caption=build_caption(image_path, "urban_pedestrian_scene", caption_map),
                label_path=find_label_path(image_path),
                weather=None,
                timeofday=None,
                scene="urban",
            ))
            if max_images and len(records) >= max_images:
                break
    return records

def load_records():
    records = scan_dataset()
    print(f"Scanned {len(records)} CityPersons images from {DATASET_ROOT}")
    if not records:
        print("No images found. Check DATASET_ROOT and Kaggle dataset mount.")
    else:
        split_counts = {}
        for record in records:
            split_counts[record.split] = split_counts.get(record.split, 0) + 1
        print("split counts:", split_counts)
        for record in records[:10]:
            print(record.split, record.bucket, record.path)
    return records

def summarize_citypersons_records(records):
    split_counts = {}
    for record in records:
        split_counts[record.split] = split_counts.get(record.split, 0) + 1
    print("CityPersons split counts:", split_counts)
    print("scene buckets:", sorted({record.bucket for record in records}))

def preview_prompt_samples(records, variants=("add_single_pedestrian", "add_two_pedestrians", "add_occluded_pedestrian", "add_distant_pedestrian"), n=3):
    for record in records[:n]:
        print("\nimage:", record.path.name)
        print("label:", {"weather": record.weather, "timeofday": record.timeofday, "scene": record.scene})
        print("train caption:", record.caption)
        for variant in variants:
            print(f"{variant} prompt:", build_generation_prompt(record, variant))

def preview_records(records, n=6):
    if not records:
        print("No dataset records to preview yet.")
        return
    sample = records[:n]
    cols = min(3, len(sample))
    rows = math.ceil(len(sample) / cols)
    plt.figure(figsize=(4 * cols, 4 * rows))
    for i, record in enumerate(sample, 1):
        image = Image.open(record.path).convert("RGB")
        plt.subplot(rows, cols, i)
        plt.imshow(image)
        plt.title(f"{record.split}/{record.bucket}\n{record.path.name[:32]}")
        plt.axis("off")
    plt.tight_layout()


In [ ]:
%%writefile /kaggle/working/sd35_utils.py
"""SD3.5 CityPersons augmentation: shared preprocessing, placement, masks, and scale helpers."""

import csv
import gc
import json
from datetime import datetime
import math
import numpy as np
import os
import random
import re
import statistics
import time
import warnings
from concurrent.futures import ThreadPoolExecutor, as_completed
from dataclasses import dataclass
from pathlib import Path
from threading import Lock
from typing import Optional

import matplotlib.pyplot as plt
import torch
from PIL import Image, ImageOps, ImageDraw, ImageFilter, ImageChops

try:
    import cv2
except ImportError:
    cv2 = None

from sd35_config import *
from sd35_data import ImageRecord

SEMANTIC_SEGMENTER = None
SEMANTIC_MASK_CACHE = {}

def load_source_image(path):
    return ImageOps.exif_transpose(Image.open(path)).convert("RGB")


def resize_center_crop(image, resolution=RESOLUTION):
    width, height = image.size
    scale = resolution / min(width, height)
    new_size = (round(width * scale), round(height * scale))
    image = image.resize(new_size, Image.BICUBIC)
    left = (image.width - resolution) // 2
    top = (image.height - resolution) // 2
    return image.crop((left, top, left + resolution, top + resolution))


def image_to_tensor(image, resolution=RESOLUTION, device="cuda", dtype=torch.float16):
    image = resize_center_crop(image, resolution)
    pixel_values = torch.tensor(list(image.getdata()), dtype=torch.float32).view(resolution, resolution, 3)
    pixel_values = pixel_values.permute(2, 0, 1).unsqueeze(0) / 127.5 - 1.0
    return pixel_values.to(device=device, dtype=dtype)


def clear_cuda():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
def records_by_split_and_bucket(records):
    grouped = {}
    for record in records:
        grouped.setdefault(record.split, {}).setdefault(record.bucket, []).append(record)
    return grouped


def choose_records_for_bucket(bucket_records, target_count, rng):
    if not bucket_records:
        return []
    if len(bucket_records) >= target_count:
        return rng.sample(bucket_records, target_count)
    return [rng.choice(bucket_records) for _ in range(target_count)]


def choose_record_for_variant(bucket_records, variant, rng):
    return rng.choice(bucket_records)


def generated_image_path(output_dir, record, variant, index):
    safe_variant = variant.replace("/", "_")
    file_name = f"{record.path.stem}_aug_{index:04d}_{safe_variant}.png"
    return Path(output_dir) / record.split / record.bucket / IMAGE_SUBDIR / file_name


def comparison_image_path(output_dir, record, variant, index):
    safe_variant = variant.replace("/", "_")
    file_name = f"{record.path.stem}_pair_{index:04d}_{safe_variant}.png"
    return Path(output_dir) / "comparison_pairs" / record.split / record.bucket / file_name


def save_comparison_pair(original, augmented, comparison_path, title):
    comparison_path = Path(comparison_path)
    comparison_path.parent.mkdir(parents=True, exist_ok=True)
    original = original.convert("RGB")
    augmented = augmented.convert("RGB").resize(original.size)
    title_h = 34
    label_h = 28
    width = original.width * 2
    height = original.height + title_h + label_h
    canvas = Image.new("RGB", (width, height), "white")
    draw = ImageDraw.Draw(canvas)
    draw.text((10, 8), title, fill=(0, 0, 0))
    draw.text((10, title_h + 6), "original", fill=(0, 0, 0))
    draw.text((original.width + 10, title_h + 6), "augmented", fill=(0, 0, 0))
    canvas.paste(original, (0, title_h + label_h))
    canvas.paste(augmented, (original.width, title_h + label_h))
    canvas.save(comparison_path)
    return comparison_path


def clamp_bbox(bbox, width, height):
    x1, y1, x2, y2 = bbox
    x1 = max(0, min(width - 1, int(round(x1))))
    y1 = max(0, min(height - 1, int(round(y1))))
    x2 = max(x1 + 1, min(width, int(round(x2))))
    y2 = max(y1 + 1, min(height, int(round(y2))))
    return (x1, y1, x2, y2)


def bbox_area(bbox):
    x1, y1, x2, y2 = bbox
    return max(0, x2 - x1) * max(0, y2 - y1)


def bbox_intersection_area(a, b):
    ax1, ay1, ax2, ay2 = a
    bx1, by1, bx2, by2 = b
    return max(0, min(ax2, bx2) - max(ax1, bx1)) * max(0, min(ay2, by2) - max(ay1, by1))


def person_overlap_depth_ok(front_bbox, occluded_bbox):
    """Allow overlap only when the occluded person is plausibly behind the pasted one."""
    inter = bbox_intersection_area(front_bbox, occluded_bbox)
    if inter <= 0:
        return True, 0.0
    front_area = max(1.0, bbox_area(front_bbox))
    occluded_area = max(1.0, bbox_area(occluded_bbox))
    overlap_ratio = inter / min(front_area, occluded_area)
    if not ALLOW_PERSON_PERSON_OVERLAP:
        return False, overlap_ratio
    if overlap_ratio > MAX_PERSON_PERSON_OVERLAP_RATIO:
        return False, overlap_ratio
    front_h = max(1.0, front_bbox[3] - front_bbox[1])
    occluded_h = max(1.0, occluded_bbox[3] - occluded_bbox[1])
    front_foot_y = float(front_bbox[3])
    occluded_foot_y = float(occluded_bbox[3])
    if front_h <= occluded_h:
        return False, overlap_ratio
    if occluded_h > front_h * OCCLUDED_PERSON_MAX_HEIGHT_RATIO:
        return False, overlap_ratio
    if front_foot_y < occluded_foot_y - OCCLUDED_PERSON_MAX_FOOT_Y_DELTA:
        return False, overlap_ratio
    if occluded_foot_y > front_foot_y + OCCLUDED_PERSON_MAX_FOOT_Y_DELTA:
        return False, overlap_ratio
    return True, overlap_ratio


def center_crop_geometry(original_size, resolution=RESOLUTION):
    width, height = original_size
    scale = resolution / min(width, height)
    resized_w = round(width * scale)
    resized_h = round(height * scale)
    crop_left = (resized_w - resolution) // 2
    crop_top = (resized_h - resolution) // 2
    return scale, crop_left, crop_top


def yolo_bbox_to_crop_bbox(parts, original_size, resolution=RESOLUTION, class_ids=None):
    class_id = int(float(parts[0]))
    if class_ids is not None and class_id not in class_ids:
        return None
    width, height = original_size
    xc, yc, bw, bh = [float(value) for value in parts[1:5]]
    x1 = (xc - bw / 2) * width
    y1 = (yc - bh / 2) * height
    x2 = (xc + bw / 2) * width
    y2 = (yc + bh / 2) * height
    scale, crop_left, crop_top = center_crop_geometry(original_size, resolution)
    crop_bbox = (
        x1 * scale - crop_left,
        y1 * scale - crop_top,
        x2 * scale - crop_left,
        y2 * scale - crop_top,
    )
    if crop_bbox[2] <= 0 or crop_bbox[0] >= resolution or crop_bbox[3] <= 0 or crop_bbox[1] >= resolution:
        return None
    clamped = clamp_bbox(crop_bbox, resolution, resolution)
    return clamped if bbox_area(clamped) > 0 else None


def load_yolo_bboxes_for_crop(record, original_size, class_ids, resolution=RESOLUTION):
    if not record.label_path or not Path(record.label_path).exists() or Path(record.label_path).suffix.lower() != ".txt":
        return []
    bboxes = []
    with Path(record.label_path).open("r", encoding="utf-8") as handle:
        for line in handle:
            parts = line.strip().split()
            if len(parts) < 5:
                continue
            try:
                bbox = yolo_bbox_to_crop_bbox(parts, original_size, resolution, class_ids=class_ids)
            except ValueError:
                continue
            if bbox:
                bboxes.append(bbox)
    return bboxes


def load_person_bboxes_for_crop(record, original_size, resolution=RESOLUTION):
    return load_yolo_bboxes_for_crop(record, original_size, PATCH_PERSON_CLASS_IDS, resolution=resolution)


def load_vehicle_bboxes_for_crop(record, original_size, resolution=RESOLUTION):
    return load_yolo_bboxes_for_crop(record, original_size, PATCH_VEHICLE_CLASS_IDS, resolution=resolution)


def load_semantic_segmenter(device=TRAIN_DEVICE):
    global SEMANTIC_SEGMENTER
    if not USE_SEMANTIC_PLACEMENT or SMART_PLACEMENT_VERSION != "v2":
        return None
    if SEMANTIC_SEGMENTER is False:
        return None
    if SEMANTIC_SEGMENTER is not None:
        return SEMANTIC_SEGMENTER
    try:
        import warnings
        from transformers import AutoImageProcessor, AutoModelForSemanticSegmentation
        with warnings.catch_warnings():
            warnings.filterwarnings(
                "ignore",
                message=r"The following named arguments are not valid for `SegformerImageProcessor.__init__`.*",
                category=UserWarning,
                module="transformers.image_processing_base",
            )
            image_processor = AutoImageProcessor.from_pretrained(
                SEMANTIC_SEGMENTATION_MODEL_ID,
                use_fast=False,
            )
        segmentation_model = AutoModelForSemanticSegmentation.from_pretrained(
            SEMANTIC_SEGMENTATION_MODEL_ID,
            low_cpu_mem_usage=False,
        ).to("cpu")
        segmentation_model.eval()
        SEMANTIC_SEGMENTER = {
            "processor": image_processor,
            "model": segmentation_model,
            "device": "cpu",  # keep SegFormer off GPU; SD3.5 already owns the GPUs/offload state
        }
        print(f"Loaded SegFormer semantic placement model on CPU: {SEMANTIC_SEGMENTATION_MODEL_ID}")
    except Exception as exc:
        SEMANTIC_SEGMENTER = False
        print("Semantic placement disabled; falling back to Smart Placement V1 rules.")
        print(type(exc).__name__, exc)
    return None if SEMANTIC_SEGMENTER is False else SEMANTIC_SEGMENTER


def label_matches(label, label_set):
    label = str(label).lower().replace("_", " ")
    return any(target in label for target in label_set)


def semantic_placement_masks(source, record, device=TRAIN_DEVICE):
    cache_key = (str(record.path), source.size)
    if cache_key in SEMANTIC_MASK_CACHE:
        return SEMANTIC_MASK_CACHE[cache_key]
    segmenter = load_semantic_segmenter(device=device)
    if segmenter is None:
        SEMANTIC_MASK_CACHE[cache_key] = None
        return None
    try:
        processor = segmenter["processor"]
        model = segmenter["model"]
        image = source.convert("RGB")
        inputs = processor(images=image, return_tensors="pt")
        inputs = {key: value.to("cpu") for key, value in inputs.items()}
        with torch.no_grad():
            outputs = model(**inputs)
        logits = torch.nn.functional.interpolate(
            outputs.logits,
            size=(source.height, source.width),
            mode="bilinear",
            align_corners=False,
        )
        semantic_ids = logits.argmax(dim=1)[0].detach().cpu().numpy()
    except Exception as exc:
        print("Semantic segmentation failed for this image; using V1 placement fallback for it.")
        print(type(exc).__name__, exc)
        SEMANTIC_MASK_CACHE[cache_key] = None
        return None

    id2label = getattr(model.config, "id2label", {}) or {}
    valid_arr = np.zeros((source.height, source.width), dtype=np.uint8)
    avoid_arr = np.zeros((source.height, source.width), dtype=np.uint8)
    valid_hits = 0
    for class_id in np.unique(semantic_ids):
        label = id2label.get(int(class_id), id2label.get(str(int(class_id)), str(class_id)))
        class_mask = semantic_ids == class_id
        if label_matches(label, VALID_PLACEMENT_LABELS):
            valid_arr[class_mask] = 255
            valid_hits += 1
        if label_matches(label, AVOID_PLACEMENT_LABELS):
            avoid_arr[class_mask] = 255
    if not valid_hits:
        print("SegFormer produced no valid road/sidewalk/terrain labels; using V1 placement fallback for this image.")
        masks = None
    else:
        masks = {
            "valid": Image.fromarray(valid_arr, mode="L"),
            "avoid": Image.fromarray(avoid_arr, mode="L"),
        }
    SEMANTIC_MASK_CACHE[cache_key] = masks
    return masks


def mask_coverage(mask, bbox):
    if mask is None:
        return 0.0
    x1, y1, x2, y2 = clamp_bbox(bbox, mask.width, mask.height)
    crop = np.asarray(mask.crop((x1, y1, x2, y2)), dtype=np.float32) / 255.0
    if crop.size == 0:
        return 0.0
    return float(crop.mean())


def foot_support_bbox(candidate):
    x1, y1, x2, y2 = candidate
    h = y2 - y1
    band_h = max(4, int(h * 0.16))
    return (x1, max(y1, y2 - band_h), x2, y2)


def perspective_scale_for_ground_y(ground_y, resolution=RESOLUTION):
    y_far = resolution * PATCH_ROAD_Y_RANGE[0]
    y_near = resolution * PATCH_ROAD_Y_RANGE[1]
    t = (ground_y - y_far) / max(1.0, y_near - y_far)
    t = max(0.0, min(1.0, t))
    return PERSPECTIVE_SCALE_FAR + t * (PERSPECTIVE_SCALE_NEAR - PERSPECTIVE_SCALE_FAR)


def reference_person_samples(existing_person_bboxes, resolution=RESOLUTION):
    samples = []
    for bbox in existing_person_bboxes:
        x1, y1, x2, y2 = bbox
        person_h = y2 - y1
        person_w = x2 - x1
        if person_h < REFERENCE_SCALE_MIN_PERSON_HEIGHT or person_h > REFERENCE_SCALE_MAX_PERSON_HEIGHT:
            continue
        if person_w <= 0 or person_h <= 0:
            continue
        aspect = person_w / max(1, person_h)
        if aspect < 0.12 or aspect > 0.85:
            continue
        samples.append({"ground_y": y2, "height": person_h, "width": person_w})
    return samples


def median(values):
    values = sorted(values)
    if not values:
        return None
    mid = len(values) // 2
    if len(values) % 2:
        return values[mid]
    return 0.5 * (values[mid - 1] + values[mid])


def fitted_reference_height_at_y(ground_y, samples, resolution=RESOLUTION):
    if len(samples) < REFERENCE_SCALE_MIN_SAMPLES:
        return None
    slopes = []
    for i, a in enumerate(samples):
        for b in samples[i + 1:]:
            dy = b["ground_y"] - a["ground_y"]
            if abs(dy) < REFERENCE_SCALE_MIN_Y_GAP:
                continue
            slope = (b["height"] - a["height"]) / dy
            if REFERENCE_SCALE_MIN_SLOPE <= slope <= REFERENCE_SCALE_MAX_SLOPE:
                slopes.append(slope)
    slope = median(slopes)
    if slope is None:
        return None
    intercepts = [sample["height"] - slope * sample["ground_y"] for sample in samples]
    intercept = median(intercepts)
    predicted = slope * ground_y + intercept
    return predicted if predicted > 0 else None


def single_reference_height_at_y(ground_y, samples, resolution=RESOLUTION):
    if not samples:
        return None
    nearest = min(samples, key=lambda sample: abs(sample["ground_y"] - ground_y))
    if abs(nearest["ground_y"] - ground_y) > REFERENCE_SCALE_MAX_Y_DISTANCE:
        return None
    ref_scale = perspective_scale_for_ground_y(nearest["ground_y"], resolution=resolution)
    target_scale = perspective_scale_for_ground_y(ground_y, resolution=resolution)
    return nearest["height"] * target_scale / max(1e-6, ref_scale)


def robust_reference_height_at_y(ground_y, existing_person_bboxes, resolution=RESOLUTION):
    samples = reference_person_samples(existing_person_bboxes, resolution=resolution)
    if not samples:
        return None
    fitted = fitted_reference_height_at_y(ground_y, samples, resolution=resolution)
    if fitted is not None:
        return fitted
    return single_reference_height_at_y(ground_y, samples, resolution=resolution)


def reference_vehicle_samples(existing_vehicle_bboxes, resolution=RESOLUTION):
    samples = []
    for bbox in existing_vehicle_bboxes:
        x1, y1, x2, y2 = bbox
        vehicle_h = y2 - y1
        vehicle_w = x2 - x1
        if vehicle_h < CAR_REFERENCE_MIN_HEIGHT or vehicle_h > CAR_REFERENCE_MAX_HEIGHT:
            continue
        if vehicle_w <= 0 or vehicle_h <= 0:
            continue
        aspect = vehicle_w / max(1, vehicle_h)
        if aspect < 1.0 or aspect > 4.8:
            continue
        same_depth_ratio = max(CAR_HEIGHT_TO_PERSON_HEIGHT_MIN_RATIO, min(CAR_HEIGHT_TO_PERSON_HEIGHT_MAX_RATIO, CAR_HEIGHT_TO_PERSON_HEIGHT_RATIO))
        person_equivalent_h = vehicle_h * same_depth_ratio
        samples.append({"ground_y": y2, "height": person_equivalent_h, "width": vehicle_w})
    return samples


def single_vehicle_reference_height_at_y(ground_y, samples):
    close = [sample for sample in samples if abs(sample["ground_y"] - ground_y) <= CAR_REFERENCE_MAX_Y_DISTANCE]
    if not close:
        return None
    weights = []
    heights = []
    for sample in close:
        distance = abs(sample["ground_y"] - ground_y)
        weights.append(1.0 / (1.0 + distance))
        heights.append(sample["height"])
    total_weight = sum(weights)
    if total_weight <= 0:
        return None
    return sum(height * weight for height, weight in zip(heights, weights)) / total_weight


def robust_vehicle_reference_height_at_y(ground_y, existing_vehicle_bboxes, resolution=RESOLUTION):
    samples = reference_vehicle_samples(existing_vehicle_bboxes, resolution=resolution)
    if not samples:
        return None
    fitted = fitted_reference_height_at_y(ground_y, samples, resolution=resolution)
    if fitted is not None:
        return fitted
    return single_vehicle_reference_height_at_y(ground_y, samples)


def combine_reference_heights(person_reference_h, vehicle_reference_h):
    if person_reference_h is not None and vehicle_reference_h is not None:
        return (1.0 - CAR_REFERENCE_SCALE_BLEND) * person_reference_h + CAR_REFERENCE_SCALE_BLEND * vehicle_reference_h
    if person_reference_h is not None:
        return person_reference_h
    return vehicle_reference_h


def fallback_person_height_for_variant(variant, resolution=RESOLUTION, ground_y=None):
    base_heights = {
        "add_single_pedestrian": 96,
        "add_two_pedestrians": 98,
        "add_small_group": 102,
        "add_occluded_pedestrian": 90,
        "add_distant_pedestrian": 68,
        "add_near_pedestrian": 150,
    }
    height = base_heights.get(variant, 96)
    if ground_y is not None:
        height *= perspective_scale_for_ground_y(ground_y, resolution=resolution)
        y_norm = max(0.0, min(1.0, ground_y / max(1, resolution)))
        max_ratio = 0.62 if variant == "add_near_pedestrian" else 0.50
        if variant == "add_distant_pedestrian":
            max_ratio = 0.26
        # Keep the envelope plausible for CityPersons perspective; SD3.5 still has slack inside it.
        height = min(height, resolution * max_ratio)
        if y_norm < 0.74:
            height = min(height, resolution * 0.34)
    return height


def min_target_height_ratio_for_variant(variant):
    if variant == "add_near_pedestrian":
        return MIN_ACCEPTED_NEAR_HEIGHT_RATIO
    if variant == "add_distant_pedestrian":
        return MIN_ACCEPTED_DISTANT_HEIGHT_RATIO
    if variant == "add_occluded_pedestrian":
        return 0.075
    if variant in {"add_two_pedestrians", "add_small_group"}:
        return 0.085
    return MIN_ACCEPTED_SINGLE_HEIGHT_RATIO


def variant_scale_multiplier(variant):
    if variant == "add_distant_pedestrian":
        return 0.82
    if variant == "add_near_pedestrian":
        return NEAR_PERSON_SCALE_MULTIPLIER
    if variant == "add_occluded_pedestrian":
        return 0.92
    return 1.0


def variant_insert_size(variant, resolution=RESOLUTION, ground_x=None, ground_y=None, existing_person_bboxes=None, existing_vehicle_bboxes=None, depth_map=None):
    if depth_map is not None and ground_x is not None and ground_y is not None:
        fallback_h = expected_person_height_from_depth(ground_x, ground_y, depth_map, resolution, variant)
    else:
        fallback_h = fallback_person_height_for_variant(variant, resolution=resolution, ground_y=ground_y)
    person_reference_h = None
    vehicle_reference_h = None
    if USE_REFERENCE_PERSON_SCALE and existing_person_bboxes and ground_y is not None:
        person_reference_h = robust_reference_height_at_y(ground_y, existing_person_bboxes, resolution=resolution)
    if existing_vehicle_bboxes and ground_y is not None:
        vehicle_reference_h = robust_vehicle_reference_height_at_y(ground_y, existing_vehicle_bboxes, resolution=resolution)
    reference_h = combine_reference_heights(person_reference_h, vehicle_reference_h)
    if reference_h is not None:
        min_h = fallback_h * REFERENCE_SCALE_MIN_FACTOR
        max_h = fallback_h * REFERENCE_SCALE_MAX_FACTOR
        reference_h = max(min_h, min(reference_h, max_h))
        target_h = REFERENCE_SCALE_BLEND * reference_h + (1.0 - REFERENCE_SCALE_BLEND) * fallback_h
    else:
        target_h = fallback_h
    target_h *= max(variant_scale_multiplier(variant), PERSON_TARGET_SCALE_MULTIPLIER)
    if ground_y is not None:
        y_norm = max(0.0, min(1.0, ground_y / max(1, resolution)))
        max_h = resolution * (0.28 + 0.34 * y_norm)
        if variant == "add_distant_pedestrian":
            max_h = min(max_h, resolution * 0.25)
        elif variant == "add_near_pedestrian":
            max_h = min(max_h, resolution * 0.72)
        min_h = resolution * min_target_height_ratio_for_variant(variant)
        target_h = max(min_h, min(target_h, max_h))
    if variant in {"add_two_pedestrians", "add_small_group"}:
        count = 2 if variant == "add_two_pedestrians" else 3
        target_w = target_h * PERSON_ASPECT_RATIO * count * 0.86
    else:
        target_w = target_h * PERSON_ASPECT_RATIO
    if FLEXIBLE_SCALE_INPAINT:
        width = target_w * SCALE_ENVELOPE_WIDTH_MULT
        height = target_h * SCALE_ENVELOPE_HEIGHT_MULT
    else:
        width = target_w
        height = target_h
    width = round(max(28, min(width, resolution - 2 * INSERTION_EDGE_MARGIN)))
    height = round(max(46, min(height, resolution - 2 * INSERTION_EDGE_MARGIN)))
    return width, height


def expand_bbox_with_context(bbox, resolution=RESOLUTION, context_ratio=PATCH_CONTEXT_RATIO):
    x1, y1, x2, y2 = bbox
    bw = x2 - x1
    bh = y2 - y1
    pad = int(max(bw, bh) * context_ratio)
    x1, y1, x2, y2 = clamp_bbox((x1 - pad, y1 - pad, x2 + pad, y2 + pad), resolution, resolution)
    patch_w = x2 - x1
    patch_h = y2 - y1
    if patch_w < PATCH_MIN_SIZE:
        extra = (PATCH_MIN_SIZE - patch_w) // 2
        x1, y1, x2, y2 = clamp_bbox((x1 - extra, y1, x2 + extra, y2), resolution, resolution)
    if patch_h < PATCH_MIN_SIZE:
        extra = (PATCH_MIN_SIZE - patch_h) // 2
        x1, y1, x2, y2 = clamp_bbox((x1, y1 - extra, x2, y2 + extra), resolution, resolution)
    return (x1, y1, x2, y2)


def candidate_insertion_score(candidate, existing_person_bboxes, existing_vehicle_bboxes=None, semantic_masks=None, resolution=RESOLUTION, placement_target=None):
    x1, y1, x2, y2 = candidate
    width = x2 - x1
    height = y2 - y1
    area = max(1, width * height)
    cx = (x1 + x2) / 2
    ground_y = y2
    y_min = resolution * PATCH_ROAD_Y_RANGE[0]
    y_max = resolution * PATCH_ROAD_Y_RANGE[1]
    y_mid = y_min + 0.68 * (y_max - y_min)
    road_score = 1.0 - min(1.0, abs(ground_y - y_mid) / max(1.0, y_max - y_min))
    center_score = 1.0 - min(1.0, abs(cx - resolution / 2) / (resolution / 2))
    margin = min(x1, y1, resolution - x2, resolution - y2)
    margin_score = min(1.0, max(0.0, margin / INSERTION_EDGE_MARGIN))
    person_overlap_ratio = 0.0
    for bbox in existing_person_bboxes:
        depth_ok, overlap = person_overlap_depth_ok(candidate, bbox)
        person_overlap_ratio = max(person_overlap_ratio, overlap)
        if not depth_ok:
            return -1e9
    vehicle_overlap_ratio = 0.0
    if existing_vehicle_bboxes:
        for bbox in existing_vehicle_bboxes:
            vehicle_overlap_ratio += bbox_intersection_area(candidate, bbox) / area
        vehicle_overlap_ratio = min(1.0, vehicle_overlap_ratio)
    size_ratio = height / resolution
    if size_ratio < 0.10:
        size_score = size_ratio / 0.10
    elif size_ratio > 0.42:
        size_score = max(0.0, 1.0 - (size_ratio - 0.42) / 0.20)
    else:
        size_score = 1.0
    slot_score = 0.0
    if placement_target is not None:
        target_x, target_y = placement_target
        dx = abs(cx / resolution - target_x)
        dy = abs(ground_y / resolution - target_y)
        slot_score = max(0.0, 1.0 - (dx / 0.24 + dy / 0.18) / 2.0)
    score = (
        1.10 * road_score
        + INSERTION_CENTER_BIAS * center_score
        + PLACEMENT_SLOT_BIAS * slot_score
        + 0.35 * margin_score
        + 0.45 * size_score
    )
    if person_overlap_ratio > 0:
        score -= INSERTION_OVERLAP_PENALTY * 0.35 * person_overlap_ratio
    if ALLOW_PERSON_VEHICLE_OVERLAP and vehicle_overlap_ratio > 0:
        score += VEHICLE_OVERLAP_FRONT_LAYER_BONUS * min(vehicle_overlap_ratio, MAX_VEHICLE_OVERLAP_RATIO)
        if vehicle_overlap_ratio > MAX_VEHICLE_OVERLAP_RATIO:
            score -= INSERTION_OVERLAP_PENALTY * (vehicle_overlap_ratio - MAX_VEHICLE_OVERLAP_RATIO)
    elif vehicle_overlap_ratio > 0:
        score -= INSERTION_OVERLAP_PENALTY * vehicle_overlap_ratio
    if REQUIRE_SEMANTIC_PLACEMENT and semantic_masks is None:
        return -1e9
    if semantic_masks:
        valid_mask = semantic_masks.get("valid")
        avoid_mask = semantic_masks.get("avoid")
        foot_bbox = foot_support_bbox(candidate)
        foot_score = mask_coverage(valid_mask, foot_bbox)
        foot_avoid_score = mask_coverage(avoid_mask, foot_bbox)
        body_valid_score = mask_coverage(valid_mask, candidate)
        avoid_score = mask_coverage(avoid_mask, candidate)
        if foot_score < MIN_FOOT_SUPPORT:
            return -1e9
        if foot_avoid_score > MAX_FOOT_AVOID_SUPPORT:
            return -1e9
        if REQUIRE_SEMANTIC_PLACEMENT and body_valid_score < MIN_BODY_VALID_SUPPORT:
            return -1e9
        if REQUIRE_SEMANTIC_PLACEMENT and avoid_score > MAX_BODY_AVOID_SUPPORT:
            return -1e9
        score += SEMANTIC_FOOT_WEIGHT * foot_score
        score += 0.45 * body_valid_score
        score -= SEMANTIC_AVOID_PENALTY * avoid_score
    return score


def ground_y_range_for_variant(variant, height):
    y_min = int(height * PATCH_ROAD_Y_RANGE[0])
    y_max = int(height * PATCH_ROAD_Y_RANGE[1])
    span = max(1, y_max - y_min)
    if variant == "add_distant_pedestrian":
        return y_min, y_min + int(span * 0.34)
    if variant == "add_near_pedestrian":
        return y_min + int(span * 0.58), y_max
    if variant == "add_occluded_pedestrian":
        return y_min + int(span * 0.20), y_min + int(span * 0.78)
    return y_min, y_max


def placement_target_for_variant(variant, rng):
    if variant == "add_distant_pedestrian":
        y_choices = PLACEMENT_SLOT_YS[:2]
    elif variant == "add_near_pedestrian":
        y_choices = PLACEMENT_SLOT_YS[-2:]
    else:
        y_choices = PLACEMENT_SLOT_YS
    target_x = rng.choice(PLACEMENT_SLOT_XS) + rng.uniform(-PLACEMENT_SLOT_JITTER, PLACEMENT_SLOT_JITTER)
    target_y = rng.choice(y_choices) + rng.uniform(-PLACEMENT_SLOT_JITTER, PLACEMENT_SLOT_JITTER)
    target_x = max(0.08, min(0.92, target_x))
    target_y = max(PATCH_ROAD_Y_RANGE[0], min(PATCH_ROAD_Y_RANGE[1], target_y))
    return target_x, target_y


def sample_ground_y_for_variant(variant, y_min, y_max, rng):
    if y_max <= y_min:
        return y_min
    u = rng.random()
    if variant == "add_distant_pedestrian":
        u = 0.90 * (u ** 1.35)
    elif variant == "add_near_pedestrian":
        u = 0.12 + 0.76 * (u ** 1.25)
    else:
        u = 0.06 + 0.84 * (u ** 1.15)
    return int(round(y_min + max(0.0, min(0.92, u)) * (y_max - y_min)))


def find_insertion_region(record, source, variant, rng, device=TRAIN_DEVICE, return_metadata=False, depth_map=None):
    width, height = source.size
    original = load_source_image(record.path)
    existing_person_bboxes = load_person_bboxes_for_crop(record, original.size, resolution=width)
    existing_vehicle_bboxes = load_vehicle_bboxes_for_crop(record, original.size, resolution=width)
    semantic_masks = semantic_placement_masks(source, record, device=device)
    y_min, y_max = ground_y_range_for_variant(variant, height)
    placement_target = placement_target_for_variant(variant, rng)
    best_bbox = None
    best_score = -1e9
    best_meta = None
    for _ in range(PATCH_MAX_PLACEMENT_TRIES):
        ground_y = sample_ground_y_for_variant(variant, y_min, y_max, rng)
        candidate_x = rng.randint(INSERTION_EDGE_MARGIN, width - INSERTION_EDGE_MARGIN)
        insert_w, insert_h = variant_insert_size(
            variant,
            resolution=width,
            ground_x=candidate_x,
            ground_y=ground_y,
            existing_person_bboxes=existing_person_bboxes,
            existing_vehicle_bboxes=existing_vehicle_bboxes,
            depth_map=depth_map
        )
        if width - insert_w - INSERTION_EDGE_MARGIN <= INSERTION_EDGE_MARGIN:
            continue
        x1 = max(INSERTION_EDGE_MARGIN, min(width - insert_w - INSERTION_EDGE_MARGIN, candidate_x - insert_w // 2))
        y1 = max(INSERTION_EDGE_MARGIN, min(height - insert_h - INSERTION_EDGE_MARGIN, ground_y - insert_h))
        candidate = (x1, y1, x1 + insert_w, y1 + insert_h)
        score = candidate_insertion_score(
            candidate,
            existing_person_bboxes,
            existing_vehicle_bboxes=existing_vehicle_bboxes,
            semantic_masks=semantic_masks,
            resolution=width,
            placement_target=placement_target,
        )
        if score > best_score:
            best_bbox = candidate
            best_score = score
            expected_person_h = insert_h / max(1e-6, SCALE_ENVELOPE_HEIGHT_MULT if FLEXIBLE_SCALE_INPAINT else 1.0)
            expected_person_w = insert_w / max(1e-6, SCALE_ENVELOPE_WIDTH_MULT if FLEXIBLE_SCALE_INPAINT else 1.0)
            best_meta = {
                "expected_person_height": expected_person_h,
                "expected_person_width": expected_person_w,
                "insert_width": insert_w,
                "insert_height": insert_h,
                "ground_y": ground_y,
            }
    if best_score <= MIN_ACCEPTED_PLACEMENT_SCORE:
        best_bbox, best_meta = None, None
    if return_metadata:
        return best_bbox, best_meta
    return best_bbox

def feather_mask(size, radius=PATCH_FEATHER_RADIUS):
    width, height = size
    mask = Image.new("L", (width, height), 0)
    inset = max(1, radius)
    draw = ImageDraw.Draw(mask)
    draw.rectangle((inset, inset, width - inset, height - inset), fill=255)
    return mask.filter(ImageFilter.GaussianBlur(radius=radius))


def guide_bboxes_for_variant(insert_bbox, variant):
    x1, y1, x2, y2 = insert_bbox
    width = x2 - x1
    height = y2 - y1
    if variant == "add_two_pedestrians":
        gap = max(6, int(width * GROUP_PERSON_GAP_RATIO))
        person_w = max(22, int((width - gap) / 2))
        left_h = height
        right_h = int(height * (1.0 - GROUP_PERSON_HEIGHT_JITTER))
        return [
            (x1, y2 - left_h, x1 + person_w, y2),
            (x2 - person_w, y2 - right_h, x2, y2),
        ]
    if variant == "add_small_group":
        gap = max(4, int(width * GROUP_PERSON_GAP_RATIO * 0.75))
        person_w = max(18, int((width - 2 * gap) / 3))
        heights = [int(height * 0.90), height, int(height * 0.82)]
        starts = [x1, x1 + person_w + gap, x2 - person_w]
        return [
            (starts[index], y2 - heights[index], starts[index] + person_w, y2)
            for index in range(3)
        ]
    if variant == "add_occluded_pedestrian":
        inset = max(2, int(width * 0.08))
        return [(x1 + inset, y1 + int(height * 0.08), x2 - inset, y2)]
    return [insert_bbox]


def draw_person_guide_on_patch(patch, patch_bbox, insert_bbox, variant):
    if not DRAW_INSERTION_GUIDE:
        return patch
    px1, py1, _, _ = patch_bbox
    overlay = Image.new("RGBA", patch.size, (0, 0, 0, 0))
    draw = ImageDraw.Draw(overlay)
    for bbox in guide_bboxes_for_variant(insert_bbox, variant):
        x1, y1, x2, y2 = [int(v) for v in bbox]
        x1 -= px1
        x2 -= px1
        y1 -= py1
        y2 -= py1
        bw = max(8, x2 - x1)
        bh = max(24, y2 - y1)
        cx = x1 + bw // 2
        head_r = max(4, int(bw * 0.16))
        head_y = y1 + max(4, int(bh * 0.10))
        shoulder_y = y1 + int(bh * 0.26)
        hip_y = y1 + int(bh * 0.60)
        foot_y = y2
        alpha = int(255 * INSERTION_GUIDE_ALPHA)
        color = (25, 25, 25, alpha)
        outline = (245, 245, 245, max(40, int(alpha * 0.34)))
        draw.ellipse((cx - head_r, head_y, cx + head_r, head_y + 2 * head_r), fill=color)
        draw.rounded_rectangle((cx - int(bw * 0.18), shoulder_y, cx + int(bw * 0.18), hip_y), radius=4, fill=color)
        draw.line((cx - int(bw * 0.10), hip_y, cx - int(bw * 0.22), foot_y), fill=color, width=max(4, int(bw * 0.09)))
        draw.line((cx + int(bw * 0.10), hip_y, cx + int(bw * 0.22), foot_y), fill=color, width=max(4, int(bw * 0.09)))
        draw.line((cx - int(bw * 0.18), shoulder_y + 6, cx - int(bw * 0.30), hip_y - 4), fill=color, width=max(2, int(bw * 0.04)))
        draw.line((cx + int(bw * 0.18), shoulder_y + 6, cx + int(bw * 0.30), hip_y - 4), fill=color, width=max(2, int(bw * 0.04)))
        draw.line((x1 + int(bw * 0.12), foot_y, x2 - int(bw * 0.12), foot_y), fill=outline, width=max(2, int(bw * 0.05)))
        draw.rounded_rectangle((x1, y1, x2, y2), radius=3, outline=outline, width=max(1, int(bw * 0.025)))
    if INSERTION_GUIDE_BLUR:
        overlay = overlay.filter(ImageFilter.GaussianBlur(radius=INSERTION_GUIDE_BLUR))
    return Image.alpha_composite(patch.convert("RGBA"), overlay).convert("RGB")


def save_patch_debug_strip(record, variant, seed, source_patch, guided_patch, generated_patch, final_patch, patch_bbox, insert_bbox, debug_index=None):
    if not SAVE_PATCH_DEBUG:
        return ""
    if debug_index is not None and debug_index >= PATCH_DEBUG_MAX_ITEMS:
        return ""
    debug_dir = PATCH_DEBUG_DIR / record.split / record.bucket
    debug_dir.mkdir(parents=True, exist_ok=True)
    panels = [
        ("source", source_patch.convert("RGB")),
        ("guide", guided_patch.convert("RGB")),
        ("generated", generated_patch.convert("RGB")),
        ("final", final_patch.convert("RGB")),
    ]
    width = max(panel.width for _, panel in panels)
    height = max(panel.height for _, panel in panels)
    label_h = 24
    canvas = Image.new("RGB", (width * len(panels), height + label_h), "white")
    draw = ImageDraw.Draw(canvas)
    for index, (label, image) in enumerate(panels):
        image = image.resize((width, height))
        x = index * width
        draw.text((x + 8, 6), label, fill=(0, 0, 0))
        canvas.paste(image, (x, label_h))
    safe_variant = variant.replace("/", "_")
    debug_path = debug_dir / f"{record.path.stem}_debug_{seed}_{safe_variant}.png"
    canvas.save(debug_path)
    return str(debug_path)


def human_mask_for_bbox(image_size, insert_bbox, variant):
    width, height = image_size
    mask = Image.new("L", image_size, 0)
    draw = ImageDraw.Draw(mask)
    for bbox in guide_bboxes_for_variant(insert_bbox, variant):
        x1, y1, x2, y2 = [int(v) for v in bbox]
        x1 = max(0, x1 - HUMAN_MASK_PADDING)
        y1 = max(0, y1 - HUMAN_MASK_PADDING)
        x2 = min(width, x2 + HUMAN_MASK_PADDING)
        y2 = min(height, y2 + HUMAN_MASK_PADDING)
        bw = max(8, x2 - x1)
        bh = max(24, y2 - y1)
        cx = x1 + bw // 2
        head_r = max(4, int(bw * 0.18))
        head_y = y1 + max(2, int(bh * 0.05))
        shoulder_y = y1 + int(bh * 0.24)
        hip_y = y1 + int(bh * 0.60)
        foot_y = y2
        draw.ellipse((cx - head_r, head_y, cx + head_r, head_y + 2 * head_r), fill=255)
        draw.rounded_rectangle((cx - int(bw * 0.28), shoulder_y, cx + int(bw * 0.28), hip_y), radius=6, fill=255)
        draw.line((cx - int(bw * 0.10), hip_y, cx - int(bw * 0.22), foot_y), fill=255, width=max(6, int(bw * 0.14)))
        draw.line((cx + int(bw * 0.10), hip_y, cx + int(bw * 0.22), foot_y), fill=255, width=max(6, int(bw * 0.14)))
        draw.line((cx - int(bw * 0.18), shoulder_y + 4, cx - int(bw * 0.30), hip_y - 4), fill=255, width=max(4, int(bw * 0.09)))
        draw.line((cx + int(bw * 0.18), shoulder_y + 4, cx + int(bw * 0.30), hip_y - 4), fill=255, width=max(4, int(bw * 0.09)))
    if HUMAN_MASK_BLUR_RADIUS:
        mask = mask.filter(ImageFilter.GaussianBlur(radius=HUMAN_MASK_BLUR_RADIUS))
    return mask


def bbox_mask_for_bbox(image_size, insert_bbox, variant=None, padding=BBOX_MASK_PADDING, blur=BBOX_MASK_BLUR_RADIUS):
    width, height = image_size
    mask = Image.new("L", image_size, 0)
    draw = ImageDraw.Draw(mask)
    bboxes = guide_bboxes_for_variant(insert_bbox, variant) if variant else [insert_bbox]
    for bbox in bboxes:
        x1, y1, x2, y2 = bbox
        x1 = max(0, int(x1 - padding))
        y1 = max(0, int(y1 - padding))
        x2 = min(width, int(x2 + padding))
        y2 = min(height, int(y2 + padding))
        radius = min(BBOX_MASK_RADIUS, max(2, (x2 - x1) // 5), max(2, (y2 - y1) // 5))
        draw.rounded_rectangle((x1, y1, x2, y2), radius=radius, fill=255)
    if blur:
        mask = mask.filter(ImageFilter.GaussianBlur(radius=blur))
    return mask


def prepare_inpaint_source(source, mask_image, insert_bbox):
    prepared = source.copy().convert("RGB")
    if PERSON_GENERATION_ONLY_MODE:
        # Keep local road/sidewalk texture so img2img understands where the feet should land.
        context = Image.blend(
            prepared,
            Image.new("RGB", prepared.size, (118, 118, 118)),
            PERSON_GENERATION_CONTEXT_DARKEN,
        )
        blur_source = prepared.filter(ImageFilter.GaussianBlur(radius=6)).convert("RGB")
        neutral = Image.new("RGB", prepared.size, (132, 132, 132))
        target_canvas = Image.blend(blur_source, neutral, PERSON_GENERATION_NEUTRAL_STRENGTH)
        target_canvas = Image.blend(prepared, target_canvas, 0.72)
        prepared = Image.composite(target_canvas, context, mask_image)
        return prepared

    blur_source = source.filter(ImageFilter.GaussianBlur(radius=20)).convert("RGB")
    mask_bbox = mask_image.getbbox() or insert_bbox
    x1, y1, x2, y2 = mask_bbox
    fill_crop = blur_source.crop((x1, y1, x2, y2))
    neutral = Image.new("RGB", fill_crop.size, (132, 132, 132))
    fill_crop = Image.blend(fill_crop, neutral, PERSON_GENERATION_NEUTRAL_STRENGTH)
    prepared.paste(fill_crop, (x1, y1), mask_image.crop((x1, y1, x2, y2)))
    return prepared


def masked_rgb_mae(image_a, image_b, mask_image):
    a = np.asarray(image_a.convert("RGB"), dtype=np.float32) / 255.0
    b = np.asarray(image_b.convert("RGB"), dtype=np.float32) / 255.0
    mask = np.asarray(mask_image.convert("L"), dtype=np.float32) / 255.0
    active = mask > 0.20
    if not np.any(active):
        return 0.0
    return float(np.mean(np.abs(a[active] - b[active])))


def masked_rgb_mae_255(image_a, image_b, mask_image):
    return 255.0 * masked_rgb_mae(image_a, image_b, mask_image)


def mask_area_ratio(mask_image, threshold=0.20):
    mask = np.asarray(mask_image.convert("L"), dtype=np.float32) / 255.0
    return float(np.mean(mask > threshold))


def fill_binary_mask_holes(arr):
    if cv2 is None or not FILL_PERSON_MASK_HOLES:
        return arr
    ys, xs = np.where(arr > 0)
    if len(xs) == 0 or len(ys) == 0:
        return arr
    x1, x2 = int(xs.min()), int(xs.max()) + 1
    y1, y2 = int(ys.min()), int(ys.max()) + 1
    roi = arr[y1:y2, x1:x2]
    h, w = roi.shape[:2]
    if h <= 2 or w <= 2:
        return arr

    # Fill only holes enclosed by the foreground silhouette. Padding prevents
    # floodFill from treating background beside a border-touching person as a hole.
    padded = np.pad(roi, ((1, 1), (1, 1)), mode="constant", constant_values=0)
    flood = padded.copy()
    flood_mask = np.zeros((flood.shape[0] + 2, flood.shape[1] + 2), dtype=np.uint8)
    cv2.floodFill(flood, flood_mask, (0, 0), 255)
    holes = cv2.bitwise_not(flood)[1:-1, 1:-1]
    filled_roi = cv2.bitwise_or(roi, holes)
    out = arr.copy()
    out[y1:y2, x1:x2] = filled_roi
    return out


def clean_binary_person_mask(mask, keep_components=1):
    mask_l = mask.convert("L")
    arr = (np.asarray(mask_l, dtype=np.uint8) >= PERSON_PASTE_HARD_THRESHOLD).astype(np.uint8) * 255
    if cv2 is not None:
        kernel = np.ones((3, 3), np.uint8)
        arr = cv2.morphologyEx(arr, cv2.MORPH_CLOSE, kernel, iterations=2)
        if PERSON_MASK_ERODE_PIXELS > 0:
            arr = cv2.erode(arr, kernel, iterations=PERSON_MASK_ERODE_PIXELS)
        num_labels, labels, stats, _ = cv2.connectedComponentsWithStats((arr > 0).astype(np.uint8), 8)
        if num_labels > 1:
            component_ids = sorted(range(1, num_labels), key=lambda i: stats[i, cv2.CC_STAT_AREA], reverse=True)
            largest_area = float(stats[component_ids[0], cv2.CC_STAT_AREA]) if component_ids else 0.0
            min_area = max(8.0, largest_area * ACCESSORY_MIN_COMPONENT_AREA_RATIO)
            keep_limit = max(int(keep_components), int(ACCESSORY_KEEP_COMPONENTS))
            keep = {
                component_id
                for component_id in component_ids[:keep_limit]
                if stats[component_id, cv2.CC_STAT_AREA] >= min_area
            }
            if keep:
                arr = np.where(np.isin(labels, list(keep)), 255, 0).astype(np.uint8)
        arr = cv2.morphologyEx(arr, cv2.MORPH_CLOSE, kernel, iterations=1)
        arr = fill_binary_mask_holes(arr)
        return Image.fromarray(arr, mode="L")
    hard = Image.fromarray(arr, mode="L")
    if PERSON_MASK_ERODE_PIXELS > 0:
        hard = hard.filter(ImageFilter.MinFilter(PERSON_MASK_ERODE_PIXELS * 2 + 1))
    return hard


def prepare_person_paste_mask(person_mask, size):
    mask = person_mask.resize(size, Image.NEAREST).convert("L")
    mask = clean_binary_person_mask(mask, keep_components=ACCESSORY_KEEP_COMPONENTS)
    trim_px = max(0, int(PERSON_MASK_TRIM_FRINGE_PIXELS))
    if trim_px > 0:
        # Remove the generated-background fringe at the silhouette boundary,
        # then restore one soft pixel so accessories are not aggressively cut.
        eroded = mask.filter(ImageFilter.MinFilter(trim_px * 2 + 1))
        mask = eroded.filter(ImageFilter.MaxFilter(trim_px * 2 + 1))
    if PERSON_PASTE_FEATHER_RADIUS and PERSON_PASTE_FEATHER_RADIUS > 0:
        soft = mask.filter(ImageFilter.GaussianBlur(radius=PERSON_PASTE_FEATHER_RADIUS))
        mask = ImageChops.multiply(mask, soft)
    return mask


def constrain_mask_to_bbox(mask, bbox, padding_ratio=0.12, min_padding=6):
    bbox_mask = Image.new("L", mask.size, 0)
    draw = ImageDraw.Draw(bbox_mask)
    x1, y1, x2, y2 = bbox
    pad = max(min_padding, int(round(max(x2 - x1, y2 - y1) * padding_ratio)))
    padded = clamp_bbox((x1 - pad, y1 - pad, x2 + pad, y2 + pad), mask.size[0], mask.size[1])
    draw.rectangle(tuple(int(round(v)) for v in padded), fill=255)
    return ImageChops.multiply(mask.convert("L"), bbox_mask)


def mask_outside_bbox_ratio(mask, bbox, padding_ratio=0.12, min_padding=6):
    mask_l = mask.convert("L")
    mask_arr = np.asarray(mask_l, dtype=np.float32) / 255.0
    active = mask_arr > 0.20
    if not np.any(active):
        return 1.0
    x1, y1, x2, y2 = bbox
    pad = max(min_padding, int(round(max(x2 - x1, y2 - y1) * padding_ratio)))
    px1, py1, px2, py2 = clamp_bbox((x1 - pad, y1 - pad, x2 + pad, y2 + pad), mask_l.size[0], mask_l.size[1])
    keep = np.zeros(active.shape, dtype=bool)
    keep[int(py1):int(py2), int(px1):int(px2)] = True
    return float(np.mean(active & ~keep) / max(1e-6, np.mean(active)))


def expand_bbox_for_detection(bbox, image_size, factor=CONTEXT_TARGET_BBOX_EXPAND_FOR_DETECTION):
    width, height = image_size
    x1, y1, x2, y2 = bbox
    cx = (x1 + x2) / 2
    cy = (y1 + y2) / 2
    bw = (x2 - x1) * factor
    bh = (y2 - y1) * factor
    return clamp_bbox((cx - bw / 2, cy - bh / 2, cx + bw / 2, cy + bh / 2), width, height)

DEPTH_PIPELINE = None

def load_depth_pipeline(device="cpu"):
    global DEPTH_PIPELINE
    if DEPTH_PIPELINE is not None:
        return DEPTH_PIPELINE
    try:
        from transformers import pipeline
        DEPTH_PIPELINE = pipeline(
            task="depth-estimation",
            model="LiheYoung/depth-anything-small-hf",
            device=device if "cuda" in str(device) else "cpu"
        )
        print("Loaded depth estimator model: LiheYoung/depth-anything-small-hf")
    except Exception as exc:
        print("Failed to load depth estimator:", type(exc).__name__, exc)
        DEPTH_PIPELINE = None
    return DEPTH_PIPELINE

def estimate_depth_map(image, device="cpu"):
    pipe = load_depth_pipeline(device)
    if pipe is None:
        return None
    try:
        result = pipe(image)
        depth_pil = result["depth"]
        depth_pil = depth_pil.resize(image.size, Image.Resampling.BILINEAR)
        depth_arr = np.asarray(depth_pil, dtype=np.float32) / 255.0
        return depth_arr
    except Exception as exc:
        print("Depth estimation failed:", type(exc).__name__, exc)
        return None

def expected_person_height_from_depth(foot_x, foot_y, depth_map=None, image_height=RESOLUTION, variant=None, jitter=0.0):
    if depth_map is None:
        return expected_person_height_from_ground_y(foot_y, image_height, variant, jitter)
    
    h, w = depth_map.shape
    fx = int(np.clip(foot_x, 0, w - 1))
    fy = int(np.clip(foot_y, 0, h - 1))
    
    pad = 2
    x_min = max(0, fx - pad)
    x_max = min(w, fx + pad + 1)
    y_min = max(0, fy - pad)
    y_max = min(h, fy + pad + 1)
    
    disparity = float(np.mean(depth_map[y_min:y_max, x_min:x_max]))
    
    base_ratio = 0.075 + disparity * (0.29 - 0.075)
    base_ratio = float(np.clip(base_ratio, 0.075, 0.29))
    
    variant_name = variant or "add_single_pedestrian"
    variant_multiplier = 1.0
    if "distant" in variant_name or "small_group" in variant_name:
        variant_multiplier = 0.82
    elif "near" in variant_name:
        variant_multiplier = 1.13
    elif "two" in variant_name:
        variant_multiplier = 0.96
    base_ratio *= max(variant_multiplier, PERSON_TARGET_SCALE_MULTIPLIER)
    if variant_name in {"add_two_pedestrians", "add_small_group"}:
        base_ratio *= 1.0 + float(jitter)
        
    return max(28.0, base_ratio * image_height)

def expected_person_height_from_ground_y(foot_y, image_height=RESOLUTION, variant=None, jitter=0.0):
    ground_y_norm = float(foot_y) / max(1.0, float(image_height))
    base_ratio = 0.065 + (ground_y_norm - 0.55) * 0.58
    base_ratio = float(np.clip(base_ratio, 0.075, 0.29))
    variant_name = variant or "add_single_pedestrian"
    variant_multiplier = 1.0
    if "distant" in variant_name or "small_group" in variant_name:
        variant_multiplier = 0.82
    elif "near" in variant_name:
        variant_multiplier = 1.13
    elif "two" in variant_name:
        variant_multiplier = 0.96
    base_ratio *= max(variant_multiplier, PERSON_TARGET_SCALE_MULTIPLIER)
    if variant_name in {"add_two_pedestrians", "add_small_group"}:
        base_ratio *= 1.0 + float(jitter)
    return max(28.0, base_ratio * image_height)


def default_scale_correction_metadata():
    return {
        "expected_person_height": "",
        "detected_person_height": "",
        "scale_ratio_before_correction": "",
        "expected_height": "",
        "detected_height": "",
        "scale_ratio_before": "",
        "scale_corrected": False,
        "resized_person_height": "",
        "resized_person_width": "",
        "scale_correction_status": "none",
        "seamless_clone_used": False,
        "fallback_alpha_paste": False,
        "fallback_alpha_used": False,
        "retry_attempts": 0,
        "last_reject_reason": "",
        "reject_reason": "",
    }


def scale_jitter_for_person(variant, index):
    if not GROUP_SCALE_JITTER_ENABLED:
        return 0.0
    if variant not in {"add_two_pedestrians", "add_small_group"}:
        return 0.0
    return (-0.04, 0.0, 0.04, -0.02, 0.02)[index % 5]


def enforce_monotonic_perspective_heights(items):
    if not PERSPECTIVE_MONOTONIC_SCALE_ENABLED or len(items) < 2:
        return items
    ordered = sorted(items, key=lambda item: item["foot_y"])
    prev_height = None
    for item in ordered:
        if prev_height is not None:
            item["expected_height"] = max(item["expected_height"], prev_height * PERSPECTIVE_MONOTONIC_MIN_RATIO)
        prev_height = item["expected_height"]
    return items


def scale_correction_policy(scale_ratio, person_conf=1.0, mask_area=1.0):
    if SCALE_CORRECTION_SOFT_MIN <= scale_ratio <= SCALE_CORRECTION_SOFT_MAX:
        return "recoverable"
    if scale_ratio < SCALE_CORRECTION_HARD_MIN or scale_ratio > SCALE_CORRECTION_HARD_MAX:
        return "unrecoverable"
    if (
        not SCALE_CORRECTION_BORDERLINE_RETRY
        or (person_conf >= SCALE_CORRECTION_BORDERLINE_MIN_CONF and mask_area >= SCALE_CORRECTION_BORDERLINE_MIN_MASK_AREA_RATIO)
    ):
        return "borderline_accept"
    return "borderline_retry"


def corrected_person_layers(generated_image, detections, variant, depth_map=None):
    width, height = generated_image.size
    margin = int(MIN_BORDER_MARGIN_RATIO * max(width, height))
    corrected_rgb = Image.new("RGB", generated_image.size, (0, 0, 0))
    combined_mask = Image.new("L", generated_image.size, 0)
    corrected_bboxes = []
    meta = default_scale_correction_metadata()
    per_person_meta = []
    keep_components = max(1, len(detections))
    planned_items = []
    for person_order, detection in enumerate(detections):
        det_bbox = detection["bbox"]
        dx1, dy1, dx2, dy2 = [int(round(v)) for v in det_bbox]
        dx1, dy1 = max(0, dx1), max(0, dy1)
        dx2, dy2 = min(width, dx2), min(height, dy2)
        foot_y = float(dy2)
        foot_x = (float(dx1) + float(dx2)) / 2.0
        planned_items.append({
            "detection": detection,
            "order": person_order,
            "foot_y": foot_y,
            "foot_x": foot_x,
            "expected_height": expected_person_height_from_depth(
                foot_x,
                foot_y,
                depth_map,
                image_height=height,
                variant=variant,
                jitter=scale_jitter_for_person(variant, person_order),
            ),
        })
    planned_items = enforce_monotonic_perspective_heights(planned_items)
    for item in planned_items:
        detection = item["detection"]
        person_order = item["order"]
        det_bbox = detection["bbox"]
        x1, y1, x2, y2 = [int(round(v)) for v in det_bbox]
        x1, y1 = max(0, x1), max(0, y1)
        x2, y2 = min(width, x2), min(height, y2)
        detected_height = max(1.0, float(y2 - y1))
        detected_width = max(1.0, float(x2 - x1))
        target_foot_y = float(y2)
        target_center_x = (float(x1) + float(x2)) / 2.0
        expected_height = item["expected_height"]
        scale_ratio = expected_height / max(1.0, detected_height)
        raw_mask = detection["mask"].resize(generated_image.size, Image.NEAREST)
        mask_bbox = raw_mask.getbbox()
        if mask_bbox is not None:
            pad = PERSON_CROP_MASK_PADDING
            mx1, my1, mx2, my2 = [int(round(v)) for v in mask_bbox]
            x1 = max(0, min(x1, mx1 - pad))
            y1 = max(0, min(y1, my1 - pad))
            x2 = min(width, max(x2, mx2 + pad))
            y2 = min(height, max(y2, my2 + pad))
        mask_area = mask_area_ratio(raw_mask)
        status = scale_correction_policy(scale_ratio, detection.get("conf", 1.0), mask_area)
        if status == "unrecoverable":
            meta.update({
                "expected_person_height": round(expected_height, 2),
                "detected_person_height": round(detected_height, 2),
                "scale_ratio_before_correction": round(scale_ratio, 4),
                "expected_height": round(expected_height, 2),
                "detected_height": round(detected_height, 2),
                "scale_ratio_before": round(scale_ratio, 4),
                "scale_correction_status": "unrecoverable",
                "last_reject_reason": "scale_unrecoverable",
                "reject_reason": "scale_unrecoverable",
            })
            return None, None, None, meta, "scale_unrecoverable"
        if status == "borderline_retry":
            meta.update({
                "expected_person_height": round(expected_height, 2),
                "detected_person_height": round(detected_height, 2),
                "scale_ratio_before_correction": round(scale_ratio, 4),
                "expected_height": round(expected_height, 2),
                "detected_height": round(detected_height, 2),
                "scale_ratio_before": round(scale_ratio, 4),
                "scale_correction_status": "borderline_retry",
                "last_reject_reason": "scale_unrecoverable",
                "reject_reason": "scale_unrecoverable",
            })
            return None, None, None, meta, "scale_unrecoverable"

        resized_height = max(8, int(round(detected_height * scale_ratio)))
        resized_width = max(4, int(round(detected_width * scale_ratio)))
        new_y2 = int(round(target_foot_y))
        new_y1 = new_y2 - resized_height
        new_x1 = int(round(target_center_x - resized_width / 2.0))
        new_x2 = new_x1 + resized_width
        if new_x1 < margin or new_x2 > width - margin or new_y1 < margin or new_y2 > height - margin:
            meta.update({
                "expected_person_height": round(expected_height, 2),
                "detected_person_height": round(detected_height, 2),
                "scale_ratio_before_correction": round(scale_ratio, 4),
                "expected_height": round(expected_height, 2),
                "detected_height": round(detected_height, 2),
                "scale_ratio_before": round(scale_ratio, 4),
                "resized_person_height": resized_height,
                "resized_person_width": resized_width,
                "scale_correction_status": "border_reject_after_resize",
                "last_reject_reason": "scale_unrecoverable",
                "reject_reason": "scale_unrecoverable",
            })
            return None, None, None, meta, "scale_unrecoverable"
        corrected_aspect = resized_height / max(1.0, resized_width)
        corrected_area_ratio = (resized_height * resized_width) / max(1.0, width * height)
        if (
            corrected_aspect < MIN_CORRECTED_ASPECT_RATIO
            or corrected_aspect > MAX_CORRECTED_ASPECT_RATIO
            or corrected_area_ratio < MIN_CORRECTED_PERSON_AREA_RATIO
        ):
            meta.update({
                "expected_person_height": round(expected_height, 2),
                "detected_person_height": round(detected_height, 2),
                "scale_ratio_before_correction": round(scale_ratio, 4),
                "expected_height": round(expected_height, 2),
                "detected_height": round(detected_height, 2),
                "scale_ratio_before": round(scale_ratio, 4),
                "resized_person_height": resized_height,
                "resized_person_width": resized_width,
                "scale_correction_status": "bad_scale_geometry_after_resize",
                "last_reject_reason": "scale_unrecoverable",
                "reject_reason": "scale_unrecoverable",
            })
            return None, None, None, meta, "scale_unrecoverable"

        crop_w = max(1, x2 - x1)
        crop_h = max(1, y2 - y1)
        scale_x = resized_width / max(1.0, detected_width)
        scale_y = resized_height / max(1.0, detected_height)
        resized_crop_w = max(4, int(round(crop_w * scale_x)))
        resized_crop_h = max(8, int(round(crop_h * scale_y)))
        crop_offset_x = int(round((x1 - int(round(det_bbox[0]))) * scale_x))
        crop_offset_y = int(round((y1 - int(round(det_bbox[1]))) * scale_y))
        paste_x = new_x1 + crop_offset_x
        paste_y = new_y1 + crop_offset_y
        person_crop = generated_image.crop((x1, y1, x2, y2)).resize((resized_crop_w, resized_crop_h), Image.LANCZOS)
        mask_crop = raw_mask.crop((x1, y1, x2, y2)).resize((resized_crop_w, resized_crop_h), Image.NEAREST)
        mask_crop = clean_binary_person_mask(mask_crop, keep_components=ACCESSORY_KEEP_COMPONENTS)
        corrected_rgb.paste(person_crop, (paste_x, paste_y), mask_crop)
        combined_mask.paste(mask_crop, (paste_x, paste_y), mask_crop)
        corrected_bboxes.append((new_x1, new_y1, new_x2, new_y2))
        per_person_meta.append({
            "expected_person_height": round(expected_height, 2),
            "detected_person_height": round(detected_height, 2),
            "scale_ratio_before_correction": round(scale_ratio, 4),
            "expected_height": round(expected_height, 2),
            "detected_height": round(detected_height, 2),
            "scale_ratio_before": round(scale_ratio, 4),
            "scale_corrected": abs(scale_ratio - 1.0) > 0.05,
            "resized_person_height": resized_height,
            "resized_person_width": resized_width,
            "scale_correction_status": "corrected" if abs(scale_ratio - 1.0) > 0.05 else "none",
        })

    if not corrected_bboxes:
        meta["last_reject_reason"] = "no_person_detected"
        return None, None, None, meta, "no_person_detected"
    any_corrected = any(item["scale_corrected"] for item in per_person_meta)
    meta.update({
        "expected_person_height": json.dumps([item["expected_person_height"] for item in per_person_meta]),
        "detected_person_height": json.dumps([item["detected_person_height"] for item in per_person_meta]),
        "scale_ratio_before_correction": json.dumps([item["scale_ratio_before_correction"] for item in per_person_meta]),
        "expected_height": json.dumps([item["expected_height"] for item in per_person_meta]),
        "detected_height": json.dumps([item["detected_height"] for item in per_person_meta]),
        "scale_ratio_before": json.dumps([item["scale_ratio_before"] for item in per_person_meta]),
        "scale_corrected": any_corrected,
        "resized_person_height": json.dumps([item["resized_person_height"] for item in per_person_meta]),
        "resized_person_width": json.dumps([item["resized_person_width"] for item in per_person_meta]),
        "scale_correction_status": "corrected" if any_corrected else "none",
    })
    combined_mask = clean_binary_person_mask(combined_mask, keep_components=max(keep_components, ACCESSORY_KEEP_COMPONENTS))
    return corrected_rgb, combined_mask, union_bboxes(corrected_bboxes), meta, "ok"


def min_accepted_person_height_ratio(variant, ground_y=None, resolution=RESOLUTION):
    if variant == "add_distant_pedestrian":
        return MIN_ACCEPTED_DISTANT_HEIGHT_RATIO
    if variant == "add_near_pedestrian":
        base = MIN_ACCEPTED_NEAR_HEIGHT_RATIO
    else:
        base = MIN_ACCEPTED_SINGLE_HEIGHT_RATIO
    if ground_y is not None:
        y_far = resolution * PATCH_ROAD_Y_RANGE[0]
        y_near = resolution * PATCH_ROAD_Y_RANGE[1]
        t = max(0.0, min(1.0, (ground_y - y_far) / max(1.0, y_near - y_far)))
        perspective_min = (
            MIN_ACCEPTED_PERSPECTIVE_HEIGHT_RATIO_FAR
            + t * (MIN_ACCEPTED_PERSPECTIVE_HEIGHT_RATIO_NEAR - MIN_ACCEPTED_PERSPECTIVE_HEIGHT_RATIO_FAR)
        )
        base = max(base, perspective_min)
        if variant != "add_distant_pedestrian" and ground_y / max(1, resolution) >= 0.72:
            base = max(base, MIN_ACCEPTED_FOREGROUND_HEIGHT_RATIO)
    return base


def max_accepted_person_height_ratio(variant, ground_y=None, resolution=RESOLUTION):
    if variant == "add_distant_pedestrian":
        return 0.27
    if ground_y is None:
        return 0.62 if variant == "add_near_pedestrian" else 0.52
    y_far = resolution * PATCH_ROAD_Y_RANGE[0]
    y_near = resolution * PATCH_ROAD_Y_RANGE[1]
    t = max(0.0, min(1.0, (ground_y - y_far) / max(1.0, y_near - y_far)))
    far_max = 0.36
    near_max = 0.62 if variant == "add_near_pedestrian" else 0.52
    return far_max + t * (near_max - far_max)


def normalize_expected_person_height(expected_person_height):
    if expected_person_height is None or expected_person_height == "":
        return None
    if isinstance(expected_person_height, (int, float)):
        return float(expected_person_height)
    if isinstance(expected_person_height, str):
        try:
            expected_person_height = json.loads(expected_person_height)
        except Exception:
            try:
                return float(expected_person_height)
            except Exception:
                return None
    if isinstance(expected_person_height, (list, tuple)):
        values = []
        for item in expected_person_height:
            try:
                values.append(float(item))
            except Exception:
                continue
        return max(values) if values else None
    return None


def validate_pasted_person_mask(pasted_mask, variant, insert_bbox, resolution=RESOLUTION, expected_person_height=None):
    bbox = pasted_mask.getbbox()
    if bbox is None:
        raise RuntimeError("Accepted mask is empty after paste.")
    x1, y1, x2, y2 = bbox
    if REJECT_IF_MASK_TOUCHES_BORDER and mask_bbox_touches_border(bbox, pasted_mask.size, margin=PERSON_BORDER_REJECT_PIXELS):
        print("Final person mask is close to image border; accepting with relaxed limb-preservation policy.")
    mask_h = y2 - y1
    mask_w = max(1, x2 - x1)
    final_aspect = mask_h / mask_w
    relaxed_max_aspect = max(MAX_PERSON_MASK_ASPECT_RATIO, 6.0)
    relaxed_min_aspect = min(MIN_PERSON_MASK_ASPECT_RATIO, 1.15)
    if final_aspect < relaxed_min_aspect or final_aspect > relaxed_max_aspect:
        raise RuntimeError(f"Accepted person final mask aspect invalid/slim (aspect={final_aspect:.2f}).")
    ground_y = insert_bbox[3] if insert_bbox is not None else y2
    expected_h = normalize_expected_person_height(expected_person_height)
    if FINAL_SCALE_VALIDATION_ENABLED:
        if expected_h is not None and expected_h > 1:
            scale_ratio = mask_h / max(1.0, expected_h)
            if scale_ratio < FINAL_MIN_SCALE_RATIO or scale_ratio > FINAL_MAX_SCALE_RATIO:
                raise RuntimeError(
                    f"Accepted person scale mismatch on full image (scale_ratio={scale_ratio:.2f}, "
                    f"mask_h={mask_h:.1f}, expected={expected_h:.1f})."
                )
        else:
            min_h = resolution * min_accepted_person_height_ratio(variant, ground_y=ground_y, resolution=resolution)
            if mask_h < min_h:
                raise RuntimeError(f"Accepted person is visually too small (mask_h={mask_h:.1f}, min_h={min_h:.1f}).")
    mask_arr = np.asarray(pasted_mask.convert("L"), dtype=np.float32) / 255.0
    active = mask_arr > 0.04
    if np.any(active):
        opaque_ratio = float(np.mean(mask_arr[active] > 0.72))
        if opaque_ratio < MIN_ACCEPTED_MASK_OPAQUE_RATIO:
            raise RuntimeError(f"Accepted person mask is too soft/transparent (opaque_ratio={opaque_ratio:.2f}).")




In [ ]:
%%writefile /kaggle/working/sd35_model.py
"""SD3.5 CityPersons augmentation: model and pipeline loading."""

import csv
import gc
import json
from datetime import datetime
import math
import numpy as np
import os
import random
import re
import statistics
import time
import warnings
from concurrent.futures import ThreadPoolExecutor, as_completed
from dataclasses import dataclass
from pathlib import Path
from threading import Lock
from typing import Optional

import matplotlib.pyplot as plt
import torch
from PIL import Image, ImageOps, ImageDraw, ImageFilter, ImageChops

try:
    import cv2
except ImportError:
    cv2 = None

from sd35_config import *
from sd35_utils import clear_cuda

os.environ["DIFFUSERS_VERBOSITY"] = "error"
warnings.filterwarnings("ignore", message="Flax classes are deprecated.*")
warnings.filterwarnings("ignore", category=FutureWarning, module="diffusers.*")

from diffusers import StableDiffusion3Img2ImgPipeline, StableDiffusion3Pipeline
from diffusers.utils import logging as diffusers_logging
diffusers_logging.set_verbosity_error()
try:
    from diffusers import StableDiffusion3InpaintPipeline
except ImportError:
    StableDiffusion3InpaintPipeline = None

PIPELINE_LOAD_LOCK = Lock()

def resolve_augmentation_devices():
    if AUGMENTATION_DEVICES:
        return AUGMENTATION_DEVICES
    if not torch.cuda.is_available():
        return ["cpu"]
    if USE_ALL_GPUS_FOR_AUGMENTATION:
        return [f"cuda:{index}" for index in range(torch.cuda.device_count())]
    return [TRAIN_DEVICE]


def build_img2img_pipeline(backend=MODEL_BACKEND, device=TRAIN_DEVICE):
    if backend == "sd35":
        pipeline_cls = StableDiffusion3Img2ImgPipeline
        model_id = SD35_MODEL_ID
        kwargs = {"torch_dtype": torch.float16, "use_safetensors": True, "low_cpu_mem_usage": True}
        if not USE_T5:
            kwargs.update({"text_encoder_3": None, "tokenizer_3": None})
    else:
        raise ValueError(f"Unsupported backend: {backend}")

    if str(device).startswith("cuda"):
        torch.cuda.set_device(torch.device(device).index or 0)

    with PIPELINE_LOAD_LOCK:
        try:
            pipe = pipeline_cls.from_pretrained(model_id, **kwargs)
        except TypeError:
            kwargs.pop("text_encoder_3", None)
            kwargs.pop("tokenizer_3", None)
            kwargs.pop("low_cpu_mem_usage", None)
            pipe = pipeline_cls.from_pretrained(model_id, **kwargs)

        if str(device).startswith("cuda") and USE_MODEL_CPU_OFFLOAD and hasattr(pipe, "enable_model_cpu_offload"):
            gpu_id = torch.device(device).index or 0
            pipe.enable_model_cpu_offload(gpu_id=gpu_id)
            print(f"Enabled model CPU offload for {device}")
        else:
            pipe.to(device)
        if hasattr(pipe, "enable_vae_slicing"):
            pipe.enable_vae_slicing()
        if hasattr(pipe, "enable_vae_tiling"):
            pipe.enable_vae_tiling()
        if hasattr(pipe, "enable_attention_slicing"):
            pipe.enable_attention_slicing()
    return pipe


def build_inpaint_pipeline(backend=MODEL_BACKEND, device=TRAIN_DEVICE):
    if backend == "sd35":
        if StableDiffusion3InpaintPipeline is None:
            raise ImportError("StableDiffusion3InpaintPipeline is not available in this diffusers version.")
        pipeline_cls = StableDiffusion3InpaintPipeline
        model_id = SD35_MODEL_ID
        kwargs = {"torch_dtype": torch.float16, "use_safetensors": True, "low_cpu_mem_usage": True}
        if not USE_T5:
            kwargs.update({"text_encoder_3": None, "tokenizer_3": None})
    else:
        raise ValueError(f"Unsupported backend: {backend}")

    if str(device).startswith("cuda"):
        torch.cuda.set_device(torch.device(device).index or 0)

    with PIPELINE_LOAD_LOCK:
        try:
            pipe = pipeline_cls.from_pretrained(model_id, **kwargs)
        except TypeError:
            kwargs.pop("text_encoder_3", None)
            kwargs.pop("tokenizer_3", None)
            kwargs.pop("low_cpu_mem_usage", None)
            pipe = pipeline_cls.from_pretrained(model_id, **kwargs)

        if str(device).startswith("cuda") and USE_MODEL_CPU_OFFLOAD and hasattr(pipe, "enable_model_cpu_offload"):
            gpu_id = torch.device(device).index or 0
            pipe.enable_model_cpu_offload(gpu_id=gpu_id)
            print(f"Enabled model CPU offload for {device}")
        else:
            pipe.to(device)
        if hasattr(pipe, "enable_vae_slicing"):
            pipe.enable_vae_slicing()
        if hasattr(pipe, "enable_vae_tiling"):
            pipe.enable_vae_tiling()
        if hasattr(pipe, "enable_attention_slicing"):
            pipe.enable_attention_slicing()
    return pipe


In [ ]:
%%writefile /kaggle/working/sd35_evaluation.py
"""SD3.5 CityPersons augmentation: segmentation, validation, and retry policy."""

import csv
import gc
import json
from datetime import datetime
import math
import numpy as np
import os
import random
import re
import statistics
import time
import warnings
from concurrent.futures import ThreadPoolExecutor, as_completed
from dataclasses import dataclass
from pathlib import Path
from threading import Lock
from typing import Optional

import matplotlib.pyplot as plt
import torch
from PIL import Image, ImageOps, ImageDraw, ImageFilter, ImageChops

try:
    import cv2
except ImportError:
    cv2 = None

from sd35_config import *
from sd35_utils import *

PERSON_SEGMENTER = None

def load_person_segmenter():
    global PERSON_SEGMENTER
    if PERSON_SEGMENTER is False:
        return None
    if PERSON_SEGMENTER is not None:
        return PERSON_SEGMENTER
    try:
        from ultralytics import YOLO
        PERSON_SEGMENTER = YOLO(CONTEXT_PERSON_SEGMENTATION_MODEL)
        print(f"Loaded person segmentation model: {CONTEXT_PERSON_SEGMENTATION_MODEL}")
        return PERSON_SEGMENTER
    except Exception as exc:
        PERSON_SEGMENTER = False
        print("Person segmentation unavailable; context_person_composite will use fallback if enabled.")
        print(type(exc).__name__, exc)
        return None


def bbox_iou(a, b):
    ax1, ay1, ax2, ay2 = a
    bx1, by1, bx2, by2 = b
    ix1 = max(ax1, bx1)
    iy1 = max(ay1, by1)
    ix2 = min(ax2, bx2)
    iy2 = min(ay2, by2)
    iw = max(0, ix2 - ix1)
    ih = max(0, iy2 - iy1)
    inter = iw * ih
    union = max(1, bbox_area(a) + bbox_area(b) - inter)
    return inter / union


def mask_bbox_from_array(mask_array, threshold=0.5):
    ys, xs = np.where(mask_array > threshold)
    if len(xs) == 0 or len(ys) == 0:
        return None
    return (float(xs.min()), float(ys.min()), float(xs.max() + 1), float(ys.max() + 1))


def mask_bbox_touches_border(mask_bbox, size, margin=PERSON_BORDER_REJECT_PIXELS):
    if mask_bbox is None:
        return True
    width, height = size
    x1, y1, x2, y2 = mask_bbox
    return x1 <= margin or y1 <= margin or x2 >= width - margin or y2 >= height - margin


def expected_new_person_count(variant):
    return {
        "add_two_pedestrians": 2,
        "add_small_group": 3,
    }.get(variant, 1)


def union_bboxes(bboxes):
    if not bboxes:
        return None
    return (
        min(float(bbox[0]) for bbox in bboxes),
        min(float(bbox[1]) for bbox in bboxes),
        max(float(bbox[2]) for bbox in bboxes),
        max(float(bbox[3]) for bbox in bboxes),
    )


def person_mask_completeness_ok(raw_mask, raw_mask_bbox, det_bbox, target_bbox):
    if raw_mask_bbox is None:
        return False, "empty mask bbox"
    mask_w = max(1.0, raw_mask_bbox[2] - raw_mask_bbox[0])
    mask_h = max(1.0, raw_mask_bbox[3] - raw_mask_bbox[1])
    det_h = max(1.0, det_bbox[3] - det_bbox[1])
    target_h = max(1.0, target_bbox[3] - target_bbox[1])
    if mask_h / det_h < MIN_PERSON_MASK_DET_HEIGHT_RATIO:
        return False, f"mask covers too little detection height ({mask_h / det_h:.2f})"
    if mask_h / target_h < MIN_PERSON_MASK_TARGET_HEIGHT_RATIO:
        return False, f"mask covers too little target height ({mask_h / target_h:.2f})"
    aspect = mask_h / mask_w
    if aspect < MIN_PERSON_MASK_ASPECT_RATIO or aspect > MAX_PERSON_MASK_ASPECT_RATIO:
        return False, f"partial body mask aspect ({aspect:.2f})"
    x1, y1, x2, y2 = [int(round(v)) for v in det_bbox]
    height, width = raw_mask.shape[:2]
    x1, y1 = max(0, x1), max(0, y1)
    x2, y2 = min(width, x2), min(height, y2)
    crop = raw_mask[y1:y2, x1:x2] > CONTEXT_PERSON_MASK_THRESHOLD
    if crop.size == 0:
        return False, "empty detection crop"
    bands = np.array_split(crop, 3, axis=0)
    row_coverages = [float(np.mean(np.any(band, axis=1))) if band.size else 0.0 for band in bands]
    top_mid_ok = min(row_coverages[:2]) >= MIN_PERSON_MASK_VERTICAL_BAND_COVERAGE
    lower_leg_floor = max(0.06, MIN_PERSON_MASK_VERTICAL_BAND_COVERAGE * 0.45)
    lower_ok = row_coverages[2] >= lower_leg_floor
    if not (top_mid_ok and lower_ok):
        return False, f"partial body vertical coverage {row_coverages}"
    return True, "ok"


def select_generated_person_mask(generated_crop, target_bbox):
    segmenter = load_person_segmenter()
    if segmenter is None:
        return None, None
    try:
        results = segmenter.predict(generated_crop, imgsz=RESOLUTION, conf=CONTEXT_PERSON_MIN_CONFIDENCE, verbose=False)
    except Exception as exc:
        print("Person segmentation failed; using fallback if enabled.")
        print(type(exc).__name__, exc)
        return None, None
    if not results:
        return None, None
    result = results[0]
    boxes = getattr(result, "boxes", None)
    masks = getattr(result, "masks", None)
    if boxes is None or masks is None or boxes.xyxy is None or masks.data is None:
        return None, None
    detection_target_bbox = expand_bbox_for_detection(target_bbox, generated_crop.size)
    target_cx = (target_bbox[0] + target_bbox[2]) / 2
    target_cy = (target_bbox[1] + target_bbox[3]) / 2
    target_h = max(1.0, target_bbox[3] - target_bbox[1])
    best_index = None
    best_score = -1e9
    xyxy = boxes.xyxy.detach().cpu().numpy()
    cls = boxes.cls.detach().cpu().numpy() if boxes.cls is not None else np.zeros(len(xyxy))
    conf = boxes.conf.detach().cpu().numpy() if boxes.conf is not None else np.ones(len(xyxy))
    for index, box in enumerate(xyxy):
        if int(cls[index]) != 0:
            continue
        raw_mask = masks.data[index].detach().cpu().numpy()
        raw_mask_bbox = mask_bbox_from_array(raw_mask, threshold=CONTEXT_PERSON_MASK_THRESHOLD)
        if REJECT_IF_MASK_TOUCHES_BORDER and mask_bbox_touches_border(raw_mask_bbox, generated_crop.size, margin=max(1, PERSON_BORDER_REJECT_PIXELS // 2)):
            print("Detected person mask is near crop border; keeping candidate for relaxed full-body validation.")
        det_bbox = tuple(float(v) for v in box)
        complete, reason = person_mask_completeness_ok(raw_mask, raw_mask_bbox, det_bbox, target_bbox)
        if not complete:
            print(f"Detected person mask looks partial ({reason}); retrying.")
            continue
        det_h = max(1.0, det_bbox[3] - det_bbox[1])
        height_ratio = det_h / target_h
        if STRICT_EARLY_PERSON_SCALE_FILTER and (height_ratio < MIN_GENERATED_HEIGHT_RATIO or height_ratio > MAX_GENERATED_HEIGHT_RATIO):
            print(f"Detected person scale mismatch (height_ratio={height_ratio:.2f}); retrying.")
            continue
        mask_h = 0.0 if raw_mask_bbox is None else raw_mask_bbox[3] - raw_mask_bbox[1]
        mask_height_ratio = mask_h / target_h
        if STRICT_EARLY_PERSON_SCALE_FILTER and mask_height_ratio < MIN_MASK_BBOX_HEIGHT_RATIO:
            print(f"Detected person mask height too small (ratio={mask_height_ratio:.2f}); retrying.")
            continue
        det_cx = (det_bbox[0] + det_bbox[2]) / 2
        det_cy = (det_bbox[1] + det_bbox[3]) / 2
        overlap = bbox_iou(det_bbox, detection_target_bbox)
        target_overlap = bbox_intersection_area(det_bbox, detection_target_bbox) / max(1, bbox_area(det_bbox))
        if overlap <= 0 and target_overlap < CONTEXT_MIN_PERSON_TARGET_OVERLAP:
            continue
        distance = math.hypot(det_cx - target_cx, det_cy - target_cy) / RESOLUTION
        score = 3.0 * overlap + 1.5 * target_overlap + float(conf[index]) - distance
        if score > best_score:
            best_index = index
            best_score = score
    if best_index is None:
        return None, None
    mask_array = masks.data[best_index].detach().cpu().numpy()
    mask = Image.fromarray((mask_array > CONTEXT_PERSON_MASK_THRESHOLD).astype(np.uint8) * 255, mode="L")
    mask = mask.resize(generated_crop.size, Image.NEAREST)
    area_ratio = mask_area_ratio(mask)
    if area_ratio < CONTEXT_MIN_PERSON_MASK_AREA_RATIO:
        print(f"Detected person mask is too small (area_ratio={area_ratio:.5f}); rejecting as ghost/unchanged.")
        return None, None
    return mask, tuple(float(v) for v in xyxy[best_index])


def select_new_generated_person_mask(generated_image, existing_person_bboxes=None, semantic_masks=None, variant=None, background_image=None, depth_map=None):
    segmenter = load_person_segmenter()
    if segmenter is None:
        return None, None, "segmenter_unavailable", default_scale_correction_metadata(), None
    try:
        results = segmenter.predict(generated_image, imgsz=RESOLUTION, conf=CONTEXT_PERSON_MIN_CONFIDENCE, verbose=False)
    except Exception as exc:
        print("Person segmentation failed; retrying if possible.")
        print(type(exc).__name__, exc)
        meta = default_scale_correction_metadata()
        meta["last_reject_reason"] = "segmenter_failed"
        return None, None, "segmenter_failed", meta, None
    if not results:
        meta = default_scale_correction_metadata()
        meta["last_reject_reason"] = "no_person_detected"
        return None, None, "no_person_detected", meta, None
    result = results[0]
    boxes = getattr(result, "boxes", None)
    masks = getattr(result, "masks", None)
    if boxes is None or masks is None or boxes.xyxy is None or masks.data is None:
        meta = default_scale_correction_metadata()
        meta["last_reject_reason"] = "no_person_detected"
        return None, None, "no_person_detected", meta, None

    existing_person_bboxes = existing_person_bboxes or []
    xyxy = boxes.xyxy.detach().cpu().numpy()
    cls = boxes.cls.detach().cpu().numpy() if boxes.cls is not None else np.zeros(len(xyxy))
    conf = boxes.conf.detach().cpu().numpy() if boxes.conf is not None else np.ones(len(xyxy))
    min_person_conf = MIN_PERSON_CONF_BY_VARIANT.get(variant, MIN_RETRY_PERSON_CONFIDENCE)
    expected_count = expected_new_person_count(variant or "add_single_pedestrian")
    candidates = []
    best_reject_reason = "no_person_detected"
    for index, box in enumerate(xyxy):
        if int(cls[index]) != 0:
            continue
        person_conf = float(conf[index])
        if person_conf < min_person_conf:
            best_reject_reason = "low_person_conf"
            print(f"Detected new person confidence too low (conf={person_conf:.2f}, min={min_person_conf:.2f}); retrying.")
            continue
        det_bbox = tuple(float(v) for v in box)
        det_area = max(1, bbox_area(det_bbox))
        old_overlap = 0.0
        old_iou = 0.0
        bad_person_depth_overlap = False
        for old_bbox in existing_person_bboxes:
            old_overlap = max(old_overlap, bbox_intersection_area(det_bbox, old_bbox) / det_area)
            old_iou = max(old_iou, bbox_iou(det_bbox, old_bbox))
            depth_ok, _overlap = person_overlap_depth_ok(det_bbox, old_bbox)
            if not depth_ok:
                bad_person_depth_overlap = True
        if bad_person_depth_overlap:
            best_reject_reason = "bad_person_depth_overlap"
            continue
        if old_overlap > MAX_PERSON_PERSON_OVERLAP_RATIO * 1.25 or old_iou > 0.06:
            best_reject_reason = "bad_person_depth_overlap"
            continue
        if (not ALLOW_PERSON_PERSON_OVERLAP) and (old_overlap > 0.18 or old_iou > 0.08):
            continue

        det_h = max(1.0, det_bbox[3] - det_bbox[1])
        det_w = max(1.0, det_bbox[2] - det_bbox[0])
        if det_h < 18 or det_w < 5:
            best_reject_reason = "too_small_or_ghost_person"
            continue

        foot_score = 0.0
        body_valid_score = 0.0
        avoid_score = 0.0
        if semantic_masks:
            valid_mask = semantic_masks.get("valid")
            avoid_mask = semantic_masks.get("avoid")
            foot_bbox = foot_support_bbox(det_bbox)
            foot_score = mask_coverage(valid_mask, foot_bbox)
            foot_avoid_score = mask_coverage(avoid_mask, foot_bbox)
            body_valid_score = mask_coverage(valid_mask, det_bbox)
            avoid_score = mask_coverage(avoid_mask, det_bbox)
            if foot_score < MIN_FOOT_SUPPORT:
                best_reject_reason = "floating_or_bad_ground"
                continue
            if foot_avoid_score > MAX_FOOT_AVOID_SUPPORT:
                best_reject_reason = "floating_or_bad_ground"
                continue
        else:
            ground_y_ratio = det_bbox[3] / max(1, generated_image.size[1])
            if ground_y_ratio < PATCH_ROAD_Y_RANGE[0] - 0.08 or ground_y_ratio > PATCH_ROAD_Y_RANGE[1] + 0.06:
                best_reject_reason = "floating_or_bad_ground"
                continue

        raw_mask = masks.data[index].detach().cpu().numpy()
        raw_mask_bbox = mask_bbox_from_array(raw_mask, threshold=CONTEXT_PERSON_MASK_THRESHOLD)
        if raw_mask_bbox is None:
            best_reject_reason = "too_small_or_ghost_person"
            continue
        if REJECT_IF_MASK_TOUCHES_BORDER and mask_bbox_touches_border(raw_mask_bbox, generated_image.size):
            best_reject_reason = "partial_or_cropped_body"
            print("Detected new person mask touches image border; rejecting likely cropped/oversized body.")
            continue
        mask_h = raw_mask_bbox[3] - raw_mask_bbox[1]
        if mask_h < 0.45 * det_h:
            best_reject_reason = "partial_or_cropped_body"
            continue

        mask_array = raw_mask > CONTEXT_PERSON_MASK_THRESHOLD
        mask = Image.fromarray(mask_array.astype(np.uint8) * 255, mode="L").resize(generated_image.size, Image.NEAREST)
        det_cx = (det_bbox[0] + det_bbox[2]) / 2.0
        expected_h = expected_person_height_from_depth(det_cx, det_bbox[3], depth_map, generated_image.size[1], variant=variant)
        scale_ratio = expected_h / max(1.0, det_h)
        policy = scale_correction_policy(scale_ratio, person_conf, mask_area_ratio(mask))
        if policy == "unrecoverable":
            best_reject_reason = "scale_unrecoverable"
            continue
        if policy == "borderline_retry":
            best_reject_reason = "scale_unrecoverable"
            continue

        distance_center = abs(((det_bbox[0] + det_bbox[2]) / 2) / max(1, generated_image.size[0]) - 0.5)
        scale_score = 1.0 - min(1.0, abs(math.log(max(scale_ratio, 1e-6))))
        score = person_conf + 2.2 * foot_score + 0.8 * body_valid_score + 0.45 * scale_score - 2.8 * avoid_score - 0.2 * distance_center
        candidates.append((score, index, det_bbox, mask, person_conf))
    if not candidates:
        meta = default_scale_correction_metadata()
        meta["last_reject_reason"] = best_reject_reason
        return None, None, best_reject_reason, meta, None
    sorted_candidates = sorted(candidates, key=lambda item: item[0], reverse=True)
    selected = sorted_candidates[:expected_count]
    if KEEP_EXTRA_GENERATED_PEOPLE and len(sorted_candidates) > expected_count:
        best_score = sorted_candidates[0][0]
        extra_limit = expected_count + MAX_EXTRA_GENERATED_PEOPLE
        for candidate in sorted_candidates[expected_count:extra_limit]:
            if best_score - candidate[0] <= EXTRA_PERSON_MIN_SCORE_DELTA:
                selected.append(candidate)
    if len(selected) < expected_count:
        if variant == "add_small_group" and ALLOW_PARTIAL_SMALL_GROUP and len(selected) >= 2:
            print(f"Accepting partial small group with {len(selected)}/{expected_count} pedestrians.")
        else:
            print(f"Only detected {len(selected)}/{expected_count} new pedestrians for {variant}; retrying.")
            meta = default_scale_correction_metadata()
            meta["last_reject_reason"] = "not_enough_new_people"
            return None, None, "not_enough_new_people", meta, None
    detections = [
        {"bbox": bbox, "mask": mask, "conf": person_conf, "index": index}
        for _score, index, bbox, mask, person_conf in selected
    ]
    scale_meta = default_scale_correction_metadata()
    if detections:
        scale_meta["person_confidence"] = round(
            sum(float(detection.get("conf", 0.0)) for detection in detections) / max(1, len(detections)),
            4,
        )
    corrected_image, corrected_mask, corrected_bbox, scale_meta_update, correction_reason = corrected_person_layers(
        generated_image,
        detections,
        variant or "add_single_pedestrian",
        depth_map=depth_map,
    )
    scale_meta.update(scale_meta_update or {})
    if corrected_mask is None:
        return None, None, correction_reason, scale_meta, None
    area_ratio = mask_area_ratio(corrected_mask)
    if area_ratio < max(CONTEXT_MIN_PERSON_MASK_AREA_RATIO, MIN_GHOST_PERSON_MASK_AREA_RATIO):
        print(f"Detected new person mask is too small (area_ratio={area_ratio:.5f}); rejecting as ghost/unchanged.")
        scale_meta["last_reject_reason"] = "too_small_or_ghost_person"
        return None, None, "too_small_or_ghost_person", scale_meta, None
    if background_image is not None:
        person_diff = masked_rgb_mae_255(background_image, corrected_image, corrected_mask)
        if person_diff < MIN_GHOST_PERSON_CONTRAST_255:
            print(f"Detected new person is ghost-like / low contrast (person_diff={person_diff:.2f}); retrying.")
            scale_meta["last_reject_reason"] = "ghost_person_low_contrast"
            return None, None, "ghost_person_low_contrast", scale_meta, None
    return corrected_mask, corrected_bbox, "ok", scale_meta, corrected_image

def adaptive_retry_params(base_strength, base_guidance, reject_reason, attempt):
    if attempt <= 0:
        return base_strength, base_guidance
    reason = reject_reason or "no_person_mask"
    if reason in {"ghost_person_low_contrast", "too_small_or_ghost_person", "low_person_conf", "no_person_mask"}:
        return min(0.84, base_strength + 0.04 * attempt), min(8.2, base_guidance + 0.60 * attempt)
    if reason == "not_enough_new_people":
        return min(0.86, base_strength + 0.03 * attempt), min(8.6, base_guidance + 0.55 * attempt)
    if reason == "too_large_for_perspective":
        return max(0.60, base_strength - 0.04 * attempt), max(6.0, base_guidance - 0.60 * attempt)
    if reason == "partial_or_cropped":
        return max(0.62, base_strength - 0.03 * attempt), max(6.0, base_guidance - 0.20 * attempt)
    return min(0.80, base_strength + 0.02 * attempt), min(7.8, base_guidance + 0.25 * attempt)


def adaptive_context_expand(base_expand, reject_reason, attempt):
    if attempt <= 0 or reject_reason != "partial_or_cropped":
        return base_expand
    return base_expand * (1.0 + 0.14 * attempt)


def build_retry_config(base_prompt, base_negative, reject_reason, strength, guidance, margin, attempt):
    attempt_prompt = base_prompt
    attempt_negative = base_negative
    attempt_strength, attempt_guidance = adaptive_retry_params(strength, guidance, reject_reason, attempt)
    attempt_margin = adaptive_context_expand(margin, reject_reason, attempt)
    if attempt <= 0 or not reject_reason:
        return attempt_prompt, attempt_negative, attempt_strength, attempt_guidance, attempt_margin
    if reject_reason in {"ghost_person_low_contrast", "too_small_or_ghost_person", "low_person_conf", "no_person_mask", "no_person_detected"}:
        attempt_strength = min(0.86, attempt_strength + 0.04)
        attempt_guidance = min(8.6, attempt_guidance + 0.35)
        attempt_prompt += ", clear solid person"
        attempt_negative += ", transparent, faded"
    elif reject_reason == "too_large_for_perspective":
        attempt_prompt += ", realistic scale"
        attempt_negative += ", wrong scale"
    elif reject_reason in {"scale_unrecoverable", "final_scale_mismatch"}:
        attempt_strength = min(0.86, attempt_strength + 0.03)
        attempt_guidance = min(8.4, attempt_guidance + 0.25)
        attempt_prompt += ", visible grounded person"
        attempt_negative += ", tiny, barely visible"
    elif reject_reason in {"partial_or_cropped", "partial_or_cropped_body", "accepted_mask_empty", "mask_too_soft"}:
        attempt_prompt += ", complete body visible"
        attempt_negative += ", cropped head, cropped feet, half body"
    elif reject_reason == "not_enough_new_people":
        attempt_strength = min(0.86, attempt_strength + 0.03)
        attempt_guidance = min(8.4, attempt_guidance + 0.30)
        attempt_prompt += ", separate people"
        attempt_negative += ", missing person, merged bodies"
    elif reject_reason == "bad_person_depth_overlap":
        attempt_strength = max(0.62, attempt_strength - 0.02)
        attempt_prompt += ", separated depth"
        attempt_negative += ", overlap, merged people"
    elif reject_reason == "floating_or_bad_ground":
        attempt_prompt += ", feet on road"
        attempt_negative += ", floating, on vehicle"
    elif reject_reason == "bad_composite_quality":
        attempt_prompt += ", clean edges"
        attempt_negative += ", sticker, halo, blurry"
    return attempt_prompt, attempt_negative, attempt_strength, attempt_guidance, attempt_margin


def variant_retry_budget(variant, base_retries=CONTEXT_GENERATION_RETRIES):
    if variant == "add_distant_pedestrian":
        return base_retries + DISTANT_EXTRA_RETRIES
    if variant == "add_small_group":
        return base_retries + SMALL_GROUP_EXTRA_RETRIES
    if variant == "add_near_pedestrian":
        return min(base_retries, NEAR_MAX_RETRIES)
    return base_retries


def should_retry(reason, attempt, max_retries, metadata=None):
    metadata = metadata or {}
    reason = normalize_reject_reason(reason or metadata.get("reject_reason") or "unknown")
    if attempt >= max_retries:
        return False
    if metadata.get("scale_unrecoverable_streak", 0) >= EARLY_STOP_SCALE_UNRECOVERABLE_STREAK:
        return False

    early_reasons = {
        "no_person_detected",
        "no_person_mask",
        "segmenter_failed",
        "segmenter_unavailable",
        "low_person_conf",
        "partial_or_cropped_body",
        "partial_or_cropped",
        "not_enough_new_people",
        "bad_person_depth_overlap",
        "scale_unrecoverable",
        "floating_or_bad_ground",
        "too_small_or_ghost_person",
    }
    if reason in early_reasons:
        if "scale" in reason and metadata.get("scale_ratio_before_correction"):
            return attempt < 1
        return True

    post_paste_reasons = {
        "ghost_person_low_contrast",
        "bad_composite_quality",
        "final_scale_mismatch",
        "accepted_mask_empty",
        "mask_too_soft",
    }
    if reason in post_paste_reasons:
        mask_area = float(metadata.get("mask_area_ratio") or 0.0)
        if mask_area >= POST_PASTE_RETRY_MIN_MASK_AREA_RATIO:
            return True
        if metadata.get("seamless_clone_used", False) and attempt == 0:
            return True
        if not metadata.get("seamless_clone_used", False):
            return True
    return False


def clamp01(value):
    try:
        return max(0.0, min(1.0, float(value)))
    except Exception:
        return 0.0


def _first_numeric(value, default=None):
    if value in ("", None):
        return default
    if isinstance(value, (int, float)):
        return float(value)
    if isinstance(value, str):
        try:
            parsed = json.loads(value)
            if isinstance(parsed, list) and parsed:
                values = [float(item) for item in parsed if item not in ("", None)]
                return sum(values) / len(values) if values else default
            return float(parsed)
        except Exception:
            try:
                return float(value)
            except Exception:
                return default
    return default


def compute_quality_scores(meta):
    person_conf = _first_numeric(meta.get("person_confidence"), default=None)
    if person_conf is None:
        person_conf = _first_numeric(meta.get("person_score"), default=1.0)
    person_score = clamp01(person_conf)

    scale_ratio = _first_numeric(
        meta.get("final_scale_ratio"),
        default=_first_numeric(meta.get("scale_ratio_before"), default=_first_numeric(meta.get("scale_ratio_before_correction"), default=1.0)),
    )
    scale_score = clamp01(1.0 - min(1.0, abs(math.log(max(scale_ratio or 1.0, 1e-6)))))

    final_diff = _first_numeric(meta.get("final_person_diff_mae255"), default=0.0)
    diff_threshold = _first_numeric(meta.get("final_person_diff_threshold"), default=FINAL_COMPOSITE_MIN_MAE_255)
    background_score = clamp01(final_diff / max(1e-6, diff_threshold * 4.0))

    edge_removed = _first_numeric(meta.get("foreground_occlusion_removed_ratio"), default=0.0)
    seamless_bonus = 0.12 if meta.get("seamless_clone_used", False) else 0.0
    fallback_penalty = 0.08 if meta.get("fallback_alpha_used", False) else 0.0
    edge_score = clamp01(1.0 - edge_removed + seamless_bonus - fallback_penalty)

    quality_score = (
        0.45 * person_score
        + 0.25 * scale_score
        + 0.20 * background_score
        + 0.10 * edge_score
    )
    return {
        "person_score": round(person_score, 4),
        "scale_score": round(scale_score, 4),
        "background_score": round(background_score, 4),
        "edge_score": round(edge_score, 4),
        "quality_score": round(clamp01(quality_score), 4),
    }


def validate_composite_result(source, result, pasted_mask, variant, insert_bbox, insert_meta):
    meta = dict(insert_meta or {})
    meta["mask_area_ratio"] = mask_area_ratio(pasted_mask)
    try:
        validate_pasted_person_mask(
            pasted_mask,
            variant,
            insert_bbox,
            resolution=source.size[0],
            expected_person_height=meta.get("expected_person_height"),
        )
    except RuntimeError as exc:
        reason = normalize_reject_reason(exc)
        if "scale mismatch" in str(exc):
            reason = "final_scale_mismatch"
        elif "empty" in str(exc):
            reason = "accepted_mask_empty"
        elif "soft" in str(exc) or "transparent" in str(exc):
            reason = "mask_too_soft"
        meta["reject_reason"] = reason
        meta["last_reject_reason"] = reason
        return False, reason, meta

    debug_mask = Image.new("L", source.size, 0)
    debug_mask.paste(pasted_mask, (0, 0))
    final_person_diff = masked_rgb_mae_255(source, result, debug_mask)
    meta["final_person_diff_mae255"] = round(final_person_diff, 4)
    final_diff_threshold = FINAL_COMPOSITE_MIN_MAE_255_SEAMLESS if meta.get("seamless_clone_used") else FINAL_COMPOSITE_MIN_MAE_255
    meta["final_person_diff_threshold"] = final_diff_threshold
    bbox = pasted_mask.getbbox()
    if bbox is not None:
        mask_h = bbox[3] - bbox[1]
        expected_h = normalize_expected_person_height(meta.get("expected_person_height"))
        if expected_h:
            meta["final_scale_ratio"] = round(mask_h / max(1.0, expected_h), 4)
    meta.update(compute_quality_scores(meta))
    if final_person_diff < final_diff_threshold:
        meta["reject_reason"] = "ghost_person_low_contrast"
        meta["last_reject_reason"] = "ghost_person_low_contrast"
        return False, "ghost_person_low_contrast", meta
    meta["reject_reason"] = ""
    return True, "ok", meta


def normalize_reject_reason(reason_text):
    text = str(reason_text)
    patterns = [
        r"last_reason=([A-Za-z0-9_]+)",
        r"Composite rejected as ([A-Za-z0-9_]+)",
        r"rejected as ([A-Za-z0-9_]+)",
        r"\\(([A-Za-z0-9_]+)\\)",
    ]
    for pattern in patterns:
        match = re.search(pattern, text)
        if match:
            return match.group(1).strip(")., ")
    known_reasons = [
        "scale_unrecoverable",
        "ghost_person_low_contrast",
        "floating_or_bad_ground",
        "not_enough_new_people",
        "bad_person_depth_overlap",
        "low_person_conf",
        "partial_or_cropped_body",
        "partial_or_cropped",
        "bad_mask_quality",
        "no_person_detected",
    ]
    for reason in known_reasons:
        if reason in text:
            return reason
    if "expected scalar type Half" in text or "mixed dtype" in text:
        return "pipeline_dtype_mismatch"
    return text.split()[0].strip(").,") if text.split() else "unknown"


In [ ]:
%%writefile /kaggle/working/sd35_pipeline.py
"""SD3.5 CityPersons augmentation: generation and compositing pipeline."""

import csv
import gc
import json
from datetime import datetime
import math
import numpy as np
import os
import random
import re
import statistics
import time
import warnings
from concurrent.futures import ThreadPoolExecutor, as_completed
from dataclasses import dataclass
from pathlib import Path
from threading import Lock
from typing import Optional

import matplotlib.pyplot as plt
import torch
from PIL import Image, ImageOps, ImageDraw, ImageFilter, ImageChops

try:
    import cv2
except ImportError:
    cv2 = None

from sd35_config import *
from sd35_data import build_generation_prompt, build_variant_negative_prompt
from sd35_utils import *
from sd35_evaluation import *

def add_contact_shadow(image, insert_bbox, variant):
    if not CONTACT_SHADOW_ENABLED:
        return image
    width, height = image.size
    overlay = Image.new("RGBA", image.size, (0, 0, 0, 0))
    draw = ImageDraw.Draw(overlay)
    for bbox in guide_bboxes_for_variant(insert_bbox, variant):
        x1, y1, x2, y2 = bbox
        bw = max(4, x2 - x1)
        bh = max(8, y2 - y1)
        ground_y = y2
        scale = perspective_scale_for_ground_y(ground_y, resolution=height)
        opacity = int(CONTACT_SHADOW_OPACITY_FAR + scale * (CONTACT_SHADOW_OPACITY_NEAR - CONTACT_SHADOW_OPACITY_FAR))
        shadow_w = max(8, int(bw * (0.70 + 0.30 * scale)))
        shadow_h = max(3, int(bh * 0.055))
        blur = max(2, int(bh * 0.025))
        cx = int((x1 + x2) / 2 + bw * 0.06)
        cy = int(ground_y - shadow_h * 0.35)
        shadow = Image.new("RGBA", image.size, (0, 0, 0, 0))
        shadow_draw = ImageDraw.Draw(shadow)
        shadow_draw.ellipse((cx - shadow_w // 2, cy - shadow_h // 2, cx + shadow_w // 2, cy + shadow_h // 2), fill=(0, 0, 0, opacity))
        shadow = shadow.filter(ImageFilter.GaussianBlur(radius=blur))
        overlay = Image.alpha_composite(overlay, shadow)
    return Image.alpha_composite(image.convert("RGBA"), overlay).convert("RGB")


def save_inpaint_debug_strip(record, variant, seed, source, mask_image, guided_source, generated, final, insert_bbox, debug_index=None):
    if not SAVE_PATCH_DEBUG:
        return ""
    if debug_index is not None and debug_index >= PATCH_DEBUG_MAX_ITEMS:
        return ""
    debug_dir = PATCH_DEBUG_DIR / record.split / record.bucket
    debug_dir.mkdir(parents=True, exist_ok=True)
    crop_bbox = expand_bbox_with_context(insert_bbox, resolution=source.width)
    panels = [
        ("source", source.crop(crop_bbox).convert("RGB")),
        ("mask", mask_image.crop(crop_bbox).convert("RGB")),
        ("guided", guided_source.crop(crop_bbox).convert("RGB")),
        ("generated", generated.crop(crop_bbox).convert("RGB")),
        ("final", final.crop(crop_bbox).convert("RGB")),
    ]
    panel_w = max(panel.width for _, panel in panels)
    panel_h = max(panel.height for _, panel in panels)
    label_h = 24
    canvas = Image.new("RGB", (panel_w * len(panels), panel_h + label_h), "white")
    draw = ImageDraw.Draw(canvas)
    for index, (label, image) in enumerate(panels):
        image = image.resize((panel_w, panel_h))
        x = index * panel_w
        draw.text((x + 8, 6), label, fill=(0, 0, 0))
        canvas.paste(image, (x, label_h))
    safe_variant = variant.replace("/", "_")
    debug_path = debug_dir / f"{record.path.stem}_inpaint_debug_{seed}_{safe_variant}.png"
    canvas.save(debug_path)
    return str(debug_path)


def context_crop_bbox_for_insert(insert_bbox, image_size, expand=CONTEXT_CROP_EXPAND, min_size=CONTEXT_CROP_MIN_SIZE):
    image_w, image_h = image_size
    x1, y1, x2, y2 = insert_bbox
    bw = x2 - x1
    bh = y2 - y1
    crop_size = int(max(min_size, bw * expand, bh * expand))
    crop_size = min(crop_size, image_w, image_h)
    cx = (x1 + x2) / 2
    cy = (y1 + y2) / 2
    left = int(round(cx - crop_size / 2))
    top = int(round(cy - crop_size / 2))
    left = max(0, min(image_w - crop_size, left))
    top = max(0, min(image_h - crop_size, top))
    return (left, top, left + crop_size, top + crop_size)


def map_bbox_to_resized_crop(bbox, crop_bbox, output_size=RESOLUTION):
    x1, y1, x2, y2 = bbox
    cx1, cy1, cx2, cy2 = crop_bbox
    scale_x = output_size / max(1, cx2 - cx1)
    scale_y = output_size / max(1, cy2 - cy1)
    return (
        int(round((x1 - cx1) * scale_x)),
        int(round((y1 - cy1) * scale_y)),
        int(round((x2 - cx1) * scale_x)),
        int(round((y2 - cy1) * scale_y)),
    )


def mask_stats_rgb(image, mask, threshold=0.18):
    arr = np.asarray(image.convert("RGB"), dtype=np.float32)
    mask_arr = np.asarray(mask.convert("L"), dtype=np.float32) / 255.0
    active = mask_arr > threshold
    if not np.any(active):
        return None, None
    pixels = arr[active]
    return pixels.mean(axis=0), pixels.std(axis=0) + 1e-6


def local_source_context_mask(mask, pad=COLOR_MATCH_CONTEXT_PAD):
    bbox = mask.getbbox()
    if bbox is None:
        return Image.new("L", mask.size, 0)
    x1, y1, x2, y2 = bbox
    x1 = max(0, x1 - pad)
    y1 = max(0, y1 - pad)
    x2 = min(mask.size[0], x2 + pad)
    y2 = min(mask.size[1], y2 + pad)
    context = Image.new("L", mask.size, 0)
    draw = ImageDraw.Draw(context)
    draw.rectangle((x1, y1, x2, y2), fill=255)
    context = ImageChops.subtract(context, mask.filter(ImageFilter.GaussianBlur(radius=2)))
    return context


def horizontal_context_mask(mask, band_ratio=EDGE_HORIZON_BAND_RATIO, pad=None):
    bbox = mask.getbbox()
    if bbox is None:
        return Image.new("L", mask.size, 0)
    width, height = mask.size
    x1, y1, x2, y2 = bbox
    pad = EDGE_BG_CONTEXT_PAD if pad is None else min(max(int(pad), 10), 20)
    band = max(8, min(20, int(height * band_ratio)))
    x1 = max(0, x1 - pad)
    x2 = min(width, x2 + pad)
    y1 = max(0, y1 - band)
    y2 = min(height, y2 + max(pad, band // 2))
    context = Image.new("L", mask.size, 0)
    draw = ImageDraw.Draw(context)
    draw.rectangle((x1, y1, x2, y2), fill=255)
    context = ImageChops.subtract(context, mask.filter(ImageFilter.GaussianBlur(radius=2)))
    return context


def blended_context_mask(mask, pad=COLOR_MATCH_CONTEXT_PAD):
    local = local_source_context_mask(mask, pad=pad)
    horizontal = horizontal_context_mask(mask, pad=pad)
    return ImageChops.lighter(local, horizontal)


def color_match_person_crop(source_crop, person_rgb, person_mask):
    if not COLOR_MATCH_PERSON_TO_SCENE:
        return person_rgb
    src_mean, src_std = mask_stats_rgb(source_crop, blended_context_mask(person_mask), threshold=0.12)
    gen_mean, gen_std = mask_stats_rgb(person_rgb, person_mask, threshold=0.18)
    if src_mean is None or gen_mean is None:
        return person_rgb
    arr = np.asarray(person_rgb.convert("RGB"), dtype=np.float32)
    corrected = (arr - gen_mean) * (src_std / gen_std) + src_mean
    corrected = np.clip(corrected, 0, 255)
    blended = arr * (1.0 - COLOR_MATCH_STRENGTH) + corrected * COLOR_MATCH_STRENGTH
    return Image.fromarray(np.clip(blended, 0, 255).astype(np.uint8), mode="RGB")


def high_frequency_std(image, mask=None, threshold=0.12):
    arr = np.asarray(image.convert("RGB"), dtype=np.float32)
    low = np.asarray(image.convert("RGB").filter(ImageFilter.GaussianBlur(radius=1.0)), dtype=np.float32)
    high = arr - low
    if mask is not None:
        mask_arr = np.asarray(mask.convert("L"), dtype=np.float32) / 255.0
        active = mask_arr > threshold
        if np.any(active):
            high = high[active]
    return float(np.std(high)) + 1e-6


def match_person_texture_to_scene(source_crop, person_rgb, person_mask):
    if not TEXTURE_MATCH_PERSON_TO_SCENE:
        return person_rgb
    context_mask = local_source_context_mask(person_mask, pad=TEXTURE_MATCH_CONTEXT_PAD)
    src_hf = high_frequency_std(source_crop, context_mask, threshold=0.10)
    gen_hf = high_frequency_std(person_rgb, person_mask, threshold=0.18)
    if gen_hf <= src_hf * 1.02 and gen_hf <= MAX_GENERATED_PERSON_SHARPNESS_STD:
        return person_rgb
    sharp_ratio = max(1.0, min(3.0, max(gen_hf / max(src_hf, 1e-6), gen_hf / max(MAX_GENERATED_PERSON_SHARPNESS_STD, 1e-6))))
    blur_radius = TEXTURE_MATCH_MIN_BLUR + (sharp_ratio - 1.0) / 2.0 * (TEXTURE_MATCH_MAX_BLUR - TEXTURE_MATCH_MIN_BLUR)
    softened = person_rgb.filter(ImageFilter.GaussianBlur(radius=blur_radius))
    arr = np.asarray(person_rgb.convert("RGB"), dtype=np.float32)
    soft = np.asarray(softened.convert("RGB"), dtype=np.float32)
    alpha_3 = foreground_harmonization_alpha(person_mask, TEXTURE_MATCH_STRENGTH, core_alpha=0.03)
    matched = arr * (1.0 - alpha_3) + soft * alpha_3
    return Image.fromarray(np.clip(matched, 0, 255).astype(np.uint8), mode="RGB")


def luminance_values(image, mask, threshold=0.12):
    arr = np.asarray(image.convert("RGB"), dtype=np.float32)
    mask_arr = np.asarray(mask.convert("L"), dtype=np.float32) / 255.0
    active = mask_arr > threshold
    if not np.any(active):
        return None
    luma = arr[..., 0] * 0.2126 + arr[..., 1] * 0.7152 + arr[..., 2] * 0.0722
    return luma[active]


def local_ring_mask(person_mask, pad=None):
    pad = max(COLOR_MATCH_CONTEXT_PAD, TEXTURE_MATCH_CONTEXT_PAD) if pad is None else pad
    return blended_context_mask(person_mask, pad=pad)


def luminance_mean_std(image, mask, threshold=0.12):
    values = luminance_values(image, mask, threshold=threshold)
    if values is None:
        return None, None
    return float(np.mean(values)), float(np.std(values)) + 1e-6


def saturation_values(image, mask, threshold=0.12):
    arr = np.asarray(image.convert("RGB"), dtype=np.float32) / 255.0
    mask_arr = np.asarray(mask.convert("L"), dtype=np.float32) / 255.0
    active = mask_arr > threshold
    if not np.any(active):
        return None
    max_c = arr.max(axis=2)
    min_c = arr.min(axis=2)
    sat = (max_c - min_c) / np.maximum(max_c, 1e-6)
    return sat[active]


def foreground_harmonization_alpha(person_mask, strength, core_alpha=None, edge_erode=None):
    mask_l = person_mask.convert("L")
    mask_arr = np.asarray(mask_l, dtype=np.float32) / 255.0
    if not np.any(mask_arr > 0.01):
        return np.zeros((*mask_arr.shape, 1), dtype=np.float32)
    edge_erode = FOREGROUND_HARMONIZATION_EDGE_ERODE if edge_erode is None else edge_erode
    core_alpha = FOREGROUND_HARMONIZATION_CORE_ALPHA if core_alpha is None else core_alpha
    hard = mask_l.point(lambda p: 255 if p >= PERSON_PASTE_HARD_THRESHOLD else 0)
    if edge_erode > 0:
        core = hard.filter(ImageFilter.MinFilter(edge_erode * 2 + 1))
    else:
        core = hard
    core_arr = np.asarray(core, dtype=np.float32) / 255.0
    edge_arr = np.clip(mask_arr - core_arr, 0.0, 1.0)
    alpha = edge_arr * strength + core_arr * min(strength, core_alpha)
    return np.expand_dims(np.clip(alpha, 0.0, 1.0), axis=2)


def local_color_transfer(source_crop, person_rgb, person_mask, strength=0.74):
    context_mask = local_ring_mask(person_mask)
    src_mean, src_std = mask_stats_rgb(source_crop, context_mask, threshold=0.10)
    gen_mean, gen_std = mask_stats_rgb(person_rgb, person_mask, threshold=0.18)
    if src_mean is None or gen_mean is None:
        return person_rgb
    arr = np.asarray(person_rgb.convert("RGB"), dtype=np.float32)
    alpha_3 = foreground_harmonization_alpha(person_mask, strength)
    ratio = np.clip(src_std / gen_std, 0.82, 1.18)
    corrected = (arr - gen_mean.reshape(1, 1, 3)) * ratio.reshape(1, 1, 3) + src_mean.reshape(1, 1, 3)
    matched = arr * (1.0 - alpha_3) + corrected * alpha_3
    return Image.fromarray(np.clip(matched, 0, 255).astype(np.uint8), mode="RGB")


def match_local_brightness(source_crop, person_rgb, person_mask, strength=0.72):
    context_mask = local_ring_mask(person_mask)
    src_mean, _ = luminance_mean_std(source_crop, context_mask, threshold=0.10)
    gen_mean, _ = luminance_mean_std(person_rgb, person_mask, threshold=0.18)
    if src_mean is None or gen_mean is None:
        return person_rgb
    shift = np.clip(src_mean - gen_mean, -18.0, 18.0)
    arr = np.asarray(person_rgb.convert("RGB"), dtype=np.float32)
    alpha_3 = foreground_harmonization_alpha(person_mask, strength)
    matched = arr + shift * alpha_3
    return Image.fromarray(np.clip(matched, 0, 255).astype(np.uint8), mode="RGB")


def match_local_contrast(source_crop, person_rgb, person_mask, strength=0.56):
    context_mask = local_ring_mask(person_mask)
    _, src_std = luminance_mean_std(source_crop, context_mask, threshold=0.10)
    gen_mean, gen_std = luminance_mean_std(person_rgb, person_mask, threshold=0.18)
    if src_std is None or gen_mean is None:
        return person_rgb
    ratio = np.clip(src_std / gen_std, 0.78, 1.16)
    arr = np.asarray(person_rgb.convert("RGB"), dtype=np.float32)
    alpha_3 = foreground_harmonization_alpha(person_mask, strength)
    matched_contrast = (arr - gen_mean) * ratio + gen_mean
    matched = arr * (1.0 - alpha_3) + matched_contrast * alpha_3
    return Image.fromarray(np.clip(matched, 0, 255).astype(np.uint8), mode="RGB")


def match_local_saturation(source_crop, person_rgb, person_mask, strength=0.45):
    context_mask = local_ring_mask(person_mask)
    src_sat = saturation_values(source_crop, context_mask, threshold=0.10)
    gen_sat = saturation_values(person_rgb, person_mask, threshold=0.18)
    if src_sat is None or gen_sat is None:
        return person_rgb
    src_mean = float(np.mean(src_sat))
    gen_mean = float(np.mean(gen_sat)) + 1e-6
    ratio = np.clip(src_mean / gen_mean, 0.72, 1.12)
    arr = np.asarray(person_rgb.convert("RGB"), dtype=np.float32)
    gray = np.sum(arr * np.array([0.2126, 0.7152, 0.0722], dtype=np.float32).reshape(1, 1, 3), axis=2, keepdims=True)
    sat_matched = gray + (arr - gray) * ratio
    alpha_3 = foreground_harmonization_alpha(person_mask, strength)
    matched = arr * (1.0 - alpha_3) + sat_matched * alpha_3
    return Image.fromarray(np.clip(matched, 0, 255).astype(np.uint8), mode="RGB")


def add_sensor_noise(source_crop, person_rgb, person_mask, strength=0.95):
    context_mask = local_ring_mask(person_mask)
    src_hf = high_frequency_std(source_crop, context_mask, threshold=0.10)
    gen_hf = high_frequency_std(person_rgb, person_mask, threshold=0.18)
    if gen_hf >= src_hf * 0.92:
        return person_rgb
    noise_std = float(np.clip((src_hf - gen_hf) * 0.38, 0.0, 3.0))
    if noise_std <= 0.05:
        return person_rgb
    arr = np.asarray(person_rgb.convert("RGB"), dtype=np.float32)
    alpha_3 = foreground_harmonization_alpha(person_mask, strength, core_alpha=0.04)
    seed = ((person_rgb.size[0] * 73856093) ^ (person_rgb.size[1] * 19349663)) & 0xFFFFFFFF
    rng = np.random.default_rng(seed)
    noise = rng.normal(0.0, noise_std, arr.shape).astype(np.float32)
    matched = arr + noise * alpha_3
    return Image.fromarray(np.clip(matched, 0, 255).astype(np.uint8), mode="RGB")


def gaussian_blur_person(person_rgb, person_mask, sigma=0.60, strength=0.55):
    blurred = person_rgb.filter(ImageFilter.GaussianBlur(radius=sigma))
    arr = np.asarray(person_rgb.convert("RGB"), dtype=np.float32)
    soft = np.asarray(blurred.convert("RGB"), dtype=np.float32)
    alpha_3 = foreground_harmonization_alpha(person_mask, strength, core_alpha=0.03)
    matched = arr * (1.0 - alpha_3) + soft * alpha_3
    return Image.fromarray(np.clip(matched, 0, 255).astype(np.uint8), mode="RGB")


def apply_subtle_scene_tone_filter(source_crop, person_rgb, person_mask):
    if not PERSON_TONE_FILTER_ENABLED:
        return person_rgb
    context_mask = local_ring_mask(person_mask)
    src_mean, _ = mask_stats_rgb(source_crop, context_mask, threshold=0.10)
    gen_mean, _ = mask_stats_rgb(person_rgb, person_mask, threshold=0.18)
    if src_mean is None or gen_mean is None:
        return person_rgb

    src_cast = src_mean - float(np.mean(src_mean))
    gen_cast = gen_mean - float(np.mean(gen_mean))
    color_shift = np.clip(
        (src_cast - gen_cast) * 0.45,
        -PERSON_TONE_FILTER_MAX_COLOR_SHIFT,
        PERSON_TONE_FILTER_MAX_COLOR_SHIFT,
    )

    src_luma, _ = luminance_mean_std(source_crop, context_mask, threshold=0.10)
    gen_luma, _ = luminance_mean_std(person_rgb, person_mask, threshold=0.18)
    brightness_shift = 0.0
    if src_luma is not None and gen_luma is not None:
        brightness_shift = float(np.clip(
            (src_luma - gen_luma) * 0.18,
            -PERSON_TONE_FILTER_MAX_BRIGHTNESS_SHIFT,
            PERSON_TONE_FILTER_MAX_BRIGHTNESS_SHIFT,
        ))

    arr = np.asarray(person_rgb.convert("RGB"), dtype=np.float32)
    filtered = arr + color_shift.reshape(1, 1, 3) + brightness_shift
    alpha_3 = foreground_harmonization_alpha(
        person_mask,
        PERSON_TONE_FILTER_STRENGTH,
        core_alpha=PERSON_TONE_FILTER_CORE_STRENGTH,
        edge_erode=1,
    )
    matched = arr * (1.0 - alpha_3) + filtered * alpha_3
    return Image.fromarray(np.clip(matched, 0, 255).astype(np.uint8), mode="RGB")


def harmonize_person_to_scene(source_crop, person_rgb, person_mask):
    person_rgb = color_match_person_crop(source_crop, person_rgb, person_mask)
    person_rgb = match_person_texture_to_scene(source_crop, person_rgb, person_mask)
    person_rgb = apply_subtle_scene_tone_filter(source_crop, person_rgb, person_mask)
    person_rgb = neutralize_person_edge_halo(source_crop, person_rgb, person_mask)
    person_rgb = soften_dark_person_edge(source_crop, person_rgb, person_mask)
    return person_rgb


def match_person_appearance_to_scene(source_crop, person_rgb, person_mask):
    context_mask = blended_context_mask(person_mask, pad=max(COLOR_MATCH_CONTEXT_PAD, TEXTURE_MATCH_CONTEXT_PAD))
    src_mean, _ = mask_stats_rgb(source_crop, context_mask, threshold=0.10)
    gen_mean, _ = mask_stats_rgb(person_rgb, person_mask, threshold=0.18)
    if src_mean is None or gen_mean is None:
        return person_rgb

    arr = np.asarray(person_rgb.convert("RGB"), dtype=np.float32)
    alpha = np.asarray(person_mask.convert("L"), dtype=np.float32) / 255.0
    alpha_3 = np.expand_dims(np.clip(alpha, 0.0, 1.0), axis=2)

    # Color temperature: match the local red-vs-blue cast without repainting clothing colors.
    src_temp = float(src_mean[0] - src_mean[2])
    gen_temp = float(gen_mean[0] - gen_mean[2])
    temp_shift = np.clip((src_temp - gen_temp) * 0.28, -10.0, 10.0)
    temp_matched = arr.copy()
    temp_matched[..., 0] += temp_shift
    temp_matched[..., 2] -= temp_shift
    arr = arr * (1.0 - alpha_3 * 0.55) + temp_matched * (alpha_3 * 0.55)

    # Contrast: match local luminance spread so the person does not look too crisp or flat.
    src_luma = luminance_values(source_crop, context_mask, threshold=0.10)
    gen_luma = luminance_values(person_rgb, person_mask, threshold=0.18)
    if src_luma is not None and gen_luma is not None:
        src_std = float(np.std(src_luma)) + 1e-6
        gen_std = float(np.std(gen_luma)) + 1e-6
        ratio = np.clip(src_std / gen_std, 0.78, 1.16)
        contrast_matched = (arr - gen_mean.reshape(1, 1, 3)) * ratio + gen_mean.reshape(1, 1, 3)
        arr = arr * (1.0 - alpha_3 * 0.38) + contrast_matched * (alpha_3 * 0.38)

    # Noise/grain: add only when the generated person is cleaner than the surrounding crop.
    src_hf = high_frequency_std(source_crop, context_mask, threshold=0.10)
    gen_hf = high_frequency_std(Image.fromarray(np.clip(arr, 0, 255).astype(np.uint8), mode="RGB"), person_mask, threshold=0.18)
    if gen_hf < src_hf * 0.92:
        noise_std = float(np.clip((src_hf - gen_hf) * 0.32, 0.0, 2.2))
        if noise_std > 0.05:
            seed = ((person_rgb.size[0] * 73856093) ^ (person_rgb.size[1] * 19349663)) & 0xFFFFFFFF
            rng = np.random.default_rng(seed)
            noise = rng.normal(0.0, noise_std, arr.shape).astype(np.float32)
            arr = arr + noise * alpha_3 * 0.85

    return Image.fromarray(np.clip(arr, 0, 255).astype(np.uint8), mode="RGB")



def soften_dark_person_edge(source_crop, person_rgb, person_mask):
    mask_l = person_mask.convert("L")
    mask_arr = np.asarray(mask_l, dtype=np.float32) / 255.0
    if not np.any(mask_arr > 0.02):
        return person_rgb

    # Only touch transparent/soft edge pixels. Do not process the hard inner
    # silhouette, otherwise background tone can make the person look thinner.
    hard_threshold = PERSON_PASTE_HARD_THRESHOLD / 255.0
    edge = ((mask_arr > EDGE_HALO_MIN_ALPHA) & (mask_arr < hard_threshold)).astype(np.float32)
    if not np.any(edge > 0.02):
        return person_rgb

    arr = np.asarray(person_rgb.convert("RGB"), dtype=np.float32)
    blurred_person = np.asarray(person_rgb.convert("RGB").filter(ImageFilter.GaussianBlur(radius=0.55)), dtype=np.float32)
    background_tone = blended_background_mean_map(source_crop, person_mask)
    target = blurred_person * 0.72 + background_tone * 0.28

    luma = np.sum(arr * np.array([0.2126, 0.7152, 0.0722], dtype=np.float32).reshape(1, 1, 3), axis=2)
    bg_luma_map = np.sum(background_tone * np.array([0.2126, 0.7152, 0.0722], dtype=np.float32).reshape(1, 1, 3), axis=2)
    dark_edge_boost = np.clip((bg_luma_map - luma) / 95.0, 0.0, 0.35)
    alpha = np.clip(edge * (0.22 + dark_edge_boost), 0.0, 0.42)
    alpha_3 = np.expand_dims(alpha, axis=2)
    softened = arr * (1.0 - alpha_3) + target * alpha_3
    return Image.fromarray(np.clip(softened, 0, 255).astype(np.uint8), mode="RGB")


def neutralize_person_edge_halo(source_crop, person_rgb, person_mask):
    if not EDGE_HALO_NEUTRALIZE:
        return person_rgb
    mask_l = person_mask.convert("L")
    mask_arr = np.asarray(mask_l, dtype=np.float32) / 255.0
    filter_size = max(3, int(EDGE_HALO_WIDTH) * 2 + 1)
    dilated = np.asarray(mask_l.filter(ImageFilter.MaxFilter(filter_size)), dtype=np.float32) / 255.0
    eroded = np.asarray(mask_l.filter(ImageFilter.MinFilter(filter_size)), dtype=np.float32) / 255.0
    ring = np.clip(dilated - eroded, 0.0, 1.0)
    soft_edge = ((mask_arr >= EDGE_HALO_MIN_ALPHA) & (mask_arr <= EDGE_HALO_MAX_ALPHA)).astype(np.float32)
    edge_alpha = np.clip(np.maximum(ring, soft_edge) * EDGE_HALO_COLOR_MATCH_STRENGTH, 0.0, 1.0)
    edge_active = edge_alpha > 0.02
    if not np.any(edge_active):
        return person_rgb

    context_mask = blended_context_mask(person_mask, pad=COLOR_MATCH_CONTEXT_PAD)
    src_mean, src_std = mask_stats_rgb(source_crop, context_mask, threshold=0.10)
    if src_mean is None:
        return person_rgb

    arr = np.asarray(person_rgb.convert("RGB"), dtype=np.float32)
    edge_pixels = arr[edge_active]
    edge_mean = edge_pixels.mean(axis=0)
    edge_std = edge_pixels.std(axis=0) + 1e-6
    corrected = (arr - edge_mean) * (src_std / edge_std) + src_mean
    corrected = np.clip(corrected, 0, 255)
    edge_alpha_3 = np.expand_dims(edge_alpha, axis=2)
    matched = arr * (1.0 - edge_alpha_3) + corrected * edge_alpha_3
    return Image.fromarray(np.clip(matched, 0, 255).astype(np.uint8), mode="RGB")


def tight_person_edge_alpha(person_mask):
    mask_l = person_mask.convert("L")
    mask_arr = np.asarray(mask_l, dtype=np.float32) / 255.0
    hard = mask_l.point(lambda p: 255 if p >= PERSON_PASTE_HARD_THRESHOLD else 0)
    filter_size = max(3, int(EDGE_HALO_WIDTH) * 2 + 1)
    dilated = np.asarray(hard.filter(ImageFilter.MaxFilter(filter_size)), dtype=np.float32) / 255.0
    inner_filter_size = max(3, 2 * max(1, int(EDGE_HALO_WIDTH) + 1) + 1)
    eroded = np.asarray(hard.filter(ImageFilter.MinFilter(inner_filter_size)), dtype=np.float32) / 255.0
    hard_arr = np.asarray(hard, dtype=np.float32) / 255.0

    outside_ring = np.clip(dilated - hard_arr, 0.0, 1.0)
    inner_ring = np.clip(hard_arr - eroded, 0.0, 1.0)
    soft_transition = ((mask_arr >= EDGE_HALO_MIN_ALPHA) & (mask_arr < PERSON_PASTE_HARD_THRESHOLD / 255.0)).astype(np.float32)
    edge = np.maximum(outside_ring, soft_transition) * (1.0 - hard_arr)
    edge = np.maximum(edge, inner_ring * 0.72)
    return np.clip(edge * EDGE_HALO_COLOR_MATCH_STRENGTH, 0.0, 1.0)


def local_background_mean_map(image, person_mask, radius=EDGE_LOCAL_BG_RADIUS, exclude_threshold=0.04):
    arr = np.asarray(image.convert("RGB"), dtype=np.float32)
    mask_arr = np.asarray(person_mask.convert("L"), dtype=np.float32) / 255.0
    bg_weight = (mask_arr <= exclude_threshold).astype(np.float32)
    if not np.any(bg_weight > 0):
        fallback = arr.reshape(-1, 3).mean(axis=0)
        return np.zeros_like(arr) + fallback.reshape(1, 1, 3)

    bg_weight_u8 = np.clip(bg_weight * 255.0, 0, 255).astype(np.uint8)
    denom = np.asarray(
        Image.fromarray(bg_weight_u8).filter(ImageFilter.GaussianBlur(radius=radius)),
        dtype=np.float32,
    ) / 255.0
    local_mean = np.zeros_like(arr)
    for channel in range(3):
        weighted = np.clip(arr[..., channel] * bg_weight, 0, 255).astype(np.uint8)
        numer = np.asarray(
            Image.fromarray(weighted).filter(ImageFilter.GaussianBlur(radius=radius)),
            dtype=np.float32,
        )
        local_mean[..., channel] = numer / np.maximum(denom, 1e-6)

    fallback_pixels = arr[bg_weight > 0]
    fallback = fallback_pixels.mean(axis=0) if fallback_pixels.size else arr.reshape(-1, 3).mean(axis=0)
    weak = denom < 0.015
    if np.any(weak):
        local_mean[weak] = fallback
    return local_mean


def bbox_background_fallback_mean(image, person_mask, exclude_threshold=0.04):
    arr = np.asarray(image.convert("RGB"), dtype=np.float32)
    mask_arr = np.asarray(person_mask.convert("L"), dtype=np.float32) / 255.0
    bbox = person_mask.getbbox()
    if bbox is None:
        fallback = arr.reshape(-1, 3).mean(axis=0)
        return np.zeros_like(arr) + fallback.reshape(1, 1, 3)
    x1, y1, x2, y2 = bbox
    pad = int(EDGE_BG_CONTEXT_PAD)
    x1 = max(0, x1 - pad)
    y1 = max(0, y1 - pad)
    x2 = min(arr.shape[1], x2 + pad)
    y2 = min(arr.shape[0], y2 + pad)
    crop = arr[y1:y2, x1:x2]
    crop_mask = mask_arr[y1:y2, x1:x2]
    bg_pixels = crop[crop_mask <= exclude_threshold]
    if bg_pixels.size == 0:
        bg_pixels = crop.reshape(-1, 3)
    fallback = bg_pixels.mean(axis=0) if bg_pixels.size else arr.reshape(-1, 3).mean(axis=0)
    return np.zeros_like(arr) + fallback.reshape(1, 1, 3)


def horizontal_background_mean_map(image, person_mask, band_ratio=EDGE_HORIZON_BAND_RATIO, exclude_threshold=0.04):
    arr = np.asarray(image.convert("RGB"), dtype=np.float32)
    mask_arr = np.asarray(person_mask.convert("L"), dtype=np.float32) / 255.0
    h, w = mask_arr.shape
    bbox = person_mask.getbbox()
    bg_weight = (mask_arr <= exclude_threshold).astype(np.float32)
    if bbox is not None:
        x1, y1, x2, y2 = bbox
        pad = int(EDGE_BG_CONTEXT_PAD)
        band = max(8, min(20, int(h * band_ratio)))
        band_mask = np.zeros_like(bg_weight)
        x1 = max(0, x1 - pad)
        x2 = min(w, x2 + pad)
        y1 = max(0, y1 - band)
        y2 = min(h, y2 + max(pad, band // 2))
        band_mask[y1:y2, x1:x2] = 1.0
        bg_weight *= band_mask
    if not np.any(bg_weight > 0):
        return bbox_background_fallback_mean(image, person_mask, exclude_threshold=exclude_threshold)
    fallback = arr[bg_weight > 0].mean(axis=0)
    row_weight = bg_weight.sum(axis=1, keepdims=True)
    row_sum = (arr * bg_weight[..., None]).sum(axis=1)
    row_mean = row_sum / np.maximum(row_weight, 1e-6)
    row_mean[row_weight[:, 0] <= 0] = fallback
    horizon = np.repeat(row_mean[:, None, :], w, axis=1)
    return horizon


def blended_background_mean_map(image, person_mask):
    return horizontal_background_mean_map(image, person_mask)

def match_pasted_edge_to_composite_mean(result_crop, person_mask):
    if not EDGE_HALO_NEUTRALIZE:
        return result_crop
    edge_alpha = tight_person_edge_alpha(person_mask)
    edge_active = edge_alpha > 0.02
    if not np.any(edge_active):
        return result_crop

    arr = np.asarray(result_crop.convert("RGB"), dtype=np.float32)
    blurred = np.asarray(result_crop.convert("RGB").filter(ImageFilter.GaussianBlur(radius=0.45)), dtype=np.float32)
    local_mean = blended_background_mean_map(result_crop, person_mask)
    softened = arr * 0.52 + blurred * 0.48
    mean_matched = softened * 0.55 + local_mean * 0.45
    edge_alpha_3 = np.expand_dims(np.clip(edge_alpha, 0.0, 1.0), axis=2)
    matched = arr * (1.0 - edge_alpha_3) + mean_matched * edge_alpha_3
    return Image.fromarray(np.clip(matched, 0, 255).astype(np.uint8), mode="RGB")


def addit_subject_guided_blend_mask(person_mask, variant=None):
    """Outside-only Add-it proxy mask.

    Keep the accepted person core fully target-side. Blur/blend only the narrow
    outside transition ring and the foot-contact shadow region, so background
    cannot eat into the person silhouette.
    """
    if not ADDIT_SUBJECT_GUIDED_BLEND_PROXY:
        return person_mask.convert("L")
    mask = person_mask.convert("L")
    bbox = mask.getbbox()
    if bbox is None:
        return mask
    x1, y1, x2, y2 = bbox
    w = max(1, x2 - x1)
    h = max(1, y2 - y1)

    hard = mask.point(lambda p: 255 if p >= PERSON_PASTE_HARD_THRESHOLD else 0)
    dilate_px = max(0, int(round(ADDIT_BLEND_CONTEXT_DILATE)))
    if dilate_px > 0:
        dilated = hard.filter(ImageFilter.MaxFilter(dilate_px * 2 + 1))
    else:
        dilated = hard

    outside_ring = ImageChops.subtract(dilated, hard)
    outside_soft = outside_ring.filter(ImageFilter.GaussianBlur(radius=max(0.0, ADDIT_BLEND_EDGE_RADIUS)))
    hard_arr = np.asarray(hard, dtype=np.float32) / 255.0
    outside_arr = np.asarray(outside_soft, dtype=np.float32) / 255.0
    outside_arr = outside_arr * (1.0 - hard_arr)
    outside_soft = Image.fromarray(np.clip(outside_arr * 255.0, 0, 255).astype(np.uint8), mode="L")

    shadow = Image.new("L", mask.size, 0)
    draw = ImageDraw.Draw(shadow)
    shadow_h = max(3, int(round(h * ADDIT_BLEND_SHADOW_EXTENSION)))
    sx1 = max(0, int(round(x1 - w * 0.18)))
    sx2 = min(mask.size[0], int(round(x2 + w * 0.18)))
    sy1 = max(0, int(round(y2 - h * 0.02)))
    sy2 = min(mask.size[1], int(round(y2 + shadow_h)))
    if sx2 > sx1 and sy2 > sy1:
        draw.ellipse((sx1, sy1, sx2, sy2), fill=int(255 * max(0.0, min(1.0, ADDIT_BLEND_SHADOW_ALPHA))))
        shadow = shadow.filter(ImageFilter.GaussianBlur(radius=max(0.0, ADDIT_BLEND_SHADOW_BLUR)))
        shadow_arr = np.asarray(shadow, dtype=np.float32) / 255.0
        shadow_arr = shadow_arr * (1.0 - hard_arr)
        shadow = Image.fromarray(np.clip(shadow_arr * 255.0, 0, 255).astype(np.uint8), mode="L")

    return ImageChops.lighter(ImageChops.lighter(hard, outside_soft), shadow)


def apply_addit_subject_guided_blend(source_crop, target_crop, person_mask, variant=None):
    if not ADDIT_CONCEPT_ENABLED or not ADDIT_SUBJECT_GUIDED_BLEND_PROXY:
        return target_crop
    blend_mask = addit_subject_guided_blend_mask(person_mask, variant=variant)
    return Image.composite(target_crop.convert("RGB"), source_crop.convert("RGB"), blend_mask)


def seamless_clone_person_crop(source_crop, person_rgb, person_mask):
    if not USE_SEAMLESS_CLONE or cv2 is None:
        return None, False
    mask = clean_binary_person_mask(person_mask, keep_components=ACCESSORY_KEEP_COMPONENTS)
    if mask_area_ratio(mask) < SEAMLESS_CLONE_MIN_MASK_AREA_RATIO:
        return None, False
    bbox = mask.getbbox()
    if bbox is None:
        return None, False
    x1, y1, x2, y2 = bbox
    center = (int(round((x1 + x2) / 2.0)), int(round((y1 + y2) / 2.0)))
    if center[0] <= 0 or center[0] >= source_crop.size[0] or center[1] <= 0 or center[1] >= source_crop.size[1]:
        return None, False
    try:
        src = cv2.cvtColor(np.asarray(source_crop.convert("RGB")), cv2.COLOR_RGB2BGR)
        obj = cv2.cvtColor(np.asarray(person_rgb.convert("RGB")), cv2.COLOR_RGB2BGR)
        m = np.asarray(mask, dtype=np.uint8)
        mode = cv2.NORMAL_CLONE if SEAMLESS_CLONE_MODE == "normal" else cv2.MIXED_CLONE
        cloned = cv2.seamlessClone(obj, src, m, center, mode)
        return Image.fromarray(cv2.cvtColor(cloned, cv2.COLOR_BGR2RGB), mode="RGB"), True
    except Exception as exc:
        print("seamlessClone failed; falling back to alpha paste:", type(exc).__name__, exc)
        return None, False


def preserve_foreground_after_seamless(cloned_crop, person_rgb, person_mask):
    if not SEAMLESS_EDGE_ONLY_BLEND:
        return cloned_crop
    mask = person_mask.convert("L")
    if mask.getbbox() is None:
        return cloned_crop
    hard = mask.point(lambda p: 255 if p >= PERSON_PASTE_HARD_THRESHOLD else 0)
    core = hard
    if SEAMLESS_FOREGROUND_CORE_ERODE > 0:
        core = core.filter(ImageFilter.MinFilter(SEAMLESS_FOREGROUND_CORE_ERODE * 2 + 1))
    core = core.filter(ImageFilter.GaussianBlur(radius=0.65))
    alpha = np.asarray(core, dtype=np.float32) / 255.0
    alpha = np.expand_dims(np.clip(alpha * SEAMLESS_FOREGROUND_PRESERVE_STRENGTH, 0.0, 1.0), axis=2)
    cloned_arr = np.asarray(cloned_crop.convert("RGB"), dtype=np.float32)
    person_arr = np.asarray(person_rgb.convert("RGB"), dtype=np.float32)
    preserved = cloned_arr * (1.0 - alpha) + person_arr * alpha
    return Image.fromarray(np.clip(preserved, 0, 255).astype(np.uint8), mode="RGB")


def bbox_mask_from_bboxes(image_size, bboxes, padding=0, blur=0):
    mask = Image.new("L", image_size, 0)
    draw = ImageDraw.Draw(mask)
    width, height = image_size
    for bbox in bboxes:
        x1, y1, x2, y2 = [int(round(v)) for v in bbox]
        x1 = max(0, x1 - padding)
        y1 = max(0, y1 - padding)
        x2 = min(width, x2 + padding)
        y2 = min(height, y2 + padding)
        if x2 > x1 and y2 > y1:
            draw.rectangle((x1, y1, x2, y2), fill=255)
    if blur and blur > 0:
        mask = mask.filter(ImageFilter.GaussianBlur(radius=blur))
    return mask


def vehicle_is_foreground_occluder(person_bbox, vehicle_bbox):
    inter = bbox_intersection_area(person_bbox, vehicle_bbox)
    if inter <= 0:
        return False, 0.0
    person_area = max(1.0, bbox_area(person_bbox))
    vehicle_area = max(1.0, bbox_area(vehicle_bbox))
    overlap_ratio = inter / min(person_area, vehicle_area)
    if overlap_ratio < MIN_OCCLUDER_OVERLAP_RATIO:
        return False, overlap_ratio
    person_h = max(1.0, person_bbox[3] - person_bbox[1])
    vehicle_h = max(1.0, vehicle_bbox[3] - vehicle_bbox[1])
    vehicle_close_enough = vehicle_bbox[3] >= person_bbox[3] - VEHICLE_OCCLUDER_MAX_FOOT_Y_DELTA
    vehicle_large_enough = vehicle_h >= person_h * OCCLUDER_MIN_HEIGHT_RATIO
    return bool(vehicle_close_enough or vehicle_large_enough), overlap_ratio


def build_foreground_occluder_mask(image_size, person_bbox, existing_person_bboxes=None, existing_vehicle_bboxes=None, variant=None):
    if not OCCLUSION_AWARE_COMPOSITE or person_bbox is None:
        return None, {"foreground_occlusion_used": False, "foreground_occluder_count": 0, "foreground_occlusion_overlap_ratio": 0.0}
    occluders = []
    max_overlap = 0.0
    for old_bbox in existing_person_bboxes or []:
        inter = bbox_intersection_area(person_bbox, old_bbox)
        if inter <= 0:
            continue
        depth_ok, overlap_ratio = person_overlap_depth_ok(person_bbox, old_bbox)
        if not depth_ok:
            occluders.append(old_bbox)
            max_overlap = max(max_overlap, overlap_ratio)
    for vehicle_bbox in existing_vehicle_bboxes or []:
        is_occluder, overlap_ratio = vehicle_is_foreground_occluder(person_bbox, vehicle_bbox)
        if is_occluder:
            occluders.append(vehicle_bbox)
            max_overlap = max(max_overlap, overlap_ratio)
    if not occluders:
        return None, {"foreground_occlusion_used": False, "foreground_occluder_count": 0, "foreground_occlusion_overlap_ratio": 0.0}
    mask = bbox_mask_from_bboxes(
        image_size,
        occluders,
        padding=OCCLUSION_MASK_BBOX_PADDING,
        blur=OCCLUSION_MASK_BLUR_RADIUS,
    )
    return mask, {
        "foreground_occlusion_used": True,
        "foreground_occluder_count": len(occluders),
        "foreground_occlusion_overlap_ratio": round(float(max_overlap), 4),
    }


def apply_foreground_occlusion_mask(person_mask, occlusion_mask, crop_bbox):
    if occlusion_mask is None:
        return person_mask, None, {"foreground_occlusion_removed_ratio": 0.0}
    cx1, cy1, cx2, cy2 = crop_bbox
    occlusion_crop = occlusion_mask.crop((cx1, cy1, cx2, cy2)).resize(person_mask.size, Image.BILINEAR)
    person_arr = np.asarray(person_mask.convert("L"), dtype=np.float32)
    occ_arr = np.asarray(occlusion_crop.convert("L"), dtype=np.float32)
    active = person_arr > 8
    if not np.any(active):
        return person_mask, occlusion_crop, {"foreground_occlusion_removed_ratio": 0.0}
    removed_ratio = float(np.mean((occ_arr > 16) & active) / max(1e-6, np.mean(active)))
    if removed_ratio > MAX_OCCLUSION_REMOVED_MASK_RATIO:
        return person_mask, None, {"foreground_occlusion_removed_ratio": round(removed_ratio, 4), "foreground_occlusion_skipped": True}
    kept = np.clip(person_arr * (1.0 - occ_arr / 255.0), 0, 255).astype(np.uint8)
    return Image.fromarray(kept, mode="L"), occlusion_crop, {"foreground_occlusion_removed_ratio": round(removed_ratio, 4)}


def paste_crop_person_to_original(source, generated_crop, person_mask_crop, crop_bbox, occlusion_mask=None):
    cx1, cy1, cx2, cy2 = crop_bbox
    crop_w = cx2 - cx1
    crop_h = cy2 - cy1
    person_rgb = generated_crop.resize((crop_w, crop_h), Image.BICUBIC)
    person_mask = prepare_person_paste_mask(person_mask_crop, (crop_w, crop_h))
    person_mask, occlusion_crop, occlusion_meta = apply_foreground_occlusion_mask(person_mask, occlusion_mask, crop_bbox)
    source_crop = source.crop(crop_bbox).resize((crop_w, crop_h), Image.BICUBIC)
    person_rgb = harmonize_person_to_scene(source_crop, person_rgb, person_mask)
    cloned_crop, seamless_used = seamless_clone_person_crop(source_crop, person_rgb, person_mask)
    result = source.copy()
    blend_meta = {
        "seamless_clone_used": bool(seamless_used),
        "fallback_alpha_paste": not bool(seamless_used),
        "fallback_alpha_used": not bool(seamless_used),
        "foreground_preserved_after_seamless": False,
        "edge_local_bg_match_used": False,
        **occlusion_meta,
    }
    if seamless_used and cloned_crop is not None:
        cloned_crop = preserve_foreground_after_seamless(cloned_crop, person_rgb, person_mask)
        blend_meta["foreground_preserved_after_seamless"] = True
        result.paste(cloned_crop, (cx1, cy1))
    else:
        result.paste(person_rgb, (cx1, cy1), person_mask)

    result_crop = result.crop(crop_bbox)
    result_crop = match_pasted_edge_to_composite_mean(result_crop, person_mask)
    blend_meta["edge_local_bg_match_used"] = True
    result_crop = apply_addit_subject_guided_blend(source_crop, result_crop, person_mask)
    if occlusion_crop is not None:
        result_crop = Image.composite(source_crop, result_crop, occlusion_crop)
    result.paste(result_crop, (cx1, cy1))
    return result, person_mask, blend_meta


def generate_context_person_composite_with_pipe(pipe, source, record, variant, prompt, negative_prompt, seed, device, strength, guidance_scale, num_inference_steps, debug_index=None):
    crop_bbox = (0, 0, source.size[0], source.size[1])
    crop_source = source.resize((RESOLUTION, RESOLUTION), Image.LANCZOS)
    original = load_source_image(record.path)
    existing_person_bboxes = load_person_bboxes_for_crop(record, original.size, resolution=source.size[0])
    existing_vehicle_bboxes = load_vehicle_bboxes_for_crop(record, original.size, resolution=source.size[0])
    semantic_masks = semantic_placement_masks(source, record, device=device)
    depth_map = estimate_depth_map(source, device="cpu")
    generation_source = crop_source
    mask_image = Image.new("L", crop_source.size, 0)
    generator_device = device if str(device).startswith("cuda") else "cpu"
    max_retries = variant_retry_budget(variant)
    last_reject_reason = None
    last_reject_meta = default_scale_correction_metadata()
    scale_unrecoverable_streak = 0
    attempt_history = []

    for attempt in range(max_retries + 1):
        attempt_seed = seed + attempt * 9973
        generator = torch.Generator(device=generator_device).manual_seed(attempt_seed)
        attempt_prompt, attempt_negative_prompt, attempt_strength, attempt_guidance, _attempt_margin = build_retry_config(
            prompt,
            negative_prompt,
            last_reject_reason,
            strength,
            guidance_scale,
            CONTEXT_CROP_EXPAND,
            attempt,
        )
        generated_crop = pipe(
            prompt=attempt_prompt,
            negative_prompt=attempt_negative_prompt,
            image=generation_source,
            strength=attempt_strength,
            guidance_scale=attempt_guidance,
            num_inference_steps=num_inference_steps,
            generator=generator,
        ).images[0].resize(crop_source.size)

        person_mask_crop, detected_bbox, reject_reason, scale_meta, corrected_generated_crop = select_new_generated_person_mask(
            generated_crop,
            existing_person_bboxes=existing_person_bboxes,
            semantic_masks=semantic_masks,
            variant=variant,
            background_image=crop_source,
            depth_map=depth_map,
        )
        reject_reason = normalize_reject_reason(reject_reason)
        scale_meta = dict(scale_meta or {})
        scale_meta["retry_attempts"] = attempt
        scale_meta["attempt"] = attempt

        if person_mask_crop is None:
            if reject_reason == "scale_unrecoverable":
                scale_unrecoverable_streak += 1
            else:
                scale_unrecoverable_streak = 0
            scale_meta["scale_unrecoverable_streak"] = scale_unrecoverable_streak
            scale_meta["reject_reason"] = reject_reason
            scale_meta["last_reject_reason"] = reject_reason
            attempt_history.append(scale_meta)
            last_reject_reason = reject_reason
            last_reject_meta = scale_meta
            if should_retry(reject_reason, attempt, max_retries, scale_meta):
                print(
                    f"Rejected generated person ({reject_reason}) on attempt {attempt + 1}; "
                    "retrying with adaptive generation params."
                )
                continue
            break

        insert_bbox = tuple(int(round(v)) for v in detected_bbox)
        outside_ratio = mask_outside_bbox_ratio(person_mask_crop, insert_bbox)
        if outside_ratio > MAX_MASK_OUTSIDE_INSERTION_RATIO:
            reject_reason = "partial_or_cropped"
            scale_unrecoverable_streak = 0
            scale_meta.update({
                "reject_reason": reject_reason,
                "last_reject_reason": reject_reason,
                "mask_outside_bbox_ratio": round(outside_ratio, 4),
                "scale_unrecoverable_streak": scale_unrecoverable_streak,
            })
            attempt_history.append(scale_meta)
            last_reject_reason = reject_reason
            last_reject_meta = scale_meta
            print(f"Generated person mask is unstable around detected bbox (outside_ratio={outside_ratio:.2f}).")
            if should_retry(reject_reason, attempt, max_retries, scale_meta):
                print(f"Retry attempt {attempt + 1} because: {reject_reason}")
                continue
            break

        person_mask_crop = constrain_mask_to_bbox(person_mask_crop, insert_bbox)
        if corrected_generated_crop is not None:
            generated_crop = corrected_generated_crop
        occlusion_mask, occlusion_meta = build_foreground_occluder_mask(
            source.size,
            detected_bbox,
            existing_person_bboxes=existing_person_bboxes,
            existing_vehicle_bboxes=existing_vehicle_bboxes,
            variant=variant,
        )
        insert_meta = {
            "expected_person_height": insert_bbox[3] - insert_bbox[1],
            "expected_person_width": insert_bbox[2] - insert_bbox[0],
            "ground_y": insert_bbox[3],
            "img2img_first": True,
            "retry_attempts": attempt,
            **occlusion_meta,
            **scale_meta,
        }
        result, pasted_mask, blend_meta = paste_crop_person_to_original(source, generated_crop, person_mask_crop, crop_bbox, occlusion_mask=occlusion_mask)
        insert_meta.update(blend_meta)
        is_valid, final_reason, final_meta = validate_composite_result(
            source,
            result,
            pasted_mask,
            variant,
            insert_bbox,
            insert_meta,
        )
        final_meta["attempt"] = attempt
        final_meta["scale_unrecoverable_streak"] = scale_unrecoverable_streak
        attempt_history.append(final_meta)

        if is_valid:
            if attempt > 0:
                print(
                    f"Recovered accepted composite after retry {attempt} for {record.path.name} "
                    f"using strength={attempt_strength:.2f}, guidance={attempt_guidance:.2f}."
                )
            debug_mask = Image.new("L", source.size, 0)
            debug_mask.paste(pasted_mask, (crop_bbox[0], crop_bbox[1]))
            result = add_contact_shadow(result, insert_bbox, variant)
            debug_generated = source.copy()
            debug_generated.paste(
                generated_crop.resize((crop_bbox[2] - crop_bbox[0], crop_bbox[3] - crop_bbox[1]), Image.LANCZOS),
                (crop_bbox[0], crop_bbox[1]),
            )
            debug_path = save_inpaint_debug_strip(
                record, variant, seed, source, debug_mask, source, debug_generated, result,
                insert_bbox, debug_index=debug_index,
            )
            final_meta["attempt_history"] = json.dumps(attempt_history)
            final_meta["reject_reason"] = ""
            return result, insert_bbox, crop_bbox, debug_path, final_meta

        final_reason = normalize_reject_reason(final_reason)
        last_reject_reason = final_reason
        last_reject_meta = final_meta
        if should_retry(final_reason, attempt, max_retries, final_meta):
            print(f"Retry attempt {attempt + 1} because final validation failed: {final_reason}")
            continue
        break

    if CONTEXT_PERSON_FALLBACK_TO_BBOX_INPAINT:
        print("No accepted generated person after retries; falling back to bbox_inpaint composite for this sample.")
        generated_full = source.copy()
        generated_full_crop = generated_crop.resize((crop_bbox[2] - crop_bbox[0], crop_bbox[3] - crop_bbox[1]), Image.LANCZOS)
        full_mask_crop = mask_image.resize(generated_full_crop.size, Image.BILINEAR)
        generated_full.paste(generated_full_crop, (crop_bbox[0], crop_bbox[1]), full_mask_crop)
        result = add_contact_shadow(generated_full, insert_bbox if "insert_bbox" in locals() else (0, 0, source.size[0], source.size[1]), variant)
        debug_path = save_inpaint_debug_strip(
            record, variant, seed, source, mask_image.resize(source.size), source, result, result,
            insert_bbox if "insert_bbox" in locals() else (0, 0, source.size[0], source.size[1]), debug_index=debug_index,
        )
        fallback_meta = dict(last_reject_meta or default_scale_correction_metadata())
        fallback_meta["attempt_history"] = json.dumps(attempt_history)
        return result, insert_bbox if "insert_bbox" in locals() else None, crop_bbox, debug_path, fallback_meta

    final_reason = normalize_reject_reason(last_reject_reason or "bad_composite_quality")
    raise RuntimeError(f"No accepted generated person after adaptive retries (last_reason={final_reason}).")


def generate_human_mask_inpaint_with_pipe(pipe, source, record, variant, prompt, negative_prompt, seed, device, strength, guidance_scale, num_inference_steps, debug_index=None):
    depth_map = estimate_depth_map(source, device="cpu")
    rng = random.Random(seed)
    insert_bbox, insert_meta = find_insertion_region(record, source, variant, rng, device=device, return_metadata=True, depth_map=depth_map)
    if insert_bbox is None:
        raise RuntimeError(f"Could not find insertion region for {record.path.name}")
    if BACKGROUND_PRESERVATION_MODE == "bbox_inpaint":
        mask_image = bbox_mask_for_bbox(source.size, insert_bbox, variant=variant)
    else:
        mask_image = human_mask_for_bbox(source.size, insert_bbox, variant)
    inpaint_source = prepare_inpaint_source(source, mask_image, insert_bbox)
    generator_device = device if str(device).startswith("cuda") else "cpu"
    generator = torch.Generator(device=generator_device).manual_seed(seed)
    generated = pipe(
        prompt=prompt,
        negative_prompt=negative_prompt,
        image=inpaint_source,
        mask_image=mask_image,
        strength=strength,
        guidance_scale=guidance_scale,
        num_inference_steps=num_inference_steps,
        generator=generator,
    ).images[0].resize(source.size)
    result = source.copy()
    result.paste(generated, (0, 0), mask_image)
    result = add_contact_shadow(result, insert_bbox, variant)
    patch_bbox = expand_bbox_with_context(insert_bbox, resolution=source.width)
    debug_path = save_inpaint_debug_strip(
        record, variant, seed, source, mask_image, inpaint_source, generated, result,
        insert_bbox, debug_index=debug_index,
    )
    return result, insert_bbox, patch_bbox, debug_path


def generate_patch_blend_with_pipe(pipe, source, record, variant, prompt, negative_prompt, seed, device, strength, guidance_scale, num_inference_steps, debug_index=None):
    depth_map = estimate_depth_map(source, device="cpu")
    rng = random.Random(seed)
    insert_bbox, insert_meta = find_insertion_region(record, source, variant, rng, device=device, return_metadata=True, depth_map=depth_map)
    if insert_bbox is None:
        raise RuntimeError(f"Could not find insertion region for {record.path.name}")
    patch_bbox = expand_bbox_with_context(insert_bbox, resolution=source.width)
    source_patch = source.crop(patch_bbox)
    guided_patch = draw_person_guide_on_patch(source_patch, patch_bbox, insert_bbox, variant)
    generator_device = device if str(device).startswith("cuda") else "cpu"
    generator = torch.Generator(device=generator_device).manual_seed(seed)
    aug_patch = pipe(
        prompt=prompt,
        negative_prompt=negative_prompt,
        image=guided_patch,
        strength=strength,
        guidance_scale=guidance_scale,
        num_inference_steps=num_inference_steps,
        generator=generator,
    ).images[0].resize(source_patch.size)
    ix1, iy1, ix2, iy2 = insert_bbox
    px1, py1, _, _ = patch_bbox
    rel_insert_bbox = (ix1 - px1, iy1 - py1, ix2 - px1, iy2 - py1)
    generated_insert = aug_patch.crop(rel_insert_bbox)
    result = source.copy()
    result.paste(generated_insert, (ix1, iy1), feather_mask(generated_insert.size))
    final_patch = result.crop(patch_bbox)
    debug_path = save_patch_debug_strip(
        record, variant, seed, source_patch, guided_patch, aug_patch, final_patch,
        patch_bbox, insert_bbox, debug_index=debug_index,
    )
    return result, insert_bbox, patch_bbox, debug_path


def generate_variant_with_pipe(pipe, record, variant, output_path, seed, device=TRAIN_DEVICE, strength=None, debug_index=None):
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    source = resize_center_crop(load_source_image(record.path), resolution=RESOLUTION)
    prompt = build_generation_prompt(record, variant)
    negative_prompt = build_variant_negative_prompt(variant)
    strength = VARIANT_STRENGTHS.get(variant, AUGMENTATION_STRENGTH) if strength is None else strength
    guidance_scale = VARIANT_GUIDANCE_SCALES.get(variant, GUIDANCE_SCALE)
    num_inference_steps = VARIANT_NUM_INFERENCE_STEPS.get(variant, NUM_INFERENCE_STEPS)
    if seed == SEED:
        print("Prompt sample:", prompt)
        print("Negative sample:", negative_prompt)
        print(f"Generation config: variant={variant}, mode={BACKGROUND_PRESERVATION_MODE}, strength={strength}, guidance_scale={guidance_scale}, steps={num_inference_steps}, resolution={RESOLUTION}")
    clear_cuda()
    scale_meta = default_scale_correction_metadata()
    if BACKGROUND_PRESERVATION_MODE == "context_person_composite":
        image, insert_bbox, patch_bbox, debug_path, scale_meta = generate_context_person_composite_with_pipe(
            pipe, source, record, variant, prompt, negative_prompt, seed, device,
            strength, guidance_scale, num_inference_steps, debug_index=debug_index,
        )
    elif BACKGROUND_PRESERVATION_MODE in {"human_mask_inpaint", "bbox_inpaint"}:
        image, insert_bbox, patch_bbox, debug_path = generate_human_mask_inpaint_with_pipe(
            pipe, source, record, variant, prompt, negative_prompt, seed, device,
            strength, guidance_scale, num_inference_steps, debug_index=debug_index,
        )
    elif BACKGROUND_PRESERVATION_MODE == "patch_blend":
        image, insert_bbox, patch_bbox, debug_path = generate_patch_blend_with_pipe(
            pipe, source, record, variant, prompt, negative_prompt, seed, device,
            strength, guidance_scale, num_inference_steps, debug_index=debug_index,
        )
    else:
        generator_device = device if str(device).startswith("cuda") else "cpu"
        generator = torch.Generator(device=generator_device).manual_seed(seed)
        image = pipe(
            prompt=prompt,
            negative_prompt=negative_prompt,
            image=source,
            strength=strength,
            guidance_scale=guidance_scale,
            num_inference_steps=num_inference_steps,
            generator=generator,
        ).images[0]
        insert_bbox = None
        patch_bbox = None
        debug_path = ""
    image.save(output_path)
    clear_cuda()
    return output_path, {
        "strength": strength,
        "guidance_scale": guidance_scale,
        "num_inference_steps": num_inference_steps,
        "generation_mode": BACKGROUND_PRESERVATION_MODE,
        "insert_bbox": insert_bbox,
        "patch_bbox": patch_bbox,
        "patch_debug_path": debug_path,
        "expected_new_person_count": expected_new_person_count(variant),
        "detected_new_person_count": expected_new_person_count(variant) if insert_bbox is not None else 0,
        **scale_meta,
    }


In [ ]:
%%writefile /kaggle/working/sd35_runner.py
"""SD3.5 CityPersons augmentation: job orchestration, manifests, autotune, and exports."""

import csv
import gc
import json
from datetime import datetime
import math
import numpy as np
import os
import random
import re
import statistics
import time
import warnings
from concurrent.futures import ThreadPoolExecutor, as_completed
from dataclasses import dataclass
from pathlib import Path
from threading import Lock
from typing import Optional

import matplotlib.pyplot as plt
import torch
from PIL import Image, ImageOps, ImageDraw, ImageFilter, ImageChops

try:
    import cv2
except ImportError:
    cv2 = None

from sd35_config import *
from sd35_data import *
from sd35_utils import *
from sd35_model import *
from sd35_evaluation import *
from sd35_pipeline import *

def variant_targets(variant):
    insertion = {
        "add_single_pedestrian": "single_pedestrian",
        "add_two_pedestrians": "two_pedestrians",
        "add_small_group": "small_group",
        "add_occluded_pedestrian": "occluded_pedestrian",
        "add_distant_pedestrian": "distant_pedestrian",
        "add_near_pedestrian": "near_pedestrian",
    }.get(variant, "pedestrian_insertion")
    return insertion, ""


def write_manifest(rows, output_dir=OUTPUT_DIR):
    manifest_path = Path(output_dir) / "manifest.csv"
    manifest_path.parent.mkdir(parents=True, exist_ok=True)
    fieldnames = [
        "split", "bucket", "source_context", "source_timeofday", "source_scene",
        "target_insertion", "target_timeofday", "original_path", "augmented_path",
        "comparison_path", "variant", "strength", "guidance_scale", "num_inference_steps",
        "generation_mode", "insert_bbox", "patch_bbox", "patch_debug_path",
        "expected_new_person_count", "detected_new_person_count",
        "expected_person_height", "detected_person_height", "scale_ratio_before_correction",
        "expected_height", "detected_height", "scale_ratio_before",
        "scale_corrected", "resized_person_height", "resized_person_width",
        "scale_correction_status", "seamless_clone_used", "fallback_alpha_paste",
        "fallback_alpha_used", "foreground_occlusion_used", "foreground_occluder_count",
        "foreground_occlusion_overlap_ratio", "foreground_occlusion_removed_ratio",
        "person_score", "scale_score", "background_score", "edge_score", "quality_score",
        "retry_attempts", "last_reject_reason", "reject_reason",
        "seed", "source_path", "label_path", "output_path",
    ]
    with manifest_path.open("w", encoding="utf-8", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)
    print(f"Saved manifest: {manifest_path}")
    return manifest_path


def build_augmentation_jobs(records, variants, target_per_bucket, target_splits):
    rng = random.Random(SEED)
    grouped = records_by_split_and_bucket(records)
    jobs = []
    job_index = 0
    for split in target_splits:
        for bucket in SCENE_BUCKETS:
            bucket_records = grouped.get(split, {}).get(bucket, [])
            if not bucket_records:
                print(f"SKIP {split}/{bucket}: no source images found")
                continue
            variant_weights = [AUGMENTATION_VARIANT_WEIGHTS.get(variant, 1.0) for variant in variants]
            print(f"Queued {target_per_bucket} images for {split}/{bucket} with EDA-aware variant weights")
            for bucket_index in range(1, target_per_bucket + 1):
                variant = rng.choices(list(variants), weights=variant_weights, k=1)[0]
                record = choose_record_for_variant(bucket_records, variant, rng)
                output_path = generated_image_path(OUTPUT_DIR, record, variant, bucket_index)
                comparison_path = comparison_image_path(OUTPUT_DIR, record, variant, bucket_index)
                jobs.append({
                    "job_index": job_index,
                    "record": record,
                    "variant": variant,
                    "output_path": output_path,
                    "comparison_path": comparison_path,
                    "seed": SEED + job_index,
                })
                job_index += 1
    return jobs

def run_augmentation_jobs_on_device(device, jobs, total_jobs, backend):
    if not jobs:
        return [], []
    if str(device).startswith("cuda"):
        torch.cuda.set_device(torch.device(device).index or 0)
    print(f"[{device}] loading pipeline for {len(jobs)} jobs")
    if BACKGROUND_PRESERVATION_MODE == "context_person_composite" and CONTEXT_PERSON_GENERATION_PIPELINE == "img2img":
        pipe = build_img2img_pipeline(backend=backend, device=device)
    elif BACKGROUND_PRESERVATION_MODE in {"human_mask_inpaint", "bbox_inpaint", "context_person_composite"}:
        pipe = build_inpaint_pipeline(backend=backend, device=device)
    else:
        pipe = build_img2img_pipeline(backend=backend, device=device)
    outputs = []
    rows = []
    local_reject_reasons = {}
    for local_index, job in enumerate(jobs, 1):
        record = job["record"]
        target_insertion, target_timeofday = variant_targets(job["variant"])
        try:
            saved, generation_config = generate_variant_with_pipe(
                pipe=pipe,
                record=record,
                variant=job["variant"],
                output_path=job["output_path"],
                seed=job["seed"],
                device=device,
                strength=None,
                debug_index=job["job_index"],
            )
        except RuntimeError as exc:
            reason = normalize_reject_reason(exc)
            local_reject_reasons[reason] = local_reject_reasons.get(reason, 0) + 1
            print(f"[{device}] rejected {record.path.name} / {job['variant']}: {exc}")
            continue
        source_preview = resize_center_crop(load_source_image(record.path), resolution=RESOLUTION)
        augmented_preview = ImageOps.exif_transpose(Image.open(saved)).convert("RGB")
        comparison_title = (
            f"{job['variant']} | mode={generation_config['generation_mode']} | strength={generation_config['strength']} | "
            f"guidance={generation_config['guidance_scale']} | "
            f"steps={generation_config['num_inference_steps']} | seed={job['seed']}"
        )
        comparison_saved = save_comparison_pair(
            source_preview,
            augmented_preview,
            job["comparison_path"],
            comparison_title,
        )
        outputs.append(saved)
        rows.append({
            "split": record.split,
            "bucket": record.bucket,
            "source_context": record.scene or "urban",
            "source_timeofday": record.timeofday or "",
            "source_scene": record.scene or "",
            "target_insertion": target_insertion,
            "target_timeofday": target_timeofday,
            "original_path": str(record.path),
            "augmented_path": str(saved),
            "comparison_path": str(comparison_saved),
            "variant": job["variant"],
            "strength": generation_config["strength"],
            "guidance_scale": generation_config["guidance_scale"],
            "num_inference_steps": generation_config["num_inference_steps"],
            "generation_mode": generation_config["generation_mode"],
            "insert_bbox": json.dumps(generation_config["insert_bbox"]),
            "patch_bbox": json.dumps(generation_config["patch_bbox"]),
            "patch_debug_path": generation_config["patch_debug_path"],
            "expected_new_person_count": generation_config["expected_new_person_count"],
            "detected_new_person_count": generation_config["detected_new_person_count"],
            "expected_person_height": generation_config.get("expected_person_height", ""),
            "detected_person_height": generation_config.get("detected_person_height", ""),
            "scale_ratio_before_correction": generation_config.get("scale_ratio_before_correction", ""),
            "expected_height": generation_config.get("expected_height", generation_config.get("expected_person_height", "")),
            "detected_height": generation_config.get("detected_height", generation_config.get("detected_person_height", "")),
            "scale_ratio_before": generation_config.get("scale_ratio_before", generation_config.get("scale_ratio_before_correction", "")),
            "scale_corrected": generation_config.get("scale_corrected", False),
            "resized_person_height": generation_config.get("resized_person_height", ""),
            "resized_person_width": generation_config.get("resized_person_width", ""),
            "scale_correction_status": generation_config.get("scale_correction_status", "none"),
            "seamless_clone_used": generation_config.get("seamless_clone_used", False),
            "fallback_alpha_paste": generation_config.get("fallback_alpha_paste", False),
            "fallback_alpha_used": generation_config.get("fallback_alpha_used", generation_config.get("fallback_alpha_paste", False)),
            "foreground_occlusion_used": generation_config.get("foreground_occlusion_used", False),
            "foreground_occluder_count": generation_config.get("foreground_occluder_count", 0),
            "foreground_occlusion_overlap_ratio": generation_config.get("foreground_occlusion_overlap_ratio", 0.0),
            "foreground_occlusion_removed_ratio": generation_config.get("foreground_occlusion_removed_ratio", 0.0),
            "person_score": generation_config.get("person_score", ""),
            "scale_score": generation_config.get("scale_score", ""),
            "background_score": generation_config.get("background_score", ""),
            "edge_score": generation_config.get("edge_score", ""),
            "quality_score": generation_config.get("quality_score", ""),
            "retry_attempts": generation_config.get("retry_attempts", 0),
            "last_reject_reason": generation_config.get("last_reject_reason", ""),
            "reject_reason": generation_config.get("reject_reason", generation_config.get("last_reject_reason", "")),
            "seed": job["seed"],
            "source_path": str(record.path),
            "label_path": str(record.label_path) if record.label_path else "",
            "output_path": str(saved),
        })
        completed = job["job_index"] + 1
        if completed == 1 or completed % 25 == 0 or local_index == len(jobs):
            print(f"[{device}] [{completed}/{total_jobs}] saved {saved.name}")
    del pipe
    clear_cuda()
    return outputs, rows, local_reject_reasons


def augment_dataset(records, variants=AUGMENTATION_VARIANTS, backend=MODEL_BACKEND, target_per_bucket=AUGMENTATIONS_PER_BUCKET, target_splits=TARGET_SPLITS, write_manifest_file=True, return_manifest_rows=False):
    if not records:
        raise FileNotFoundError("No images found. Mount dataset folder and rerun scan_dataset().")
    devices = resolve_augmentation_devices()
    jobs = build_augmentation_jobs(records, variants, target_per_bucket, target_splits)
    if not jobs:
        print("No augmentation jobs were queued.")
        return []
    total_jobs = len(jobs)
    print(f"Using augmentation devices: {devices}")
    shards = [jobs[index::len(devices)] for index in range(len(devices))]
    all_outputs = []
    manifest_rows = []
    reject_histogram = {}
    for device, shard in zip(devices, shards):
        if not shard:
            continue
        outputs, rows, device_rejects = run_augmentation_jobs_on_device(device, shard, total_jobs, backend)
        all_outputs.extend(outputs)
        manifest_rows.extend(rows)
        for reason, count in device_rejects.items():
            reject_histogram[reason] = reject_histogram.get(reason, 0) + count
    manifest_rows = sorted(manifest_rows, key=lambda row: row["seed"])
    all_outputs = [Path(row["output_path"]) for row in manifest_rows]
    if write_manifest_file:
        write_manifest(manifest_rows, OUTPUT_DIR)
    accepted = len(all_outputs)
    rejected = sum(reject_histogram.values())
    retried = sum(int(row.get("retry_attempts") or 0) for row in manifest_rows)
    scale_corrected_count = sum(1 for row in manifest_rows if str(row.get("scale_corrected")).lower() == "true")
    seamless_clone_used_count = sum(1 for row in manifest_rows if str(row.get("seamless_clone_used")).lower() == "true")
    fallback_alpha_paste_count = sum(1 for row in manifest_rows if str(row.get("fallback_alpha_paste")).lower() == "true")
    print(f"Generated {len(all_outputs)} images in {OUTPUT_DIR}")
    print("Smoke/evaluation counters:")
    print(f"  total attempts: {total_jobs}")
    print(f"  accepted: {accepted}")
    print(f"  rejected: {rejected}")
    print(f"  retried: {retried}")
    print(f"  scale_corrected_count: {scale_corrected_count}")
    print(f"  seamless_clone_used_count: {seamless_clone_used_count}")
    print(f"  fallback_alpha_paste_count: {fallback_alpha_paste_count}")
    print(f"  reject reasons histogram: {reject_histogram}")
    if total_jobs:
        estimated_before_scale_correction_accepts = max(0, accepted - scale_corrected_count)
        print(f"  accept rate before scale correction (estimated): {estimated_before_scale_correction_accepts / total_jobs:.3f}")
        print(f"  accept rate after scale correction: {accepted / total_jobs:.3f}")
        if accepted:
            print(f"  scale correction share of accepted: {scale_corrected_count / accepted:.3f}")
            print(f"  seamless clone success rate: {seamless_clone_used_count / accepted:.3f}")
    global LAST_MANIFEST_ROWS, LAST_REJECT_HISTOGRAM, LAST_AUGMENTATION_SUMMARY
    LAST_MANIFEST_ROWS = manifest_rows
    LAST_REJECT_HISTOGRAM = reject_histogram
    LAST_AUGMENTATION_SUMMARY = {
        "total_jobs": total_jobs,
        "accepted": accepted,
        "rejected": rejected,
        "accept_rate": accepted / total_jobs if total_jobs else 0.0,
        "reject_histogram": reject_histogram,
    }
    if return_manifest_rows:
        return all_outputs, manifest_rows
    return all_outputs


## 10.10 Quality-Guided Autotune

LAST_AUTOTUNE_RECOMMENDATIONS = {}


def summarize_quality_rows(rows):
    rows = rows or []
    accepted_rows = [row for row in rows if float(row.get("quality_score", 0.0) or 0.0) > 0.0]
    if not accepted_rows:
        return {
            "count": 0,
            "person_mean": 0.0,
            "scale_mean": 0.0,
            "background_mean": 0.0,
            "edge_mean": 0.0,
            "quality_mean": 0.0,
            "person_p10": 0.0,
            "scale_p10": 0.0,
            "background_p10": 0.0,
            "edge_p10": 0.0,
            "quality_p10": 0.0,
        }

    def values(key):
        return np.array([float(row.get(key, 0.0) or 0.0) for row in accepted_rows], dtype=np.float32)

    summary = {"count": len(accepted_rows)}
    for key in ("person_score", "scale_score", "background_score", "edge_score", "quality_score"):
        arr = values(key)
        short = key.replace("_score", "")
        summary[f"{short}_mean"] = round(float(arr.mean()), 4)
        summary[f"{short}_p10"] = round(float(np.percentile(arr, 10)), 4)
    return summary


def _bounded_value(name, proposed):
    if name not in EFFECTIVE_CONFIG:
        return proposed
    current = EFFECTIVE_CONFIG[name]
    if isinstance(current, bool) or not isinstance(current, (int, float)):
        return proposed
    if current == 0:
        return proposed
    max_ratio = float(AUTOTUNE_SETTINGS.get("max_adjustment_ratio", 1.35))
    lower = current / max_ratio
    upper = current * max_ratio
    bounded = min(max(float(proposed), lower), upper)
    if isinstance(current, int):
        return int(round(bounded))
    return round(bounded, 4)


def _blend_value(name, proposed):
    current = EFFECTIVE_CONFIG.get(name)
    if isinstance(current, bool) or not isinstance(current, (int, float)):
        return proposed
    aggressiveness = float(AUTOTUNE_SETTINGS.get("aggressiveness", 0.60))
    blended = current + (float(proposed) - float(current)) * aggressiveness
    return _bounded_value(name, blended)


def _format_delta(old, new):
    if isinstance(old, (int, float)) and old != 0:
        return round((float(new) - float(old)) / abs(float(old)), 4)
    if isinstance(old, (int, float)):
        return round(float(new) - float(old), 4)
    return None


def _build_change_report(overrides, generation_delta=None):
    changes = {}
    for name, new_value in overrides.items():
        old_value = EFFECTIVE_CONFIG.get(name)
        changes[name] = {
            "old": old_value,
            "new": new_value,
            "delta": _format_delta(old_value, new_value),
        }
    if generation_delta:
        changes["generation_delta"] = {
            "old": None,
            "new": generation_delta,
            "delta": None,
        }
    return changes


def recommend_parameter_updates(rows=None, reject_histogram=None, summary=None):
    rows = LAST_MANIFEST_ROWS if rows is None else rows
    reject_histogram = LAST_REJECT_HISTOGRAM if reject_histogram is None else reject_histogram
    summary = LAST_AUGMENTATION_SUMMARY if summary is None else summary
    quality = summarize_quality_rows(rows)
    total_jobs = int((summary or {}).get("total_jobs", len(rows or [])))
    accepted = int((summary or {}).get("accepted", quality["count"]))
    accept_rate = accepted / max(1, total_jobs)
    rejects = reject_histogram or {}
    recommended = {}
    generation_delta = None

    if quality["count"] < AUTOTUNE_SETTINGS["min_samples"]:
        return {
            "quality_summary": quality,
            "accept_rate": round(accept_rate, 4),
            "reject_histogram": rejects,
            "recommended_overrides": {},
            "changes": {},
            "note": "Not enough accepted samples for stable autotune.",
        }

    if accept_rate < 0.30:
        recommended["CONTEXT_GENERATION_RETRIES"] = min(
            AUTOTUNE_SETTINGS["max_retry_budget"],
            int(CONTEXT_GENERATION_RETRIES) + 1,
        )

    low_person_rejects = sum(rejects.get(reason, 0) for reason in ("low_person_conf", "no_person_detected", "no_person_mask", "segmenter_failed"))
    if low_person_rejects >= max(2, total_jobs * 0.15) or quality["person_p10"] < 0.45:
        recommended["CONTEXT_PERSON_MIN_CONFIDENCE"] = max(0.08, CONTEXT_PERSON_MIN_CONFIDENCE - 0.02)
        recommended["MIN_RETRY_PERSON_CONFIDENCE"] = max(0.08, MIN_RETRY_PERSON_CONFIDENCE - 0.02)

    if rejects.get("bad_scale", 0) >= max(2, total_jobs * 0.12) or quality["scale_p10"] < 0.55:
        recommended["SCALE_CORRECTION_MAX_ATTEMPTS"] = min(4, int(SCALE_CORRECTION_MAX_ATTEMPTS) + 1)
        recommended["SCALE_CORRECTION_HEIGHT_STEP"] = min(0.12, SCALE_CORRECTION_HEIGHT_STEP + 0.02)

    if rejects.get("bad_person_depth_overlap", 0) >= max(1, total_jobs * 0.08):
        recommended["MAX_PERSON_PERSON_OVERLAP_RATIO"] = max(0.04, MAX_PERSON_PERSON_OVERLAP_RATIO - 0.02)
        recommended["OCCLUDED_PERSON_MAX_HEIGHT_RATIO"] = max(0.55, OCCLUDED_PERSON_MAX_HEIGHT_RATIO - 0.05)

    if quality["background_p10"] < 0.55:
        recommended["SEAMLESS_CLONE_MIN_MASK_AREA"] = max(0.003, SEAMLESS_CLONE_MIN_MASK_AREA - 0.001)
        recommended["LOCAL_BRIGHTNESS_STRENGTH"] = min(0.45, LOCAL_BRIGHTNESS_STRENGTH + 0.04)
        recommended["PERSON_COLOR_MATCH_STRENGTH"] = min(0.45, PERSON_COLOR_MATCH_STRENGTH + 0.04)

    if quality["edge_p10"] < 0.55:
        recommended["EDGE_HALO_COLOR_MATCH_STRENGTH"] = min(0.48, EDGE_HALO_COLOR_MATCH_STRENGTH + 0.04)
        recommended["PERSON_PASTE_FEATHER_RADIUS"] = min(0.60, PERSON_PASTE_FEATHER_RADIUS + 0.08)

    if quality["quality_mean"] < AUTOTUNE_SETTINGS["target_quality_score"]:
        generation_delta = {"strength": 0.01, "guidance": 0.10, "steps": 0}

    bounded = {name: _blend_value(name, value) for name, value in recommended.items()}
    changes = _build_change_report(bounded, generation_delta)

    return {
        "quality_summary": quality,
        "accept_rate": round(accept_rate, 4),
        "reject_histogram": rejects,
        "recommended_overrides": bounded,
        "generation_delta": generation_delta,
        "changes": changes,
    }



def json_safe_value(value):
    if value is None or isinstance(value, (str, int, float, bool)):
        return value
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, dict):
        return {str(key): json_safe_value(item) for key, item in value.items()}
    if isinstance(value, (list, tuple, set)):
        return [json_safe_value(item) for item in value]
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, np.ndarray):
        return value.tolist()
    return str(value)

def save_autotune_snapshot(report, before_config, after_config):
    if not AUTOTUNE_SETTINGS.get("save_snapshot", True):
        return None
    snapshot_dir = Path(AUTOTUNE_SETTINGS.get("snapshot_dir", "autotune_snapshots"))
    snapshot_dir.mkdir(parents=True, exist_ok=True)
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    snapshot_path = snapshot_dir / f"autotune_snapshot_{timestamp}.json"
    payload = json_safe_value({
        "timestamp": timestamp,
        "run_preset": RUN_PRESET,
        "before_config": before_config,
        "after_config": after_config,
        "report": report,
    })
    snapshot_path.write_text(json.dumps(payload, indent=2, ensure_ascii=False), encoding="utf-8")
    return snapshot_path


def print_autotune_report(report):
    print("=== AUTOTUNE REPORT ===")
    print("Quality summary:", report.get("quality_summary"))
    print("Accept rate:", report.get("accept_rate"))
    print("Reject histogram:", report.get("reject_histogram"))
    note = report.get("note")
    if note:
        print("Note:", note)
    changes = report.get("changes", {})
    if not changes:
        print("No parameter changes recommended.")
        return
    for name, change in changes.items():
        old = change["old"]
        new = change["new"]
        delta = change["delta"]
        if isinstance(old, (int, float)) and isinstance(new, (int, float)) and delta is not None:
            print(f"  {name:35s}: {old:.4g} -> {new:.4g} ({delta:+.1%})")
        else:
            print(f"  {name:35s}: {old} -> {new}")



def _refresh_variant_generation_maps():
    global AUGMENTATION_VARIANT_WEIGHTS, VARIANT_STRENGTHS, VARIANT_GUIDANCE_SCALES, VARIANT_NUM_INFERENCE_STEPS
    AUGMENTATION_VARIANT_WEIGHTS = {name: cfg["weight"] for name, cfg in VARIANT_PROFILE.items()}
    VARIANT_STRENGTHS = {name: cfg["strength"] for name, cfg in VARIANT_PROFILE.items()}
    VARIANT_GUIDANCE_SCALES = {name: cfg["guidance"] for name, cfg in VARIANT_PROFILE.items()}
    VARIANT_NUM_INFERENCE_STEPS = {name: cfg["steps"] for name, cfg in VARIANT_PROFILE.items()}


def _bump_variant_generation(strength_delta=0.0, guidance_delta=0.0, step_delta=0):
    for profile in VARIANT_PROFILE.values():
        profile["strength"] = round(float(np.clip(profile["strength"] + strength_delta, 0.55, 0.86)), 4)
        profile["guidance"] = round(float(np.clip(profile["guidance"] + guidance_delta, 5.5, 8.5)), 4)
        profile["steps"] = int(np.clip(int(profile["steps"] + step_delta), 24, 48))
    _refresh_variant_generation_maps()

def apply_parameter_updates(report):
    updates = dict((report or {}).get("recommended_overrides", {}))
    generation_delta = (report or {}).get("generation_delta")
    if generation_delta:
        _bump_variant_generation(
            strength_delta=generation_delta.get("strength", 0.0),
            guidance_delta=generation_delta.get("guidance", 0.0),
            step_delta=generation_delta.get("steps", 0),
        )
    for name, value in updates.items():
        globals()[name] = value
        EFFECTIVE_CONFIG[name] = value
    LAST_AUTOTUNE_RECOMMENDATIONS.clear()
    LAST_AUTOTUNE_RECOMMENDATIONS.update(updates)
    if generation_delta:
        LAST_AUTOTUNE_RECOMMENDATIONS["generation_delta"] = generation_delta
    return dict(LAST_AUTOTUNE_RECOMMENDATIONS)


def autotune_from_last_run(apply=True, dry_run=True, save_snapshot=None):
    before_config = dict(EFFECTIVE_CONFIG)
    report = recommend_parameter_updates()
    if AUTOTUNE_SETTINGS.get("print_report", True):
        print_autotune_report(report)
    should_apply = bool(apply and not dry_run and AUTOTUNE_SETTINGS.get("enabled", True))
    if should_apply:
        applied = apply_parameter_updates(report)
        print("Applied runtime parameter updates:", applied)
    else:
        print("Dry run only; no runtime parameters were changed.")
    after_config = dict(EFFECTIVE_CONFIG)
    if save_snapshot is None:
        save_snapshot = AUTOTUNE_SETTINGS.get("save_snapshot", True)
    if save_snapshot:
        snapshot_path = save_autotune_snapshot(report, before_config, after_config)
        if snapshot_path:
            print("Saved autotune snapshot:", snapshot_path)
    return report

## 10.11 Reset Runtime Config

def reset_runtime_config():
    global EFFECTIVE_CONFIG, VARIANT_PROFILE
    EFFECTIVE_CONFIG = dict(BASE_EFFECTIVE_CONFIG)
    globals().update(EFFECTIVE_CONFIG)
    VARIANT_PROFILE.clear()
    VARIANT_PROFILE.update({name: dict(profile) for name, profile in BASE_VARIANT_PROFILE.items()})
    _refresh_variant_generation_maps()
    if "LAST_AUTOTUNE_RECOMMENDATIONS" in globals():
        LAST_AUTOTUNE_RECOMMENDATIONS.clear()
    print("Runtime config reset to notebook defaults.")
    return dict(EFFECTIVE_CONFIG)


# Run this cell, then call reset_runtime_config() whenever you want to undo runtime autotune changes.

def run_smoke(records, smoke_images=10, smoke_splits=None):
    smoke_splits = smoke_splits or ["train"]
    generated_paths, manifest_rows = augment_dataset(
        records,
        variants=AUGMENTATION_VARIANTS,
        target_per_bucket=smoke_images,
        target_splits=smoke_splits,
        return_manifest_rows=True,
    )
    autotune_report = autotune_from_last_run(apply=True, dry_run=True)
    return generated_paths, manifest_rows, autotune_report


def export_outputs():
    import shutil
    export_base = Path("/kaggle/working/sd35_citypersons_augmented_export")
    if export_base.with_suffix(".zip").exists():
        export_base.with_suffix(".zip").unlink()
    shutil.make_archive(str(export_base), "zip", str(OUTPUT_DIR))
    print(f"Saved export: {export_base.with_suffix('.zip')}")
    return export_base.with_suffix(".zip")


if __name__ == "__main__":
    ensure_output_dirs()
    records = load_records()
    run_smoke(records)


## 3. Imports

In [ ]:

import sys
sys.path.insert(0, "/kaggle/working")

from sd35_config import *
from sd35_data import *
from sd35_utils import *
from sd35_model import *
from sd35_evaluation import *
from sd35_pipeline import *
from sd35_runner import *

ensure_output_dirs()


## 4. Runtime Check

In [ ]:
import torch
print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu count:', torch.cuda.device_count())
    for index in range(torch.cuda.device_count()):
        print(index, torch.cuda.get_device_name(index))


## 5. Hugging Face Login

In [ ]:
from huggingface_hub import login

hf_token = None
try:
    from kaggle_secrets import UserSecretsClient
    hf_token = UserSecretsClient().get_secret("HF_TOKEN")
except Exception as exc:
    import os
    hf_token = os.environ.get("HF_TOKEN")
    if not hf_token:
        raise RuntimeError("HF_TOKEN not found. Add a Kaggle secret named HF_TOKEN or set os.environ['HF_TOKEN'].") from exc

login(token=hf_token)
print("Hugging Face login OK from HF_TOKEN secret.")


## 6. Dataset Scan And Preview

In [ ]:
records = load_records()
summarize_citypersons_records(records)
preview_prompt_samples(records)
preview_records(records)


## 7. First Run

In [ ]:
SMOKE_IMAGES = 10
SMOKE_SPLITS = ["train"]

generated_paths, manifest_rows, autotune_report = run_smoke(
    records,
    smoke_images=SMOKE_IMAGES,
    smoke_splits=SMOKE_SPLITS,
)
generated_paths[:5]


## 8. Export Outputs

In [ ]:
# Run when you want a zip artifact.
# export_outputs()
